# STELES PIPELINE 

## Part II - Wavelength Solution, Spatial Distortion and Blaze correction, and 1d extraction

by Felipe Navarete (LNA/MCTI)

*VERSION FOR DATA TAKEN AFTER MISSION 14 - 2026-01-28 OR LATER*

In [ ]:
from steles_pipeline.steles_imports import * #fits, np, os, pd, savgol_filter, UnivariateSpline
from steles_pipeline.__init__ import DB_DIR

# local packages
from steles_pipeline import STELES_BasePipeline        # BasePipeline routines
from steles_pipeline import steles_spec as steles_spec # spectroscopic routines
from steles_pipeline import steles_qc   as steles_qc   # Quality Control plots routines

#######################################################################################
print('functions were loaded.')

In [ ]:
class STELES_Pipeline_WaveSolution_Test(STELES_BasePipeline):
        
    # =========================================================
    # Checkpointing System
    # =========================================================
    logger_name = "STELES-PartII"
    
    def create_arc1d(self):
        
        if self._get_bool("dry_run"):
            self.log.info(f"[DRY RUN] Would execute: create_arc1d()")
            return
        
        output = self.proc_dir / ( self.cfg["fileout_arc"] + f"_rectified_1d{FITS_EXTENSION}" )
        
        if self._check_done([output]) and not self.cfg["overwrite"]:
            self.log.info("Skipping creating 1d arc spectrum")
            return        

        self.log.info("Creating the 1d arc spectrum for wavelength solution")

        # Collapse arc
        ymid = int(0.5 * self.n_spat)
        y_range = [ymid-1, ymid+1]   # use only center of the slit
        y_range = [0, self.n_spat] # use full range

        arc1d = steles_spec.collapse_spectrum2d(self.arc_cube, row_range=y_range, return_average=True)

        # --- Write output ---
        self._write_fits(arc1d, None, output, overwrite=True)
        
        self.arc1d = arc1d
    

    def save_wavelength_qc_table(self):

        rows = []
        for s in self.wavesol_stats:
            rows.append({
                "order":      s["order"],
                "n_lines":    s["n_lines"],
                "dispersion": s["dispersion"],
                "rms":        s["rms"],
                "residuals":  s["residuals"],
                "min_wave":   s["min_wave"],
                "mid_wave":   s["mid_wave"],
                "max_wave":   s["max_wave"]
            })

        df = pd.DataFrame(rows)

        out = self.proc_dir / ( self.cfg["fileout_arc"] + f"_wavelength_qc{CSV_EXTENSION}" )
        
        df.to_csv(out, index=False)

        self.log.info(f"Saved wavelength QC table: {out}")   
        
    def wavelength_solution(self):
        
        if self._get_bool("dry_run"):
            self.log.info(f"[DRY RUN] Would execute: wavelength_solution()")
            return
        
        output = self.proc_dir / ( self.cfg["fileout_arc"] + f"_wavelength_solution{FITS_EXTENSION}" )
        
        if self._check_done([output]) and not self.cfg["overwrite"]:
            self.log.info("Skipping computing wavelength solution")
            return
        
        self.log.info("Computing wavelength solution")

        self.load_arc1d()
        wave1d = self.arc1d.copy()
        log_lines = []
        wavelength_info = self.wavelength_info.copy()
        
        self.wavesol_stats = []

        for order in range(1, self.n_orders + 1):

            spec = self.arc1d[order-1]

            info = wavelength_info[wavelength_info['order'] == order]
            wl = info['wavelength'].to_numpy().copy()
            px = info['pixel'].to_numpy().copy()
            
            px += self.cfg["wavelength_pixel_offset"]

            if not np.isnan(px).any():
                
                file_out = self.wave_dir / ( self.cfg["fileout_arc"] + f"_n{order:02d}_" )

                w_axis, rms, poly, refined_peak_positions, wl = steles_spec.fit_wavelength_solution(
                    spec, wl, px, order,
                    fit_window=self.cfg["window_fit_peak"],
                    max_fwhm_threshold=self.cfg["max_fwhm_threshold"],
                    wavesolution_ndeg=self.cfg["wavesolution_ndeg"],
                    wavesolution_niter=self.cfg["wavesolution_niter"],
                    wavesolution_rms_threshold=self.cfg["wavesolution_rms_threshold"],
                    show_plots=self.cfg["show_plots"],
                    save_plots=True,
                    plot_directory=self.wave_dir,
                    plot_prefix=file_out,
                    displ_percent_max=99,
                    ref_thar_directory=DB_DIR,
                    ref_thar_file="ThAr_atlas_3000_10600_75k"
                )
                
                # model prediction at peak pixels
                model_wavelength = poly(refined_peak_positions)
                residuals = model_wavelength - wl
                
                # compute dispersion per order
                dispersion = np.abs(np.diff(w_axis))
                disp_med = np.nanmedian(dispersion)
                n_lines =  len(refined_peak_positions)
                # central, min and max wavelength (in A)
                mid_wave = w_axis[int(0.5*len(w_axis))]
                min_wave = np.nanmin(w_axis)
                max_wave = np.nanmax(w_axis)

            else:
                w_axis = np.arange(len(spec))
                rms = np.nan
                residuals = np.array([])
                disp_med = np.nan
                n_lines =  0
                # central, min and max wavelength (in A)
                mid_wave = np.nan
                min_wave = np.nan
                max_wave = np.nan
  
            wave1d[order-1] = w_axis
            log_lines.append((order, rms))
            
            self.wavesol_stats.append({"order": order,
                                       "min_wave": min_wave,
                                       "mid_wave": mid_wave,
                                       "max_wave": max_wave,
                                       "rms": rms,
                                       "dispersion": disp_med,
                                       "n_lines": n_lines,
                                       "residuals": residuals})
        
        
        # show orders with relatively high RMS > 0.5 - should be set by the user?
        bad_orders = [s for s in self.wavesol_stats if s["rms"] > 0.5]
        if len(bad_orders) > 0:
            self.log.warning(f"{len(bad_orders)} orders have large RMS")

        # save the wavelength solution
        self.wave1d = wave1d
        
        self._write_fits(wave1d, None, output, overwrite=True)
        
        self.save_wavelength_qc_table()


    def wavelength_solution_debug(self, order, ThAr_peak_wavelengths, ThAr_peak_pixels, print_orders=False, poly_degree=3):
        
        self.load_rectified_products()
        
        self.create_arc1d()
        
        if self._get_bool("dry_run"):
            self.log.info(f"[DRY RUN] Would execute: wavelength_solution_debug()")
            return
        
        output = self.proc_dir / ( self.cfg["fileout_arc"] + f"_wavelength_solution_debug{FITS_EXTENSION}")
        
        if self._check_done([output]) and not self.cfg["overwrite"]:
            self.log.info("Skipping computing wavelength solution")
            return
        
        self.log.info("Computing wavelength solution")

        self.load_arc1d()
        wave1d = self.arc1d.copy()
        log_lines = []
        wavelength_info = self.wavelength_info.copy()
        
        self.wavesol_stats = []

        if order > self.n_orders + 1:
            self.log.info(f"Order n={order} is larger than the available number of orders ({self.n_orders}). Try again with ")
            return
        
        spec = self.arc1d[order-1]
        #spec = np.flip(spec)

        if print_orders:
            for n in range(self.n_orders):
        
                info = wavelength_info[wavelength_info['order'] == n+1]
                wl = info['wavelength'].to_numpy().copy()
                px = info['pixel'].to_numpy().copy()

                wl_txt = 'ThAr_wavelengths = [ '
                for w in wl:
                    wl_txt += f' {w:.4f},'
                wl_txt += ']'

                px_txt = 'ThAr_pixels      = [ '
                for p in px:
                    px_txt += f' {int(p)},'
                px_txt += ']'

                print(f"n_order={n+1}")
                print(wl_txt)
                print(px_txt)
                print("")
            return
            

        wl = np.asarray(ThAr_peak_wavelengths, dtype=float)
        px = np.asarray(ThAr_peak_pixels, dtype=float)

        px += self.cfg["wavelength_pixel_offset"]

        file_out = self.wave_dir / ( "debug_" + self.cfg["fileout_arc"] + f"_n{order:02d}" )
        
        w_axis, rms, poly, refined_peak_positions, wl = steles_spec.fit_wavelength_solution(
            spec, wl, px, order,
            fit_window=self.cfg["window_fit_peak"],
            max_fwhm_threshold=self.cfg["max_fwhm_threshold"],
            wavesolution_ndeg=self.cfg["wavesolution_ndeg"],
            wavesolution_niter=self.cfg["wavesolution_niter"],
            wavesolution_rms_threshold=self.cfg["wavesolution_rms_threshold"],
            wave_peak_threshold=self.cfg["wavesolution_peak_offset_threshold"],
            show_plots=self.cfg["show_plots"],
            save_plots=True,
            plot_directory=self.wave_dir,
            plot_prefix=file_out,
            displ_percent_max=99,
            ref_thar_directory=DB_DIR,
            ref_thar_file="ThAr_atlas_3000_10600_75k"
        )

        # model prediction at peak pixels
        model_wavelength = poly(refined_peak_positions)
        residuals = model_wavelength - wl

        # compute dispersion per order
        dispersion = np.abs(np.diff(w_axis))
        disp_med = np.nanmedian(dispersion)
        n_lines =  len(refined_peak_positions)
        # central, min and max wavelength (in A)
        mid_wave = w_axis[int(0.5*len(w_axis))]
        min_wave = np.nanmin(w_axis)
        max_wave = np.nanmax(w_axis)

        #wave1d[order-1] = w_axis
        #log_lines.append((order, rms))
            
        self.wavesol_stats.append({"order": order,
                                   "min_wave": min_wave,
                                   "mid_wave": mid_wave,
                                   "max_wave": max_wave,
                                   "rms": rms,
                                   "dispersion": disp_med,
                                   "n_lines": n_lines,
                                   "residuals": residuals})
        
        
        # show orders with relatively high RMS > 0.5 - should be set by the user?
        bad_orders = [s for s in self.wavesol_stats if s["rms"] > 0.5]
        if len(bad_orders) > 0:
            self.log.warning(f"{len(bad_orders)} orders have large RMS")

        # save the wavelength solution
        #self.wave1d = wave1d
        #self._write_fits(wave1d, None, output, overwrite=True)
        #self.save_wavelength_qc_table()
        
        
    # plots
    def qc_wavelength_rms(self):

        if self._get_bool("dry_run"):
            self.log.info("[DRY RUN] Would execute: qc_wavelength_rms()")
            return

        if (not self.cfg.get("save_plots", True) and not self.cfg.get("show_plots", False)):
            self.log.info("Skipping wavelength RMS QC plot (disabled)")
            return

        stats, output_path = steles_qc.qc_wavelength_rms(wavesol_stats=self.wavesol_stats,
                                                         output_dir=self.plot_dir,
                                                         output_name=self.cfg["fileout_arc"],
                                                         save_plot=self.cfg.get("save_plots", True),
                                                         show_plot=self.cfg.get("show_plots", False),
                                                         figsize=(10, 4),
                                                         nbins=20,)
        
        self.wavelength_rms_qc = stats
        self.qa_metrics["wavelength_rms"] = stats
        

        if output_path is not None:
            self.log.info(f"Saved RMS histogram: {output_path}")
        
    def qc_line_residuals(self):

        if self._get_bool("dry_run"):
            self.log.info("Would execute: qc_line_residuals()")
            return

        if not self.cfg.get("save_plots", True) and not self.cfg.get("show_plots", False):
            self.log.info("Skipping line residual QC plot (disabled)")
            return

        stats, output_path = steles_qc.qc_line_residuals(wavesol_stats=self.wavesol_stats,
                                                         n_orders=self.n_orders,
                                                         output_dir=self.plot_dir,
                                                         output_name=self.cfg["fileout_arc"],
                                                         save_plot=self.cfg.get("save_plots", True),
                                                         show_plot=self.cfg.get("show_plots", False))

        self.line_residual_qc = stats
        self.qa_metrics["line_residuals"] = stats
        
        if output_path is not None:
            self.log.info(f"Saved residuals plot: {output_path}")  

    def qc_dispersion_stability(self):

        if self._get_bool("dry_run"):
            self.log.info("Would execute: qc_dispersion_stability()")
            return

        if not self.cfg.get("save_plots", True) and not self.cfg.get("show_plots", False):
            self.log.info("Skipping dispersion QC plot (disabled)")
            return

        stats, output_path = steles_qc.qc_dispersion_stability(wavesol_stats=self.wavesol_stats,
                                                               n_orders=self.n_orders,
                                                               output_dir=self.plot_dir,
                                                               output_name=self.cfg["fileout_arc"],
                                                               save_plot=self.cfg.get("save_plots", True),
                                                               show_plot=self.cfg.get("show_plots", False))

        self.dispersion_qc = stats
        self.qa_metrics["dispersion"] = stats
        
        if output_path is not None:
            self.log.info(f"Saved dispersion plot: {output_path}")
            
    def qc_rms_over_dispersion(self, velocity_scale=True):

        if self._get_bool("dry_run"):
            self.log.info("Would execute: qc_rms_over_dispersion()")
            return

        if not self.cfg.get("save_plots", True) and not self.cfg.get("show_plots", False):
            self.log.info("Skipping RMS/dispersion QC plot (disabled)")
            return

        stats, output_path = steles_qc.qc_rms_over_dispersion(wavesol_stats=self.wavesol_stats,
                                                              n_orders=self.n_orders,
                                                              output_dir=self.plot_dir,
                                                              output_name=self.cfg["fileout_arc"],
                                                              velocity_scale=velocity_scale,
                                                              save_plot=self.cfg.get("save_plots", True),
                                                              show_plot=self.cfg.get("show_plots", False))

        self.rms_dispersion_qc = stats
        self.qa_metrics["rms_dispersion"] = stats

        if output_path is not None:
            self.log.info(f"Saved RMS/dispersion plot: {output_path}")       
        
    #=========================================================
    # 8. Run controller
    #========================================================= 
    def run_wavesol_debug(self, order, ThAr_peak_wavelengths, ThAr_peak_pixels, print_orders=False, poly_degree=3):
        self.wavelength_solution_debug(order, ThAr_peak_wavelengths, ThAr_peak_pixels, print_orders=print_orders, poly_degree=poly_degree)
            
print("LOADED")

In [ ]:
#2026-10-09 (SV-October)
suffix_b = 'B'
suffix_r = 'R'

config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    #"raw_directory":  "D:/SOAR/Steles/2026-10-09/", # set full path contaninig the raw data
    "raw_directory":  "/home/fnavarete/astro/data/SOAR/Steles/2026-10-09/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,   # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": [f"q20w_{suffix_b}_0016",
                  f"q20w_{suffix_b}_0017",
                  f"q20w_{suffix_b}_0018",
                  f"q20w_{suffix_b}_0019",
                  f"q20w_{suffix_b}_0020",
                  f"q20w_{suffix_b}_0021",
                  f"q20w_{suffix_b}_0022",
                  f"q20w_{suffix_b}_0023",
                  f"q20w_{suffix_b}_0024",
                  f"q20w_{suffix_b}_0025",], # 150s
    #"longflat_list": ["q20w_B_0035","q20w_B_0036","q20w_B_0037"], # 900s
    #"arc_list":  [f"thar_{suffix_b}_0103",
    #              f"thar_{suffix_b}_0104",],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": f"master_flat_{suffix_b}",
    #"fileout_longflat": "master_longflat_B",
    #"fileout_arc":  f"master_thar_{suffix_b}_0103",
    #
    #"sci_list":    [f"sci_{suffix_b}_0102"],
    #"fileout_sci": F"master_HD216956_tracing_{suffix_b}_0102",
    #
    #"sci_list":    [f"sci_{suffix_b}_0100"],
    #"fileout_sci": F"master_HD216956_{suffix_b}_0100",
    #
    ##################################################################################
    #"sci_list":    [f"sci_{suffix_b}_0105"],
    #"fileout_sci": f"master_SO2026V-034_HD176986_{suffix_b}_0105",
    #"sci_list":    [f"sci_{suffix_b}_0106"],
    #"fileout_sci": f"master_SO2026V-034_HD176986_{suffix_b}_0106",
    #"sci_list":    [f"sci_{suffix_b}_0107"],
    #"fileout_sci": f"master_SO2026V-034_HD176986_{suffix_b}_0107",
    #"sci_list":    [f"sci_{suffix_b}_0108"],
    #"fileout_sci": f"master_SO2026V-034_HD176986_{suffix_b}_0108",
    #
    #"arc_list":  [f"thar_{suffix_b}_0109",
    #              f"thar_{suffix_b}_0110",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_b}_0109",
    #######################################################
    #"sci_list":    [f"sci_{suffix_b}_0111"],
    #"fileout_sci": f"master_SO2026V-034_HD192263_{suffix_b}_0111",
    #"sci_list":    [f"sci_{suffix_b}_0112"],
    #"fileout_sci": f"master_SO2026V-034_HD192263_{suffix_b}_0112",
    #"sci_list":    [f"sci_{suffix_b}_0113"],
    #"fileout_sci": f"master_SO2026V-034_HD192263_{suffix_b}_0113",
    #"sci_list":    [f"sci_{suffix_b}_0114"],
    #"fileout_sci": f"master_SO2026V-034_HD192263_{suffix_b}_0114",
    #"sci_list":    [f"sci_{suffix_b}_0115"],
    #"fileout_sci": f"master_SO2026V-034_HD192263_{suffix_b}_0115",
    #
    #"arc_list":  [f"thar_{suffix_b}_0116",
    #              f"thar_{suffix_b}_0117",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_b}_0116",
    #######################################################
    #"sci_list":    [f"sci_{suffix_b}_0118"],
    #"fileout_sci": f"master_SO2026V-034_TIC393818343_{suffix_b}_0118",
    #"sci_list":    [f"sci_{suffix_b}_0119"],
    #"fileout_sci": f"master_SO2026V-034_TIC393818343_{suffix_b}_0119",
    #"sci_list":    [f"sci_{suffix_b}_0120"],
    #"fileout_sci": f"master_SO2026V-034_TIC393818343_{suffix_b}_0120",
    #
    #"arc_list":  [f"thar_{suffix_b}_0121",
    #              f"thar_{suffix_b}_0122",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_b}_0121",
    #######################################################
    #"sci_list":    [f"sci_{suffix_b}_0123"],
    #"fileout_sci": f"master_SO2026V-053_epsInd_{suffix_b}_0123",
    #"sci_list":    [f"sci_{suffix_b}_0124"],
    #"fileout_sci": f"master_SO2026V-053_epsInd_{suffix_b}_0124",
    #"sci_list":    [f"sci_{suffix_b}_0125"],
    #"fileout_sci": f"master_SO2026V-053_epsInd_{suffix_b}_0125",
    #"sci_list":    [f"sci_{suffix_b}_0126"],
    #"fileout_sci": f"master_SO2026V-053_epsInd_{suffix_b}_0126",
    #
    #"arc_list":  [f"thar_{suffix_b}_0127",
    #              f"thar_{suffix_b}_0128",],
    #"fileout_arc": f"master_SO2026V-053_thar_{suffix_b}_0127",
    #######################################################
    #"sci_list":    [f"sci_{suffix_b}_0129"],
    #"fileout_sci": f"master_SO2026V-034_HD215152_{suffix_b}_0129",
    #"sci_list":    [f"sci_{suffix_b}_0130"],
    #"fileout_sci": f"master_SO2026V-034_HD215152_{suffix_b}_0130",
    #"sci_list":    [f"sci_{suffix_b}_0132"],
    #"fileout_sci": f"master_SO2026V-034_HD215152_{suffix_b}_0132",
    #
    #"arc_list":  [f"thar_{suffix_b}_0133",
    #              f"thar_{suffix_b}_0134",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_b}_0133", # REDO WAVE SOL
    #######################################################
    #"sci_list":    [f"sci_{suffix_b}_0135"],
    #"fileout_sci": f"master_SO2026V-034_HD216770_{suffix_b}_0135",
    #"sci_list":    [f"sci_{suffix_b}_0136"],
    #"fileout_sci": f"master_SO2026V-034_HD216770_{suffix_b}_0136",
    #"sci_list":    [f"sci_{suffix_b}_0137"],
    #"fileout_sci": f"master_SO2026V-034_HD216770_{suffix_b}_0137",
    #
    #"arc_list":  [f"thar_{suffix_b}_0138",
    #              f"thar_{suffix_b}_0139",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_b}_0138",
    #######################################################
    #"sci_list":    [f"sci_{suffix_b}_0140"],
    #"fileout_sci": f"master_SO2026V-034_HD224693_{suffix_b}_0140",
    #"sci_list":    [f"sci_{suffix_b}_0141"],
    #"fileout_sci": f"master_SO2026V-034_HD224693_{suffix_b}_0141",
    #"sci_list":    [f"sci_{suffix_b}_0142"],
    #"fileout_sci": f"master_SO2026V-034_HD224693_{suffix_b}_0142",
    #
    #"arc_list":  [f"thar_{suffix_b}_0143",
    #              f"thar_{suffix_b}_0144",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_b}_0143",
    #######################################################
    #"sci_list":    [f"std_{suffix_b}_0146",],
    #"fileout_sci": f"master_calib_specphot_HR9087_{suffix_b}_0146",
    #
    #"arc_list":  [f"thar_{suffix_b}_0147",
    #              f"thar_{suffix_b}_0148",],
    #"fileout_arc": f"master_calib_thar_{suffix_b}_0147", # DO NOT REPROCESS - SINGLE OBS
    #######################################################
    #"sci_list":    [f"stdRV_{suffix_b}_0149",],
    #"fileout_sci": f"master_calib_RV_HIP003001_{suffix_b}_0149",
    #
    #"arc_list":  [f"thar_{suffix_b}_0150",
    #              f"thar_{suffix_b}_0151",],
    #"fileout_arc": f"master_calib_thar_{suffix_b}_0150", # DO NOT REPROCESS - SINGLE OBS
    #######################################################
    #"sci_list":    [f"sci_{suffix_b}_0153"],
    #"fileout_sci": f"master_SO2026V-034_HD6434_{suffix_b}_0153",
    #"sci_list":    [f"sci_{suffix_b}_0154"],
    #"fileout_sci": f"master_SO2026V-034_HD6434_{suffix_b}_0154", # blue feature is vanishing starting on this observation!
    #"sci_list":    [f"sci_{suffix_b}_0155"],
    #"fileout_sci": f"master_SO2026V-034_HD6434_{suffix_b}_0155",
    #
    #"arc_list":  [f"thar_{suffix_b}_0156",
    #              f"thar_{suffix_b}_0157",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_b}_0156",
    #######################################################
    #"sci_list":    [f"sci_{suffix_b}_0158"],
    #"fileout_sci": f"master_SO2026V-034_HD10180_{suffix_b}_0158", # blue feature is absent on this observation!
    #"sci_list":    [f"sci_{suffix_b}_0159"],
    #"fileout_sci": f"master_SO2026V-034_HD10180_{suffix_b}_0159", # no data - skip this frame
    #"sci_list":    [f"sci_{suffix_b}_0162"],
    #"fileout_sci": f"master_SO2026V-034_HD10180_{suffix_b}_0162",
    #
    #"arc_list":  [f"thar_{suffix_b}_0163",
    #              f"thar_{suffix_b}_0164",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_b}_0163",
    #######################################################
    #"sci_list":    [f"sci_{suffix_b}_0165"],
    #"fileout_sci": f"master_SO2026V-034_GJ86_{suffix_b}_0165",
    #"sci_list":    [f"sci_{suffix_b}_0166"],
    #"fileout_sci": f"master_SO2026V-034_GJ86_{suffix_b}_0166",
    "sci_list":    [f"sci_{suffix_b}_0167"],
    "fileout_sci": f"master_SO2026V-034_GJ86_{suffix_b}_0167",
    #
    "arc_list":  [f"thar_{suffix_b}_0168",
                  f"thar_{suffix_b}_0169",],
    "fileout_arc": f"master_SO2026V-034_thar_{suffix_b}_0168",
    #######################################################
    # arc scaling parameters
    "saturation_threshold": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    "aperture_extract": [-15,+11], # blue 2026-10-09
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    "wavesolution_peak_offset_threshold": 1.5, # sigma-threshold for removing lines that deviates too much from the reference position
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    #"raw_directory":  "D:/SOAR/Steles/2026-10-09/", # set full path contaninig the raw data
    "raw_directory":  "/home/fnavarete/astro/data/SOAR/Steles/2026-10-09/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,   # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": [f"q20w_{suffix_r}_0017",
                  f"q20w_{suffix_r}_0018",
                  f"q20w_{suffix_r}_0019",
                  f"q20w_{suffix_r}_0020",
                  f"q20w_{suffix_r}_0021",
                  f"q20w_{suffix_r}_0022",
                  f"q20w_{suffix_r}_0023",
                  f"q20w_{suffix_r}_0024",
                  f"q20w_{suffix_r}_0025",], # 9s
    #"longflat_list": ["q20w_B_0035","q20w_B_0036","q20w_B_0037"], # 900s
    #"arc_list":  [f"thar_{suffix_r}_0103",
    #              f"thar_{suffix_r}_0104",],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": f"master_flat_{suffix_r}",
    #"fileout_longflat": "master_longflat_B",
    #"fileout_arc":  f"master_thar_{suffix_r}_0103",
    #
    #"sci_list":    [f"sci_{suffix_r}_0102"],
    #"fileout_sci": F"master_HD216956_tracing_{suffix_r}_0102",
    #
    #"sci_list":    [f"sci_{suffix_r}_0100"],
    #"fileout_sci": F"master_HD216956_{suffix_r}_0100",
    #
    ##################################################################################
    #"sci_list":    [f"sci_{suffix_r}_0105"],
    #"fileout_sci": f"master_SO2026V-034_HD176986_{suffix_r}_0105",
    #"sci_list":    [f"sci_{suffix_r}_0106"],
    #"fileout_sci": f"master_SO2026V-034_HD176986_{suffix_r}_0106",
    #"sci_list":    [f"sci_{suffix_r}_0107"],
    #"fileout_sci": f"master_SO2026V-034_HD176986_{suffix_r}_0107",
    #"sci_list":    [f"sci_{suffix_r}_0108"],
    #"fileout_sci": f"master_SO2026V-034_HD176986_{suffix_r}_0108",
    #
    #"arc_list":  [f"thar_{suffix_r}_0109",
    #              f"thar_{suffix_r}_0110",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_r}_0109",
    #######################################################
    #"sci_list":    [f"sci_{suffix_r}_0111"],
    #"fileout_sci": f"master_SO2026V-034_HD192263_{suffix_r}_0111",
    #"sci_list":    [f"sci_{suffix_r}_0112"],
    #"fileout_sci": f"master_SO2026V-034_HD192263_{suffix_r}_0112",
    #"sci_list":    [f"sci_{suffix_r}_0113"],
    #"fileout_sci": f"master_SO2026V-034_HD192263_{suffix_r}_0113",
    #"sci_list":    [f"sci_{suffix_r}_0114"],
    #"fileout_sci": f"master_SO2026V-034_HD192263_{suffix_r}_0114",
    #"sci_list":    [f"sci_{suffix_r}_0115"],
    #"fileout_sci": f"master_SO2026V-034_HD192263_{suffix_r}_0115",
    #
    #"arc_list":  [f"thar_{suffix_r}_0116",
    #              f"thar_{suffix_r}_0117",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_r}_0116",
    #######################################################
    #"sci_list":    [f"sci_{suffix_r}_0118"],
    #"fileout_sci": f"master_SO2026V-034_TIC393818343_{suffix_r}_0118",
    #"sci_list":    [f"sci_{suffix_r}_0119"],
    #"fileout_sci": f"master_SO2026V-034_TIC393818343_{suffix_r}_0119",
    #"sci_list":    [f"sci_{suffix_r}_0120"],
    #"fileout_sci": f"master_SO2026V-034_TIC393818343_{suffix_r}_0120",
    #
    #"arc_list":  [f"thar_{suffix_r}_0121",
    #              f"thar_{suffix_r}_0122",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_r}_0121",   # REDO WAVELENGTH SOLUTION!!!!
    #######################################################  
    #"sci_list":    [f"sci_{suffix_r}_0123"],
    #"fileout_sci": f"master_SO2026V-053_epsInd_{suffix_r}_0123",
    #"sci_list":    [f"sci_{suffix_r}_0124"],
    #"fileout_sci": f"master_SO2026V-053_epsInd_{suffix_r}_0124",
    #"sci_list":    [f"sci_{suffix_r}_0125"],
    #"fileout_sci": f"master_SO2026V-053_epsInd_{suffix_r}_0125",
    #"sci_list":    [f"sci_{suffix_r}_0126"],
    #"fileout_sci": f"master_SO2026V-053_epsInd_{suffix_r}_0126",
    #
    #"arc_list":  [f"thar_{suffix_r}_0127",
    #              f"thar_{suffix_r}_0128",],
    #"fileout_arc": f"master_SO2026V-053_thar_{suffix_r}_0127",
    #######################################################
    #"sci_list":    [f"sci_{suffix_r}_0129"],
    #"fileout_sci": f"master_SO2026V-034_HD215152_{suffix_r}_0129",
    #"sci_list":    [f"sci_{suffix_r}_0130"],
    #"fileout_sci": f"master_SO2026V-034_HD215152_{suffix_r}_0130",
    #"sci_list":    [f"sci_{suffix_r}_0132"],
    #"fileout_sci": f"master_SO2026V-034_HD215152_{suffix_r}_0132",
    #
    #"arc_list":  [f"thar_{suffix_r}_0133",
    #              f"thar_{suffix_r}_0134",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_r}_0133",
    #######################################################
    #"sci_list":    [f"sci_{suffix_r}_0135"],
    #"fileout_sci": f"master_SO2026V-034_HD216770_{suffix_r}_0135",
    #"sci_list":    [f"sci_{suffix_r}_0136"],
    #"fileout_sci": f"master_SO2026V-034_HD216770_{suffix_r}_0136",
    #"sci_list":    [f"sci_{suffix_r}_0137"],
    #"fileout_sci": f"master_SO2026V-034_HD216770_{suffix_r}_0137",
    #
    #"arc_list":  [f"thar_{suffix_r}_0138",
    #              f"thar_{suffix_r}_0139",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_r}_0138",
    #######################################################  
    #"sci_list":    [f"sci_{suffix_r}_0140"],
    #"fileout_sci": f"master_SO2026V-034_HD224693_{suffix_r}_0140",
    #"sci_list":    [f"sci_{suffix_r}_0141"],
    #"fileout_sci": f"master_SO2026V-034_HD224693_{suffix_r}_0141",
    #"sci_list":    [f"sci_{suffix_r}_0142"],
    #"fileout_sci": f"master_SO2026V-034_HD224693_{suffix_r}_0142",
    #
    #"arc_list":  [f"thar_{suffix_r}_0143",
    #              f"thar_{suffix_r}_0144",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_r}_0143",
    #######################################################
    #"sci_list":    [f"std_{suffix_r}_0146",],
    #"fileout_sci": f"master_calib_specphot_HR9087_{suffix_r}_0146",
    #
    #"arc_list":  [f"thar_{suffix_r}_0147",
    #              f"thar_{suffix_r}_0148",],
    #"fileout_arc": f"master_calib_thar_{suffix_r}_0147",
    #######################################################
    #"sci_list":    [f"stdRV_{suffix_r}_0149",],
    #"fileout_sci": f"master_calib_RV_HIP003001_{suffix_r}_0149",
    #
    #"arc_list":  [f"thar_{suffix_r}_0150",
    #              f"thar_{suffix_r}_0151",],
    #"fileout_arc": f"master_calib_thar_{suffix_r}_0150",
    #######################################################
    #"sci_list":    [f"sci_{suffix_r}_0153"],
    #"fileout_sci": f"master_SO2026V-034_HD6434_{suffix_r}_0153",
    #"sci_list":    [f"sci_{suffix_r}_0154"],
    #"fileout_sci": f"master_SO2026V-034_HD6434_{suffix_r}_0154",
    #"sci_list":    [f"sci_{suffix_r}_0155"],
    #"fileout_sci": f"master_SO2026V-034_HD6434_{suffix_r}_0155",
    #
    #"arc_list":  [f"thar_{suffix_r}_0156",
    #              f"thar_{suffix_r}_0157",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_r}_0156",
    #######################################################
    #"sci_list":    [f"sci_{suffix_r}_0158"],
    #"fileout_sci": f"master_SO2026V-034_HD10180_{suffix_r}_0158",
    #"sci_list":    [f"sci_{suffix_r}_0161"],
    #"fileout_sci": f"master_SO2026V-034_HD10180_{suffix_r}_0161",
    #"sci_list":    [f"sci_{suffix_r}_0162"],
    #"fileout_sci": f"master_SO2026V-034_HD10180_{suffix_r}_0162",
    #
    #"arc_list":  [f"thar_{suffix_r}_0163",
    #              f"thar_{suffix_r}_0164",],
    #"fileout_arc": f"master_SO2026V-034_thar_{suffix_r}_0163",
    #######################################################
    #"sci_list":    [f"sci_{suffix_r}_0165"],
    #"fileout_sci": f"master_SO2026V-034_GJ86_{suffix_r}_0165",
    #"sci_list":    [f"sci_{suffix_r}_0166"],
    #"fileout_sci": f"master_SO2026V-034_GJ86_{suffix_r}_0166",
    "sci_list":    [f"sci_{suffix_r}_0167"],
    "fileout_sci": f"master_SO2026V-034_GJ86_{suffix_r}_0167",
    #
    "arc_list":  [f"thar_{suffix_r}_0168",
                  f"thar_{suffix_r}_0169",],
    "fileout_arc": f"master_SO2026V-034_thar_{suffix_r}_0168",
    #######################################################
    # arc scaling parameters
    "saturation_threshold": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 38],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    "aperture_extract": [-10,+5], # red 2026-10-09
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    "wavesolution_peak_offset_threshold": 1.0, # sigma-threshold for removing lines that deviates too much from the reference position
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

#pipe_b = STELES_Pipeline(config_blue)
#pipe_b.run()
#pipe_b.run_trace()
#pipe_b.run_extract_science()
#pipe_b.run_extract_arc_science()
#pipe_b.run_extract_all()

#pipe_r = STELES_Pipeline(config_red)
#pipe_r.run()
#pipe_r.run_extract_science()
#pipe_r.run_extract_arc_science()
#pipe_r.run_extract_all()



In [ ]:
def print_wv_px(n_order, ThAr_wavelengths, ThAr_pixels):
    for i,wv in enumerate(ThAr_wavelengths):
        print(f"{n_order}\t{wv}\t{ThAr_pixels[i]}")

# Orders blue channel
n_order=2
ThAr_wavelengths = [  9399.0890, 9431.5996, 9461.0279, 9470.6819, 9474.8793, 9497.1891, 9505.3930, 9508.4513, 9548.0303]
ThAr_pixels      = [        276,       556,       828,       923,       963,      1191,      1278,      1313,      1776]
ThAr_pixels      = [        276,       556,       828,       923,       963,      1191,      1278,      1313,      1776]
wavelength_pixel_offset = -2

ThAr_pixels = np.array(ThAr_pixels)#.to_numpy().copy()
ThAr_pixels += wavelength_pixel_offset # add a constant offset to match the pixel position
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

pipe = STELES_Pipeline_WaveSolution_Test(config_red)
order=n_order-1
pipe.run_wavesol_debug(order, ThAr_wavelengths, ThAr_pixels, print_orders=False, poly_degree=1)

In [ ]:
def print_wv_px(n_order, ThAr_wavelengths, ThAr_pixels):
    for i,wv in enumerate(ThAr_wavelengths):
        print(f"{n_order}\t{wv}\t{ThAr_pixels[i]}")

# Orders blue channel
n_order=2
ThAr_wavelengths = [  9399.0890, 9431.5996, 9461.0279, 9470.6819, 9474.8793, 9497.1891, 9505.3930, 9508.4513, 9548.0303]
ThAr_pixels      = [        276,       556,       828,       923,       963,      1191,      1278,      1313,      1776]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=3
ThAr_wavelengths = [ 9203.9617, 9266.2070, 9276.2732, 9289.5620, 9291.5310, 9354.2200, 9383.2722,]
ThAr_pixels      = [       219,       777,       873,      1006,      1027,      1739,      2138,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=4
ThAr_wavelengths = [ 9048.2500, 9094.8290, 9194.6380, 9203.9620, 9224.4990, ]
ThAr_pixels      = [       434,       880,      2068,      2209,      2547, ]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=5
ThAr_wavelengths = [8849.9800, 8868.8330, 8892.9870, 8905.6580, 8949.1230, 8955.8470, 8967.6400, 9016.5900, 9017.5910, 9045.3530, 9048.2500,]
ThAr_pixels      = [      218,       386,       611,       735,      1190,      1266,      1403,      2050,      2065,      2517,      2568,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=6
ThAr_wavelengths = [ 8709.2341, 8732.4240, 8748.0309, 8758.2434, 8761.6862, 8799.0875, 8820.4210, 8868.8334,]
ThAr_pixels      = [       459,       683,       842,       949,       986,      1411,      1681,      2412,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=7
ThAr_wavelengths = [ 8539.7930, 8554.9440, 8573.1210, 8605.7760, 8620.4600, 8639.4420, 8645.3090, 8665.4860, 8678.4080,]
ThAr_pixels      = [       382,       527,       709,      1056,      1223,      1452,      1527,      1796,      1984,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=8
ThAr_wavelengths = [ 8384.7240, 8416.7270, 8446.5120, 8464.2370, 8478.3580, 8490.3065, 8510.6240, 8539.7930,]
ThAr_pixels      = [       385,       704,      1024,      1230,      1402,      1556,      1836,      2296,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=9
ThAr_wavelengths = [ 8252.3940, 8275.6270, 8320.8550, 8330.4490, 8358.7260, 8369.3400, 8379.2260, 8384.7240,]
ThAr_pixels      = [       559,       803,      1325,      1447,      1836,      1998,      2157,      2251,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=10
ThAr_wavelengths = [ 8085.2190, 8119.1810, 8129.4051, 8138.4753, 8159.7280, 8169.7865, 8186.9110, 8252.3940,]
ThAr_pixels      = [       333,       681,       792,       892,      1139,      1261,      1481,      2525,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=11
ThAr_wavelengths = [ 7978.9730, 7987.9730, 8032.4310, 8037.2180, 8046.1169, 8053.3090, 8075.6518, 8085.2190,]
ThAr_pixels      = [       679,       778,      1307,      1367,      1486,      1585,      1916,      2071,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=12
ThAr_wavelengths = [  7798.3580, 7817.7669, 7847.5390, 7865.9700, 7886.2830, 7891.0750, 7916.4420, 7937.7340, 7954.5920,]
ThAr_pixels      = [   209,      403,      722,      932,     1178,     1238,     1579,     1899,     2180,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=13
ThAr_wavelengths = [ 7670.0570, 7678.1270, 7685.3075, 7710.2692, 7771.9468, 7788.9340, 7817.7670,]
ThAr_pixels      = [       225,       307,       381,       650,      1404,      1644,      2106,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=14
ThAr_wavelengths = [ 7549.3140, 7567.7420, 7585.7920, 7589.3151, 7628.8818, 7647.3790, 7670.0575, 7678.1270, 7685.3070, 7704.8169,]
ThAr_pixels      = [       277,       472,       672,       712,      1194,      1443,      1779,      1909,      2030,      2392,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=15
ThAr_wavelengths = [ 7425.2940, 7428.9400, 7430.2530, 7435.3680, 7447.8490, 7471.1640, 7481.3545, 7484.3267, 7549.3140, 7567.7420, 7569.5115, 7585.7920,]
ThAr_pixels      = [       254,       293,       306,       360,       496,       762,       885,       921,      1827,      2145,      2177,     2499,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=16
ThAr_wavelengths = [ 7311.7160, 7316.0050, 7328.2850, 7353.2930, 7372.1184, 7412.3368, 7425.2942, 7428.9400, 7430.2530, 7435.3680, 7436.2970, 7444.7490, 7447.8490,]
ThAr_pixels      = [       300,       346,       482,       771,      1004,      1557,      1758,      1818,      1840,      1927,      1942,      2092,     2148,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=17
ThAr_wavelengths = [ 7200.0450, 7212.6900, 7218.0540, 7242.0919, 7265.1720, 7296.2659, 7298.1434, 7324.8073, 7326.1491, 7328.2850, 7329.4916, 7341.1510,]
ThAr_pixels      = [       329,       470,       531,       817,      1112,      1554,      1583,      2025,      2049,      2088,     2112,     2338,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=18
ThAr_wavelengths = [ 7084.1690, 7086.7044, 7107.4780, 7124.5610, 7150.2840, 7173.3725, 7191.1328, 7200.0450, 7208.0060, 7212.6900, 7218.0540, 7219.1515, 7229.9386,]
ThAr_pixels      = [       273,       301,       537,       741,      1071,      1393,      1666,      1812,      1950,      2035,      2135,      2157,      2374,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=19
ThAr_wavelengths = [ 6989.6550, 7000.8040, 7018.5680, 7030.2510, 7045.7970, 7060.6538, 7068.7360, 7084.1690, 7086.7044, 7124.5610, 7125.8200,]
ThAr_pixels      = [       419,       549,       766,       915,      1124,      1334,      1454,      1698,      1740,      2465,      2493,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)


n_order=20
ThAr_wavelengths = [6871.2890, 6874.7530, 6888.1740, 6911.2260, 6937.6640, 6943.6105, 6951.4776, 6989.6550, 7000.8040, 7002.8830,]
ThAr_pixels      = [      258,       297,       453,       734,      1082,      1165,      1278,      1892,      2100,      2141,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=21
ThAr_wavelengths = [ 6778.3120, 6791.2351, 6824.6770, 6829.0350, 6834.9250, 6861.2690, 6871.2890, 6874.7530, 6879.5824, 6888.1740, 6911.2264,]
ThAr_pixels      = [       352,       506,       932,       991,      1074,      1461,      1622,      1680,      1762,      1916,      2378,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=22
ThAr_wavelengths = [6677.2820, 6711.2521, 6713.9700, 6719.1990, 6727.4580, 6738.1800, 6752.8330, 6766.6120, 6778.3120, 6791.2350,]
ThAr_pixels      = [      319,       739,       775,       844,       956,      1106,      1321,      1538,      1735,      1970,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=23
ThAr_wavelengths = [ 6583.9060, 6588.5400, 6591.4840, 6593.9390, 6604.8530, 6619.9460, 6638.2207, 6639.7400, 6643.6980, 6658.6774, 6677.2817, 6711.2521, 6713.9700,]
ThAr_pixels      = [       344,       400,       436,       466,       603,       800,      1052,      1074,      1132,      1358,      1662,      2321,      2382,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=24
ThAr_wavelengths = [ 6490.7372, 6509.0503, 6512.3640, 6522.0432, 6531.3420, 6554.1600, 6558.8756, 6577.2150, 6583.9060, 6588.5400, 6591.4840, 6593.9390, 6604.8530,]
ThAr_pixels      = [       339,       568,       610,       738,       864,      1192,      1264,      1558,     1673,     1756,      1809,     1855,     2067,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=25
ThAr_wavelengths = [ 6411.8990, 6416.3070, 6431.5550, 6457.2820, 6461.0596, 6483.0825, 6490.7370, 6512.3640, 6522.0432,]
ThAr_pixels      = [       481,       538,       739,      1104,     1244,     1511,     1643,     2052,     2257,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=26
ThAr_wavelengths = [ 6307.6570, 6327.2780, 6337.6201, 6342.8590, 6376.9310, 6387.3960, 6400.6960, 6406.4460, 6411.8990, 6416.3070, 6431.5550,]
ThAr_pixels      = [       276,       523,       661,       731,      1228,      1397,      1623,      1728,      1828,     1913,     2232,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=27
ThAr_wavelengths = [ 6234.8550, 6257.4240, 6261.4180, 6274.1170, 6300.9170, 6307.6570, 6327.2780, 6342.8590,]
ThAr_pixels      = [       433,      736,      791,      974,     1393,     1507,     1867,     2192,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=28
ThAr_wavelengths = [ 6145.4410, 6151.9930, 6164.4800, 6172.2780, 6198.2230, 6215.9380, 6224.5270, 6226.3700, 6234.8550, 6257.4240, 6261.4180,]
ThAr_pixels      = [       353,      438,      604,      712,     1092,     1377,     1525,     1557,     1712,     2171,     2262,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=29
ThAr_wavelengths = [6059.3730, 6085.3750, 6088.0300, 6114.9230, 6138.6460, 6145.4410, 6151.9930, 6178.4315, 6182.6217,]
ThAr_pixels      = [      286,       632,       669,      1065,      1454,      1575,      1696,      2247,      2347,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=30
ThAr_wavelengths = [5994.1290, 6010.1610, 6021.0360, 6032.1270, 6049.0510, 6059.3730, 6069.0210, 6073.1040, 6085.3750, 6088.0300,]
ThAr_pixels      = [      465,       687,       845,      1013,      1285,      1462,      1638,      1715,      1962,      2019,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=31
ThAr_wavelengths = [ 5912.0853, 5914.6710, 5944.6470, 5973.6649, 5975.0648, 5994.1290, 6010.1610,]
ThAr_pixels      = [       395,       430,       859 ,     1323,      1347,      1694,      2023,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=32
ThAr_wavelengths = [5860.3100, 5863.7180, 5882.6242, 5885.7020, 5905.5710, 5912.0850, 5914.6710, 5918.9440, 5928.8130, 5929.9343, ]
ThAr_pixels      = [      721,       772,      1064,      1114,      1455,      1575,      1624,      1707,      1908,      1932, ]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=33
ThAr_wavelengths = [5760.5510, 5789.6450, 5800.8300, 5804.1410, 5815.4220, 5832.3700, 5853.4750, 5854.1210, 5860.3100, 5863.7180,]
ThAr_pixels      = [      345,       760,       932,       985,      1169,      1467,      1882,      1896,      2030,      2108,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=34
ThAr_wavelengths = [5720.1830, 5725.3890, 5739.5200, 5753.0260, 5760.5508, 5763.5290, 5773.9460, 5789.6450, 5800.8300,]
ThAr_pixels      = [      792,      872,     1100,     1333,     1470,     1526,     1731,     2068,     2340,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=35
ThAr_wavelengths = [5650.7040, 5665.1800, 5677.0530, 5700.9180, 5707.1030, 5719.6227, 5720.1828, 5725.3890,]
ThAr_pixels      = [      796,     1028,     1230,     1676,     1804,     2082,     2095,     2220,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=36
ThAr_wavelengths = [5587.0260, 5588.7506, 5595.0635, 5601.6030, 5612.0679, 5615.3190, 5639.7490, 5650.7040,]
ThAr_pixels      = [      866,       893,       997,      1107,      1291,      1350,      1834,      2081,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=37
ThAr_wavelengths = [5499.2550, 5539.2620, 5548.1760, 5557.0450, 5573.3540, 5587.0263,]
ThAr_pixels      = [      537,      1183,      1344,      1514,      1850,      2167,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=38
ThAr_wavelengths = [5451.6520, 5464.2055, 5495.8738, 5499.2550, 5504.3020, 5506.1130, 5509.9940, 5514.8731,]
ThAr_pixels      = [      803,      1011,      1597,      1667,      1774,      1813,      1899,      2012,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=39
ThAr_wavelengths = [5403.1993, 5415.5226, 5417.4858, 5431.1120, 5451.6520, 5452.2187, 5464.2055,]
ThAr_pixels      = [     1044,      1277,      1313,      1579,      2033,      2047,      2389,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=40
ThAr_wavelengths = [5317.4940, 5326.9756, 5343.5810, 5360.1501, 5374.8219, 5382.9280, 5386.6107,]
ThAr_pixels      = [      662,       815,      1098,      1409,      1710,      1890,      1977,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

In [ ]:
#2026-08-29 (pre-SV)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-08-29_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,   # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["q20w_B_0010","q20w_B_0011","q20w_B_0012","q20w_B_0013","q20w_B_0014"],
    #"sci_list":  ["HD175191_B_0079"],
    "sci_list":  ["HD175191_B_0082"],
    "arc_list":  ["HD175191_thar_B_0087", "HD175191_thar_B_0088"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat_B",
    #"fileout_sci":  "master_HD175191_B_0079",
    "fileout_sci":  "master_HD175191_B_0082",
    "fileout_arc":  "master_HD175191_thar_B_0087",
    # arc scaling parameters
    "saturation_threshold": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    "aperture_extract": [-10,+13], # blue 2026-01-29
    "aperture_extract": [-13,+13], # blue 2026-08-29
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-08-29_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,   # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["q20w_R_0010","q20w_R_0011","q20w_R_0012","q20w_R_0013","q20w_R_0014",
                  "q20w_R_0015","q20w_R_0016","q20w_R_0017","q20w_R_0018","q20w_R_0019",
                  "q20w_R_0020","q20w_R_0021","q20w_R_0022","q20w_R_0023","q20w_R_0024",
                  "q20w_R_0025","q20w_R_0026","q20w_R_0027","q20w_R_0028","q20w_R_0029"],
    #"sci_list":  ["HD175191_R_0079"],
    "sci_list":  ["HD175191_R_0082"],
    "arc_list":  ["HD175191_thar_R_0087", "HD175191_thar_R_0088"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat_R",
    "fileout_sci":  "master_HD175191_R_0082",
    "fileout_arc":  "master_HD175191_thar_R_0087",
    # name of master frames (stored in the <RED> directory)
    # arc scaling parameters
    "saturation_threshold": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    "aperture_extract": [-9,+8], # blue 2026-08-29
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:
def print_wv_px(n_order, ThAr_wavelengths, ThAr_pixels):
    for i,wv in enumerate(ThAr_wavelengths):
        print(f"{n_order}\t{wv}\t{ThAr_pixels[i]}")

# Orders blue channel
n_order=2
ThAr_wavelengths = [  9399.0890, 9431.5996, 9461.0279, 9470.6819, 9474.8793, 9497.1891, 9505.3930, 9508.4513, 9548.0303]
ThAr_pixels      = [        276,       556,       828,       923,       963,      1191,      1278,      1313,      1776]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=3
ThAr_wavelengths = [ 9203.9617, 9266.2070, 9276.2732, 9289.5620, 9291.5310, 9354.2200, 9383.2722,]
ThAr_pixels      = [       219,       777,       873,      1006,      1027,      1739,      2138,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=4
ThAr_wavelengths = [ 9048.2500, 9094.8290, 9194.6380, 9203.9620, 9224.4990, ]
ThAr_pixels      = [       434,       880,      2068,      2209,      2547, ]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=5
ThAr_wavelengths = [8849.9800, 8868.8330, 8892.9870, 8905.6580, 8949.1230, 8955.8470, 8967.6400, 9016.5900, 9017.5910, 9045.3530, 9048.2500,]
ThAr_pixels      = [      218,       386,       611,       735,      1190,      1266,      1403,      2050,      2065,      2517,      2568,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=6
ThAr_wavelengths = [ 8709.2341, 8732.4240, 8748.0309, 8758.2434, 8761.6862, 8799.0875, 8820.4210, 8868.8334,]
ThAr_pixels      = [       459,       683,       842,       949,       986,      1411,      1681,      2412,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=7
ThAr_wavelengths = [ 8539.7930, 8554.9440, 8573.1210, 8605.7760, 8620.4600, 8639.4420, 8645.3090, 8665.4860, 8678.4080,]
ThAr_pixels      = [       382,       527,       709,      1056,      1223,      1452,      1527,      1796,      1984,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=8
ThAr_wavelengths = [ 8384.7240, 8416.7270, 8446.5120, 8464.2370, 8478.3580, 8490.3065, 8510.6240, 8539.7930,]
ThAr_pixels      = [       385,       704,      1024,      1230,      1402,      1556,      1836,      2296,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=9
ThAr_wavelengths = [ 8252.3940, 8275.6270, 8320.8550, 8330.4490, 8358.7260, 8369.3400, 8379.2260, 8384.7240,]
ThAr_pixels      = [       559,       803,      1325,      1447,      1836,      1998,      2157,      2251,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=10
ThAr_wavelengths = [ 8085.2190, 8119.1810, 8129.4051, 8138.4753, 8159.7280, 8169.7865, 8186.9110, 8252.3940,]
ThAr_pixels      = [       333,       681,       792,       892,      1139,      1261,      1481,      2525,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=11
ThAr_wavelengths = [ 7978.9730, 7987.9730, 8032.4310, 8037.2180, 8046.1169, 8053.3090, 8075.6518, 8085.2190,]
ThAr_pixels      = [       679,       778,      1307,      1367,      1486,      1585,      1916,      2071,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=12
ThAr_wavelengths = [  7798.3580, 7817.7669, 7847.5390, 7865.9700, 7886.2830, 7891.0750, 7916.4420, 7937.7340, 7954.5920,]
ThAr_pixels      = [   209,      403,      722,      932,     1178,     1238,     1579,     1899,     2180,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=13
ThAr_wavelengths = [ 7670.0570, 7678.1270, 7685.3075, 7710.2692, 7771.9468, 7788.9340, 7817.7670,]
ThAr_pixels      = [       225,       307,       381,       650,      1404,      1644,      2106,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=14
ThAr_wavelengths = [ 7549.3140, 7567.7420, 7585.7920, 7589.3151, 7628.8818, 7647.3790, 7670.0575, 7678.1270, 7685.3070, 7704.8169,]
ThAr_pixels      = [       277,       472,       672,       712,      1194,      1443,      1779,      1909,      2030,      2392,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=15
ThAr_wavelengths = [ 7425.2940, 7428.9400, 7430.2530, 7435.3680, 7447.8490, 7471.1640, 7481.3545, 7484.3267, 7549.3140, 7567.7420, 7569.5115, 7585.7920,]
ThAr_pixels      = [       254,       293,       306,       360,       496,       762,       885,       921,      1827,      2145,      2177,     2499,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=16
ThAr_wavelengths = [ 7311.7160, 7316.0050, 7328.2850, 7353.2930, 7372.1184, 7412.3368, 7425.2942, 7428.9400, 7430.2530, 7435.3680, 7436.2970, 7444.7490, 7447.8490,]
ThAr_pixels      = [       300,       346,       482,       771,      1004,      1557,      1758,      1818,      1840,      1927,      1942,      2092,     2148,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=17
ThAr_wavelengths = [ 7200.0450, 7212.6900, 7218.0540, 7242.0919, 7265.1720, 7296.2659, 7298.1434, 7324.8073, 7326.1491, 7328.2850, 7329.4916, 7341.1510,]
ThAr_pixels      = [       329,       470,       531,       817,      1112,      1554,      1583,      2025,      2049,      2088,     2112,     2338,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=18
ThAr_wavelengths = [ 7084.1690, 7086.7044, 7107.4780, 7124.5610, 7150.2840, 7173.3725, 7191.1328, 7200.0450, 7208.0060, 7212.6900, 7218.0540, 7219.1515, 7229.9386,]
ThAr_pixels      = [       273,       301,       537,       741,      1071,      1393,      1666,      1812,      1950,      2035,      2135,      2157,      2374,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

n_order=19
ThAr_wavelengths = [ 6989.6550, 7000.8040, 7018.5680, 7030.2510, 7045.7970, 7060.6538, 7068.7360, 7084.1690, 7086.7044, 7124.5610, 7125.8200,]
ThAr_pixels      = [       419,       549,       766,       915,      1124,      1334,      1454,      1698,      1740,      2465,      2493,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)


n_order=20
ThAr_wavelengths = [6871.2890, 6874.7530, 6888.1740, 6911.2260, 6937.6640, 6943.6105, 6951.4776, 6989.6550, 7000.8040, 7002.8830,]
ThAr_pixels      = [      258,       297,       453,       734,      1082,      1165,      1278,      1892,      2100,      2141,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=21
ThAr_wavelengths = [ 6778.3120, 6791.2351, 6824.6770, 6829.0350, 6834.9250, 6861.2690, 6871.2890, 6874.7530, 6879.5824, 6888.1740, 6911.2264,]
ThAr_pixels      = [       352,       506,       932,       991,      1074,      1461,      1622,      1680,      1762,      1916,      2378,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=22
ThAr_wavelengths = [6677.2820, 6711.2521, 6713.9700, 6719.1990, 6727.4580, 6738.1800, 6752.8330, 6766.6120, 6778.3120, 6791.2350,]
ThAr_pixels      = [      319,       739,       775,       844,       956,      1106,      1321,      1538,      1735,      1970,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=23
ThAr_wavelengths = [ 6583.9060, 6588.5400, 6591.4840, 6593.9390, 6604.8530, 6619.9460, 6638.2207, 6639.7400, 6643.6980, 6658.6774, 6677.2817, 6711.2521, 6713.9700,]
ThAr_pixels      = [       344,       400,       436,       466,       603,       800,      1052,      1074,      1132,      1358,      1662,      2321,      2382,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=24
ThAr_wavelengths = [ 6490.7372, 6509.0503, 6512.3640, 6522.0432, 6531.3420, 6554.1600, 6558.8756, 6577.2150, 6583.9060, 6588.5400, 6591.4840, 6593.9390, 6604.8530,]
ThAr_pixels      = [       339,       568,       610,       738,       864,      1192,      1264,      1558,     1673,     1756,      1809,     1855,     2067,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=25
ThAr_wavelengths = [ 6411.8990, 6416.3070, 6431.5550, 6457.2820, 6461.0596, 6483.0825, 6490.7370, 6512.3640, 6522.0432,]
ThAr_pixels      = [       481,       538,       739,      1104,     1244,     1511,     1643,     2052,     2257,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=26
ThAr_wavelengths = [ 6307.6570, 6327.2780, 6337.6201, 6342.8590, 6376.9310, 6387.3960, 6400.6960, 6406.4460, 6411.8990, 6416.3070, 6431.5550,]
ThAr_pixels      = [       276,       523,       661,       731,      1228,      1397,      1623,      1728,      1828,     1913,     2232,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=27
ThAr_wavelengths = [ 6234.8550, 6257.4240, 6261.4180, 6274.1170, 6300.9170, 6307.6570, 6327.2780, 6342.8590,]
ThAr_pixels      = [       433,      736,      791,      974,     1393,     1507,     1867,     2192,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=28
ThAr_wavelengths = [ 6145.4410, 6151.9930, 6164.4800, 6172.2780, 6198.2230, 6215.9380, 6224.5270, 6226.3700, 6234.8550, 6257.4240, 6261.4180,]
ThAr_pixels      = [       353,      438,      604,      712,     1092,     1377,     1525,     1557,     1712,     2171,     2262,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=29
ThAr_wavelengths = [6059.3730, 6085.3750, 6088.0300, 6114.9230, 6138.6460, 6145.4410, 6151.9930, 6178.4315, 6182.6217,]
ThAr_pixels      = [      286,       632,       669,      1065,      1454,      1575,      1696,      2247,      2347,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=30
ThAr_wavelengths = [5994.1290, 6010.1610, 6021.0360, 6032.1270, 6049.0510, 6059.3730, 6069.0210, 6073.1040, 6085.3750, 6088.0300,]
ThAr_pixels      = [      465,       687,       845,      1013,      1285,      1462,      1638,      1715,      1962,      2019,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=31
ThAr_wavelengths = [ 5912.0853, 5914.6710, 5944.6470, 5973.6649, 5975.0648, 5994.1290, 6010.1610,]
ThAr_pixels      = [       395,       430,       859 ,     1323,      1347,      1694,      2023,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=32
ThAr_wavelengths = [5860.3100, 5863.7180, 5882.6242, 5885.7020, 5905.5710, 5912.0850, 5914.6710, 5918.9440, 5928.8130, 5929.9343, ]
ThAr_pixels      = [      721,       772,      1064,      1114,      1455,      1575,      1624,      1707,      1908,      1932, ]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=33
ThAr_wavelengths = [5760.5510, 5789.6450, 5800.8300, 5804.1410, 5815.4220, 5832.3700, 5853.4750, 5854.1210, 5860.3100, 5863.7180,]
ThAr_pixels      = [      345,       760,       932,       985,      1169,      1467,      1882,      1896,      2030,      2108,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=34
ThAr_wavelengths = [5720.1830, 5725.3890, 5739.5200, 5753.0260, 5760.5508, 5763.5290, 5773.9460, 5789.6450, 5800.8300,]
ThAr_pixels      = [      792,      872,     1100,     1333,     1470,     1526,     1731,     2068,     2340,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=35
ThAr_wavelengths = [5650.7040, 5665.1800, 5677.0530, 5700.9180, 5707.1030, 5719.6227, 5720.1828, 5725.3890,]
ThAr_pixels      = [      796,     1028,     1230,     1676,     1804,     2082,     2095,     2220,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=36
ThAr_wavelengths = [5587.0260, 5588.7506, 5595.0635, 5601.6030, 5612.0679, 5615.3190, 5639.7490, 5650.7040,]
ThAr_pixels      = [      866,       893,       997,      1107,      1291,      1350,      1834,      2081,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=37
ThAr_wavelengths = [5499.2550, 5539.2620, 5548.1760, 5557.0450, 5573.3540, 5587.0263,]
ThAr_pixels      = [      537,      1183,      1344,      1514,      1850,      2167,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=38
ThAr_wavelengths = [5451.6520, 5464.2055, 5495.8738, 5499.2550, 5504.3020, 5506.1130, 5509.9940, 5514.8731,]
ThAr_pixels      = [      803,      1011,      1597,      1667,      1774,      1813,      1899,      2012,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=39
ThAr_wavelengths = [5403.1993, 5415.5226, 5417.4858, 5431.1120, 5451.6520, 5452.2187, 5464.2055,]
ThAr_pixels      = [     1044,      1277,      1313,      1579,      2033,      2047,      2389,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)
n_order=40
ThAr_wavelengths = [5317.4940, 5326.9756, 5343.5810, 5360.1501, 5374.8219, 5382.9280, 5386.6107,]
ThAr_pixels      = [      662,       815,      1098,      1409,      1710,      1890,      1977,]
print_wv_px(n_order-1, ThAr_wavelengths, ThAr_pixels)

In [ ]:


n_order=43
ThAr_wavelengths = [  5154.2430, 5158.6040, 5176.9610, 5195.8140, 5199.1640, 5211.2300, 5219.1100,]
ThAr_pixels      = [   903,      980,     1324,     1721,     1797,     2090,     2302,]

n_order=44
ThAr_wavelengths = [  5090.5460, 5096.4850, 5111.2781, 5115.0450, 5134.7460, 5154.2430, 5158.6040, 5163.4584,]
ThAr_pixels      = [   780,      885,     1154,     1225,     1634,     2100,     2215,     2351,]

n_order=45
ThAr_wavelengths = [  5028.6556, 5039.2303, 5044.7195, 5062.0371, 5064.9454, 5067.9737, 5090.5460, 5096.4850, 5100.6211,]
ThAr_pixels      = [   670,      851,      948,     1276,     1335,     1397,     1901,     2052,     2163,]

n_order=46
ThAr_wavelengths = [  4985.3725, 5009.3344, 5017.1628, 5019.8062, 5028.6556, 5029.8916, 5039.2303, 5044.7195, 5047.0434, 5049.7960, 5050.7842,]
ThAr_pixels      = [   856,     1310,     1475,     1531,     1732,     1758,     1989,     2132,     2197,     2273,     2302,]

n_order=47
ThAr_wavelengths = [  4933.2091, 4939.6422, 4943.0642, 4945.4587, 4965.0795, 4972.1597, 4980.1859, 4982.4875, 4985.3725, 5002.0972,]
ThAr_pixels      = [   872,      991,     1055,     1100,     1502,     1662,     1852,     1908,     1982,     2452,]

n_order=48
ThAr_wavelengths = [  4878.7330, 4879.8635, 4889.0422, 4894.9551, 4919.8157, 4927.7803, 4933.2091, 4939.6422, 4943.0642, 4945.4587,]
ThAr_pixels      = [   829,      848,     1018,     1131,     1661,     1851,     1989,     2163,     2257,     2329,]

In [ ]:

n_order=40
ThAr_wavelengths = [5317.4940, 5326.9756, 5343.5810, 5360.1501, 5374.8219, 5382.9280, 5386.6107,]
ThAr_pixels      = [      662,       815,      1098,      1409,      1710,      1890,      1977,]
"REDO 41-48"

n_order=41 # maybe check again... dlambda is too small!
ThAr_wavelengths = [5265.5514, 5281.0687, 5296.2787, 5309.6093, 5317.4944, 5326.2774, ]
ThAr_pixels      = [      820,      1077,      1350,      1619,      1795,      2016, ]

n_order=42
ThAr_wavelengths = [  5195.8140, 5199.1640, 5211.2300, 5219.1100, 5231.1600, 5247.6550, 5258.3600, 5266.7100,]
ThAr_pixels      = [   628,      680,      882,     1020,     1241,     1575,     1810,     2006,]


ThAr_wavelengths = [  5219.1100, 5231.1600,  5258.3600, 5260.1041, 5266.7100,]
ThAr_pixels      = [       1059,      1241,       1683,      1721,      1805,]
ThAr_wavelengths = [  5180, 5280,]
ThAr_pixels      = [     1, 2750,]



pipe = STELES_Pipeline_WaveSolution_Test(config_red)
order=n_order-1
pipe.run_wavesol_debug(order, ThAr_wavelengths, ThAr_pixels, print_orders=False, poly_degree=1)

In [ ]:
#Blue channel ( 2026-08-29)
######################################
n_order=1
ThAr_wavelengths = [  5509.9938, 5514.8730, 5521.7530, 5528.2272, 5539.2618, 5542.8900, 5548.1748, 5551.3719, 5557.0454]
ThAr_pixels      = [         42,       156,       319,       477,       760,       856,      1000,      1088,      1251]

n_order=2
ThAr_wavelengths = [ 5464.2055, 5467.1704, 5470.7591, 5479.0751, 5490.1194, 5492.6435, 5495.8738, 5499.2554, 5504.3018, 5506.1128, 5509.9938, 5514.8730,]
ThAr_pixels      = [       502,       577,       670,       894,      1204,      1278,      1374,      1477,      1635,      1694,      1820,      1984,]

n_order=3
ThAr_wavelengths = [  5386.6107, 5390.4400, 5394.7608, 5402.6048, 5415.5226, 5417.4858, 5424.0079, 5425.6783, 5426.4075,     5431.1120, 5434.1510, 5435.8920, 5437.3870, 5439.9891, ]
ThAr_pixels      = [        150,       243,       350,       550,       894,       950,      1133,      1185,      1204,     1346,      1438,      1492,      1539,      1624, ]

n_order=4
ThAr_wavelengths = [  5326.9756, 5340.4982, 5343.5812, 5355.6365, 5360.1501, 5361.1556,  5370.7096, 5372.7027, 5382.9276,]
ThAr_pixels      = [        204,       547,       629,       959,      1089,      1118,       1406,      1469,      1803,]

n_order=5
ThAr_wavelengths = [  5266.7103, 5270.2660, 5274.1188, 5291.8163, 5294.3971, 5306.9864, 5307.4655, 5310.2665, 5320.7699, 5322.8988, 5326.9756,]
ThAr_pixels      = [        209,       298,       396,       872,       945,      1318,      1333,      1420,      1763,      1836,      1988,]

n_order=6
ThAr_wavelengths = [  5209.7246, 5211.2305, 5213.3492, 5219.1099, 5233.2254, 5238.8137, 5247.6547, 5258.3602,]
ThAr_pixels      = [       262,       301,       356,       506,       897,      1060,      1328,      1676,]


n_order=7
ThAr_wavelengths = [ 5154.2430, 5158.6041, 5161.5396, 5163.4584, 5173.6715, 5176.9610, 5187.7462, 5195.8136, 5199.1637,]
ThAr_pixels      = [       323,       437,       517,       567,        852,       947,      1273,      1531,      1644,]

n_order=8
ThAr_wavelengths = [ 5096.4848, 5100.6211, 5101.1299, 5111.2781, 5115.0448, 5118.2023, 5128.4897, 5134.7460, 5141.7827, 5145.3083,]
ThAr_pixels      = [       292,       401,       414,       692,       799,       892,      1200,      1399,      1633,      1755,]

n_order=9
ThAr_wavelengths = [ 5039.2303, 5044.7195, 5049.7960, 5050.7842, 5051.8887, 5066.1355, 5067.9737, 5074.6465, 5075.4659, 5090.5455,]
ThAr_pixels      = [       241,       387,       525,       552,       583,       996,      1051,      1258,      1285,      1793,]

n_order=10
ThAr_wavelengths = [ 4982.4875, 4985.3725, 4989.3086, 5002.0972, 5003.5981, 5015.8893, 5019.8062, 5028.6556, 5029.8916, 5039.2303,]
ThAr_pixels      = [       172,       248,       354,       710,       754,      1125,      1248,      1541,      1583,      1918,]

n_order=11
ThAr_wavelengths = [ 4936.7746, 4939.6422, 4945.4587, 4950.6264, 4961.7260, 4972.1597, 4980.1859, 4982.4875, 4985.3725,]
ThAr_pixels      = [       368,       446,       610,       760,      1096,      1437,      1716,      1799,      1907,]

n_order=12
ThAr_wavelengths = [ 4878.0094, 4878.7330, 4889.0422, 4894.9551, 4899.2401, 4904.7516, 4910.1576, 4919.8157, 4927.7803, 4933.2091,]
ThAr_pixels      = [       181,       200,       484,       653,       782,       947,      1117,      1434,      1715,      1917,]

n_order=13
ThAr_wavelengths = [ 4826.7004, 4831.1213, 4840.8492, 4845.1626, 4850.4397, 4852.8685, 4861.2167, 4865.4775, 4872.9169, 4874.3645, 4878.7330, 4879.8635,]
ThAr_pixels      = [       166,       286,       560,       687,       847,       920,      1187,      1327,      1585,      1637,      1797,      1840,]

n_order=14
ThAr_wavelengths = [ 4778.2940, 4789.3868, 4795.9131, 4808.1337, 4809.6140, 4813.0069, 4822.8548, 4826.7004, 4831.1213,]
ThAr_pixels      = [       202,       514,       707,      1087,      1135,      1248,      1589,      1730,      1899,]

n_order=15
ThAr_wavelengths = [4732.0532, 4735.9058, 4739.6764, 4743.6870, 4749.2002, 4752.4141, 4757.2196, 4764.8646, 4766.6006, 4778.2940,]
ThAr_pixels      = [      271,       380,       488,       608,       773,       873,      1026,      1280,      1338,      1761,]

n_order=16
ThAr_wavelengths = [ 4683.3517, 4686.1946, 4695.0381, 4702.3161, 4703.9898, 4708.2940, 4712.4814, 4723.4382, 4726.8683, 4729.1282, 4732.0532,]
ThAr_pixels      = [       244,       324,       583,       806,       858,       996,      1133,      1514,      1640,      1724,      1837,]

n_order=17
ThAr_wavelengths = [ 4638.6849, 4643.5566, 4650.2343, 4655.2126, 4659.5698, 4663.2026, 4668.1716, 4669.9842, 4673.6609, 4676.0555, 4683.3517, 4686.1946,]
ThAr_pixels      = [       304,       446,       648,       801,       942,      1060,      1228,      1290,      1420,      1507,      1782,      1893,]

n_order=18
ThAr_wavelengths = [ 4588.4261, 4592.6660, 4595.4206, 4603.1446, 4611.8597, 4613.6044, 4621.1629, 4628.4409, 4637.2328, 4638.6849,]
ThAr_pixels      = [       178,       300,       381,       613,       889,       945,      1199,      1457,      1790,      1847,]

n_order=19
ThAr_wavelengths = [ 4545.9156, 4555.8127, 4567.2403, 4570.9722, 4579.3495, 4588.4261, 4589.8978, 4592.6660, ]
ThAr_pixels      = [       249,       544,       906,      1030,      1321,      1657,      1714,      1824, ]

n_order=20
ThAr_wavelengths = [ 4505.2167, 4513.2231, 4513.6799, 4515.1182, 4519.2592, 4521.1939, 4522.3230, 4530.5523, 4535.2546, 4540.9990, 4545.0519, 4545.9156,]
ThAr_pixels      = [       349,       595,       609,       653,       786,       849,       886,      1165,      1333,      1545,      1702,      1736,]

n_order=21
ThAr_wavelengths = [ 4458.0015, 4465.3406, 4469.5253, 4478.5958, 4483.3468, 4486.8973, 4490.9816, 4493.3337, 4498.5384, 4498.9401, 4499.9832, 4502.9268, 4505.2167,]
ThAr_pixels      = [       232,       453,       583,       877,      1037,      1161,      1307,      1393,      1590,      1605,      1646,      1763,      1856,]

n_order=22
ThAr_wavelengths = [  4414.4863, 4416.8447, 4426.0011, 4430.1890, 4430.9963, 4433.8380, 4439.4614, 4448.8792, 4452.5655, 4458.0015,]
ThAr_pixels      = [        200,       270,       552,       686,       712,       805,       995,      1330,      1468,      1680,]

n_order=23
ThAr_wavelengths = [ 4375.9542, 4379.6668, 4385.0566, 4391.1105, 4393.7590, 4400.0968, 4400.9863, 4402.9271, 4408.8828, 4409.8991, 4414.4863, 4416.8447,]
ThAr_pixels      = [       293,       407,       577,       774,       863,      1082,      1113,      1183,      1403,      1441,      1620,      1715,]

n_order=24
ThAr_wavelengths = [ 4333.5612, 4335.3379,  4345.1680, 4348.0640, 4352.2049, 4362.0662, 4363.7945, 4365.9301, 4367.8316, 4370.7532, 4371.3290, 4375.9542, 4378.1768, 4379.6668, 4379.6668,]
ThAr_pixels      = [       246,       300,        611,       706,       845,      1192,      1256,      1335,      1407,      1520,      1543,      1729,      1821,      1885,      1885,]

n_order=25
ThAr_wavelengths = [  4300.1008, 4300.6495, 4307.1762, 4308.1220, 4315.2543, 4318.4157, 4320.1264, 4328.9154, 4331.1995, 4332.0297, 4333.5612, 4335.3379,]
ThAr_pixels      = [        453,       471,       684,       715,       960,      1072,      1134,      1465,      1556,      1589,      1650,      1723,]

n_order=26
ThAr_wavelengths = [ 4257.4963, 4260.3330, 4269.9426, 4273.3574, 4275.1596, 4277.5282, 4282.0413, 4288.6688, 4291.8098, 4300.1008,]
ThAr_pixels      = [       351,       443,       760,       878,       941,      1025,      1190,      1444,      1568,      1916,]

n_order=27
ThAr_wavelengths = [ 4214.8285, 4217.4308, 4218.6649, 4222.6373, 4228.1580, 4235.4636, 4237.2198, 4250.3146, 4257.4963, 4259.3619,]
ThAr_pixels      = [       226,       307,       347,       475,       658,       910,       973,      1467,      1763,      1843,]

n_order=28
ThAr_wavelengths = [ 4181.8836, 4190.7129, 4191.0294, 4193.0164, 4194.9361, 4198.3170, 4200.6745, 4210.9232, 4217.4308, 4218.6649, 4222.6373,]
ThAr_pixels      = [       385,       679,       690,       759,       825,       946,      1031,      1422,      1689,      1741,      1914,]

n_order=29
ThAr_wavelengths = [ 4142.7010, 4143.6491, 4154.7205, 4158.5905, 4164.1795, 4165.7661, 4170.5333, 4181.8836,]
ThAr_pixels      = [       324,       354,       727,       864,      1067,      1127,      1310,      1778,]

n_order=30
ThAr_wavelengths = [ 4100.3414, 4103.9121, 4108.4198, 4112.7545, 4115.7589, 4116.7137, 4127.4120, 4128.6400, 4131.0021, 4131.7235, 4134.0681, 4135.4800, 4142.7010,]
ThAr_pixels      = [       140,       254,       401,       547,       649,       682,      1069,      1116,      1206,      1234,      1326,      1383,      1683,]

n_order=31
ThAr_wavelengths = [ 4072.0047, 4072.3849, 4075.5030, 4076.6284, 4086.5205, 4088.7264, 4089.1379, 4094.7470, 4100.3414, 4103.9121, 4108.4198,]
ThAr_pixels      = [       388,       402,       505,       544,       894,       975,       990,      1204,      1428,      1576,      1773,]

n_order=32
ThAr_wavelengths = [ 4030.8424, 4035.4600, 4036.0479, 4036.5652, 4042.8937, 4044.4179, 4052.9208, 4059.2529, 4063.4071, 4064.3315, 4067.4507, 4072.3849, 4075.5030,]
ThAr_pixels      = [       197,       349,       368,       385,       603,       656,       965,      1208,      1374,      1412,      1542,      1757,      1898,]

n_order=33
ThAr_wavelengths = [ 4001.0581, 4001.8935, 4005.0928, 4008.2102, 4012.4952, 4013.8566, 4019.1289, 4024.8025, 4027.0091, 4030.8424, 4035.4600, 4036.0479, 4036.5652,]
ThAr_pixels      = [       360,       389,       498,       607,       760,       810,      1007,      1228,      1317,      1476,      1675,      1701,      1723,]

n_order=34
ThAr_wavelengths = [ 3967.3921, 3972.1545, 3974.4766, 3979.3559, 3984.8795, 3987.2061, 3990.4922, 3992.0535, 3993.2989, 4001.0581, 4001.8935, 4005.0928,]
ThAr_pixels      = [       378,       543,       625,       802,      1011,      1101,      1231,      1294,      1345,      1678,      1714,      1860,]

n_order=35
ThAr_wavelengths = [ 3929.6693, 3933.6611, 3944.2534, 3947.5046, 3948.0305, 3948.9789, 3950.3951, 3955.1699, 3955.8903, 3959.3000, 3962.4196, 3967.3921, 3972.1545,]
ThAr_pixels      = [       237,       373,       751,       871,       891,       927,       981,      1169,      1198,      1338,      1469,      1687,      1907,]

n_order=36
ThAr_wavelengths = [  3895.4192, 3898.4374, 3903.1024, 3911.9091, 3916.4176, 3919.0234, 3923.7995, 3925.7188, 3928.6233, 3929.6693, 3932.9113,]
ThAr_pixels      = [       195,       298,        459,       779,       950,      1052,      1244,      1324,      1447,      1492,      1635,]

n_order=37
ThAr_wavelengths = [ 3869.6633, 3873.8224, 3879.6441, 3886.9159, 3891.9792, 3895.4192, 3898.4374, 3903.1024,]
ThAr_pixels      = [       426,       575,       791,      1073,      1280,      1426,      1558,      1770,]

n_order=38
ThAr_wavelengths = [3836.5851, 3837.8752, 3839.6953, 3845.4055, 3850.5813, 3852.1353, 3854.5108, 3859.8396, 3863.4059, 3868.5284, 3869.6633, 3873.8224,]
ThAr_pixels      = [      386,       432,       498,       708,       906,       966,      1061,      1280,      1433,      1661,      1714,      1911,]

n_order=39
ThAr_wavelengths = [  3809.4561, 3813.0680, 3818.6855, 3820.7926, 3825.1331, 3828.3846, 3830.7736, 3834.6787, 3837.8752, 3839.6953,]
ThAr_pixels      = [        542,       676,       891,       974,      1151,      1287,      1389,      1562,      1709,      1795,]

n_order=40
ThAr_wavelengths = [  3771.3708, 3776.2711, 3783.2964, 3786.3824, 3789.1680, 3790.7950, 3795.3800, 3799.3820, 3807.8740, 3809.4561,]
ThAr_pixels      = [        284,       460,       722,       841,       951,      1017,      1206,      1378,      1765,      1842,]

n_order=41
ThAr_wavelengths = [3741.1830, 3742.9234, 3750.4939, 3751.0219, 3752.5689, 3753.5177, 3755.2121, 3757.6941, 3758.4671, 3763.5053, 3765.2700, 3766.1186, 3767.9007, 3770.5200, 3771.3708, 3772.6498, 3776.2711,]
ThAr_pixels      = [      292,       354,       633,       654,       713,       750,       815,       914,       945,      1152,      1227,      1263,      1340,      1456,      1494,      1552,      1722,]

n_order=42
ThAr_wavelengths = [  3717.1713, 3718.2065, 3719.4347, 3721.8254, 3727.9027, 3729.3087, 3730.3683, 3732.9854, 3737.8890, 3741.1830, 3742.9234,]
ThAr_pixels      = [       507,       546,        593,       684,       924,       981,      1024,      1133,      1345,      1493,      1573,]

n_order=43
ThAr_wavelengths = [  3680.0609, 3687.9841, 3690.6238, 3691.4117, 3692.5664, 3695.2889, 3698.1061, 3699.8808, 3703.2299, 3706.7672, 3717.1713, 3718.2065, 3719.4347,]
ThAr_pixels      = [        228,       518,       619,       649,       694,       801,       914,       986,      1127,      1280,      1761,      1812,      1873,]

n_order=44
ThAr_wavelengths = [  3656.6938, 3658.8087, 3659.6294, 3660.4370, 3661.6214, 3663.2025, 3666.9811, 3668.1398, 3671.5398, 3675.5675, 3678.2701, 3680.0609, 3687.9841, 3690.6238,]
ThAr_pixels      = [        433,       512,       544,       575,       620,       682,       832,       879,      1019,      1191,      1310,      1390,      1764,      1896,]

n_order=45
ThAr_wavelengths = [ 3625.6280, 3629.8503, 3635.9433, 3639.8329, 3642.2490, 3643.5123, 3645.7065, 3656.6938, 3658.8087, 3659.6294, 3660.4370,]
ThAr_pixels      = [       339,       498,       735,       893,       993,      1046,      1142,      1642,      1745,      1786,      1827,]

n_order=46
ThAr_wavelengths = [  3598.1199, 3604.6819, 3605.8792, 3608.3770, 3609.4452, 3612.8666, 3615.1327, 3615.8503, 3618.3633, 3622.7954, 3629.8503,]
ThAr_pixels      = [        360,       612,       660,       760,       803,       944,      1040,      1071,      1180,      1378,      1714,]

n_order=47
ThAr_wavelengths = [ 3567.2638, 3576.6156, 3581.6084, 3582.3546, 3584.1756, 3588.4407, 3591.4524, 3592.7794, 3598.1199, 3605.8792,]
ThAr_pixels      = [       242,       598,       799,       830,       906,      1087,      1220,      1280,      1528,      1915,]

n_order=48
ThAr_wavelengths = [ 3544.0179, 3548.5144, 3549.5959, 3551.4019, 3554.3058, 3556.9041, 3559.5081, 3561.0304, 3565.0298, 3567.2638, 3569.8204, 3576.6156,]
ThAr_pixels      = [       391,       566,       609,       681,       800,       909,      1020,      1086,      1264,      1367,      1487,      1824,]

n_order=49
ThAr_wavelengths = [  3518.4040, 3523.5061, 3531.4505, 3535.3196, 3539.5872, 3544.0179, 3545.5956, 3545.8450, 3548.5144, 3549.5959, 3551.4019,]
ThAr_pixels      = [        438,       640,       972,      1141,      1336,      1546,      1624,      1636,      1771,      1827,      1922,]

n_order=50
ThAr_wavelengths = [  3486.5512, 3489.5076, 3496.8107, 3498.6210, 3501.4559, 3509.7785, 3511.1574, 3514.3877, 3518.4040, 3519.9936,]
ThAr_pixels      = [        230,       343,       631,       705,       824,      1188,      1250,      1402,      1597,      1676,]

n_order=51
ThAr_wavelengths = [  3462.8505, 3468.2198, 3469.9208, 3471.9593, 3476.7474, 3478.2324, 3480.5055, 3486.5512, 3489.5076, 3491.5360, 3496.8107,]
ThAr_pixels      = [       319,        530,       598,       682,       884,       949,      1048,      1324,      1467,      1565,      1836,]

n_order=52
ThAr_wavelengths = [  3437.3071, 3442.5790, 3446.5474, 3451.7023, 3454.0952, 3457.0691, 3462.8505, 3464.1272, 3468.2198, 3469.3454, 3469.9208,]
ThAr_pixels      = [       322,       531,       694,       914,      1019,       1153,      1425,      1488,      1694,      1752,      1782,]

n_order=53
ThAr_wavelengths = [ 3413.0130, 3417.4978, 3421.2100, 3422.6561, 3423.9897, 3428.9992, 3433.9988, 3437.3071, 3439.3987, 3442.5790,]
ThAr_pixels      = [       360,       540,       694,       755,       812,      1032,      1262,      1420,      1524,      1686,]

n_order=54
ThAr_wavelengths = [ 3388.5309, 3392.0349, 3396.7278, 3397.5161, 3398.5448, 3402.6952, 3405.5584, 3408.7499, 3413.0130, 3415.8846, 3417.4978, 3421.2100,]
ThAr_pixels      = [       377,       518,       714,       748,       792,       974,      1104,      1253,      1459,      1604,      1687,      1886,]

n_order=55
ThAr_wavelengths = [ 3366.5171, 3372.8230, 3373.4925, 3374.9749, 3376.4359, 3380.8595, 3385.5316, 3388.5309, 3389.4632, 3392.0349, 3396.7278,]
ThAr_pixels      = [       482,       744,       772,       837,       902,      1102,      1323,      1471,      1518,      1650,      1902,]

n_order=56
ThAr_wavelengths = [ 3336.1618, 3337.8703, 3340.7254, 3346.5560, 3348.7684, 3354.1796, 3358.6020, 3360.9982, 3364.6855, 3367.5822, 3372.8230,]
ThAr_pixels      = [       232,       299,       414,       657,       752,       993,      1197,      1314,      1498,      1647,      1931,]

n_order=57
ThAr_wavelengths = [ 3319.9101, 3322.0933, 3325.1207, 3327.1931, 3330.4770, 3333.1290, 3336.1618, 3337.8703, 3340.7254, 3345.8804,]
ThAr_pixels      = [       540,       632,       764,       855,      1004,      1128,      1274,      1357,      1500,      1770,]

n_order=58
ThAr_wavelengths = [ 3298.0498, 3299.6687, 3304.2383, 3305.3036, 3307.2283, 3309.3654, 3313.6784, 3314.8268, 3322.0933, 3324.7527,]
ThAr_pixels      = [       611,       681,       884,       932,      1021,      1121,      1329,      1387,      1768,      1916,]

n_order=59
ThAr_wavelengths = [  3272.0268, 3275.0676, 3281.7016, 3286.5829, 3287.7893, 3291.7394, 3292.5209, 3298.0498,]
ThAr_pixels      = [        491,       621,       916,      1145,      1204,      1400,      1439,      1732,]

n_order=60
ThAr_wavelengths = [  3243.6887, 3244.4488, 3245.7603, 3249.8003, 3251.9159, 3253.8660, 3256.2738, 3262.6684, 3263.5700, 3272.0268, 3273.3170, 3273.9157, 3275.0676, ]
ThAr_pixels      = [        264,       295,       349,       519,       610,       695,       803,      1100,      1144,      1571,      1640,      1674,      1736, ]

n_order=61
ThAr_wavelengths = [  3229.0096, 3238.1157, 3243.6887, 3244.4488, 3245.7603, 3249.8003, 3251.9159, 3253.8659,]
ThAr_pixels      = [       596,      1009,      1280,      1319,      1385,      1597,      1712,      1821,]

n_order=62
ThAr_wavelengths = [3204.3210, 3205.0010, 3210.3088, 3211.1943, 3212.0220, 3220.3508, 3229.0096, 3230.8679,]
ThAr_pixels      = [      490,       519,       754,       795,       832,      1231,      1685,      1789,]

n_order=63
ThAr_wavelengths = [  3178.2433, 3179.0482, 3180.1937, 3181.0376, 3181.6722,  3184.9490, 3192.5860, 3195.6890, 3202.5210, 3204.3210, 3205.0010]
ThAr_pixels      = [       313,       346,       395,       431,       459,         602,       952,      1102,      1448,      1544,      1581]

n_order=64
ThAr_wavelengths = [  3161.3726, 3169.6685, 3171.2755, 3173.4266, 3174.2043, 3178.2433, 3179.0482, 3180.1937, 3181.0376, 3181.6722,  3184.9490]
ThAr_pixels      = [       518,       895,       971,      1075,      1113,      1317,      1359,      1418,      1463,      1497,        1677]

n_order=65
ThAr_wavelengths = [  3140.2720,  3142.8360, 3145.6370, 3150.4550, 3154.3009, 3156.4006, 3161.3726,]
ThAr_pixels      = [       528,         643,       771,      1001,      1190,      1298,      1562,]

n_order=66
ThAr_wavelengths = [   3117.6840, 3119.5262,  3122.9630, 3124.3870, 3136.2161, 3136.8293, 3139.0170, 3140.2720, 3142.8360, 3145.6370,]
ThAr_pixels      = [         461,       542,        699,       764,      1349,      1381,      1499,      1568,      1711,      1875,]

n_order=67
ThAr_wavelengths = [ 3093.4019, 3100.9382, 3102.6639, 3105.7474, 3107.0261, 3108.2968, 3110.0209, 3116.2630, 3119.5262, 3122.9634, 3124.3874,]
ThAr_pixels      = [       309,       645,       722,       869,       931,       993,      1078,      1403,      1581,      1774,      1858,]

n_order=68
ThAr_wavelengths = [ 3075.0565, 3078.8280, 3080.2170, 3082.9810, 3085.0260, 3090.0937, 3093.4019, 3100.9382, 3102.6639,]
ThAr_pixels      = [       402,       570,       634,       762,       860,      1110,      1280,      1695,      1792,]

n_order=69
ThAr_wavelengths = [ 3049.0924, 3060.4389, 3067.7294, 3068.9075, 3072.1150, 3075.0565, 3078.8280, 3080.2170, 3082.9810, 3085.0260, ]
ThAr_pixels      = [       153,       656,      1007,      1066,      1231,      1387,      1596,      1675,      1838,      1962,]

n_order=70
ThAr_wavelengths = [ 3030.4877, 3033.5083, 3038.5987, 3046.9513, 3047.8163, 3049.0924, 3060.4389,]
ThAr_pixels      = [       209,       341,       570,       970,      1013,      1078,      1696,]

n_order=71
ThAr_wavelengths = [ 3015.7252, 3017.1322, 3018.4899, 3019.4219, 3022.09, 3023.98, 3025.45, 3026.70, 3028.90, 3030.4877, 3033.5083, 3035.15]
ThAr_pixels      = [       426,       490,       553,       596,     722,     812,     884,     949,    1058,      1139,      1299,    1386]

n_order=72
ThAr_wavelengths = [ 2999.0940, 3006.9314, 3012.7078, 3015.7252, 3017.1321, 3018.4899, 3019.4218, 3022.0955, 3023.98, 3025.45]
ThAr_pixels      = [       345,       895,      1232,      1396,      1474,      1551,      1605,      1762,    1877,    1969]

n_order=73
ThAr_wavelengths = [ 2983.8170, 2993.7990, 2999.0940,  3006.9314]
ThAr_pixels      = [       316,       963,      1349,       2024]


In [ ]:

#for i,wv in enumerate(ThAr_wavelengths):
#    print(f"{n_order}\t{wv}\t{ThAr_pixels[i]}")

In [ ]:
# Orders blue channel

n_order=6
ThAr_wavelengths = [  5209.7246, 5211.2305, 5213.3492, 5219.1099, 5233.2254, 5238.8137, 5247.6547, 5258.3602,]
ThAr_pixels      = [       262,       301,       355,       506,       894,      1057,      1325,      1672,]

for i,wv in enumerate(ThAr_wavelengths):
    print(f"{n_order}\t{wv}\t{ThAr_pixels[i]}")

pipe = STELES_Pipeline_WaveSolution_Test(config_blue)
order=n_order
pipe.run_wavesol_debug(order, ThAr_wavelengths, ThAr_pixels, print_orders=False, poly_degree=3)

In [ ]:
#2026-01-29 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-29_test_bench/blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,   # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_20260128_0033","flat_isb_3000mas_20260128_0034", "flat_isb_3000mas_20260128_0035"],
    "sci_list":  ["sci_0055"],
    "arc_list":  ["thar_600mas_0013", "thar_600mas_0014"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HR1996_0055",
    "fileout_arc":  "master_thar_0013",
    # arc scaling parameters
    "saturation_threshold": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    "aperture_extract": [-10,+13], # blue 2026-01-29
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-29_test_bench/red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,   # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_20260128_0034","flat_isb_3000mas_20260128_0035", "flat_isb_3000mas_20260128_0036"],
    "sci_list":  ["sci_0055"],
    "arc_list":  ["thar_600mas_0013", "thar_600mas_0014"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HR1996_0055",
    "fileout_arc":  "master_thar_0013",
    # arc scaling parameters
    "saturation_threshold": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    "aperture_extract": [-10,+4], # blue 2026-01-29
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:
# Orders red channel
n_order=1
ThAr_peak_wavelengths = np.array([9629.5693, 9632.6439, 9664.6983, 9700.5631, 9746.4629])
ThAr_peak_pixels      = np.array([      570,       600,       894,      1254,      1782])

n_order=2
ThAr_peak_wavelengths = np.array([9399.089, 9431.5996, 9461.0279, 9470.6819, 9474.8793, 9497.1891, 9505.3930, 9561.2452, 9582.8133])
ThAr_peak_pixels      = np.array([     277,       558,       832,       927,       968,     1198,      1285,      1963,       2274])

n_order=3
ThAr_peak_wavelengths = np.array([9194.638, 9203.962, 9224.499, 9289.562, 9291.531, 9354.220, 9399.089])
ThAr_peak_pixels      = np.array([     139,      217,      395,     1011,     1031,     1751,     2401])

n_order=4
ThAr_peak_wavelengths = np.array([9048.250, 9094.829, 9194.638, 9203.962, 9224.499])
ThAr_peak_pixels      = np.array([     437,      883,     2083,     2224,      2569])

n_order=5
ThAr_peak_wavelengths = np.array([8841.183, 8849.98, 8868.833, 8892.987, 8905.658, 8949.123, 8955.847, 8967.64, 9016.590, 9017.591, 9045.353, 9048.250])
ThAr_peak_pixels      = np.array([     143,     217,      387,      615,      738,     1197,     1273,    1411,     2063,     2079,     2537,     2588])

n_order=6
ThAr_peak_wavelengths = np.array([ 8667.9442, 8678.4083, 8709.2341, 8748.0309, 8758.2434, 8761.6862, 8799.0875, 8841.1833, 8868.8334,])
ThAr_peak_pixels      = np.array([        86,       178,       460,       845,       953,       991,      1420,      1983,      2431,])

n_order=7
ThAr_peak_wavelengths = np.array([ 8539.7930, 8554.9440, 8573.1210, 8605.7760, 8620.4600, 8639.4420, 8645.3090, 8665.4860, 8678.4080,])
ThAr_peak_pixels      = np.array([       384,       530,       713,      1061,      1232,      1461,      1536,      1808,      1997,])

n_order=8
ThAr_peak_wavelengths = np.array([ 8384.7240, 8416.7270, 8417.9980, 8421.2250, 8446.5120, 8464.2370, 8478.3580, 8510.6240, 8539.7930])
ThAr_peak_pixels      = np.array([       387,       708,       721,       755,      1030,      1235,      1411,      1847,      2312])

n_order=9
ThAr_peak_wavelengths = np.array([ 8252.3940, 8275.6270, 8320.8550, 8330.4490, 8358.7260, 8369.3400, 8379.2260, 8384.7240,])
ThAr_peak_pixels      = np.array([       562,       806,      1332,      1453,      1849,      2010,      2172,      2266,])

n_order=10
ThAr_peak_wavelengths = np.array([ 8085.2190, 8093.6240, 8119.1810, 8138.4753, 8143.1380, 8159.7280, 8169.7865, 8186.9110, 8252.3940,])
ThAr_peak_pixels      = np.array([       334,       418,       684,       898,       950,      1145,      1266,      1489,      2543,])

n_order=11
ThAr_peak_wavelengths = np.array([7954.5920, 7978.9730, 7987.9730, 8032.4310, 8037.2180, 8046.1169, 8053.3090, 8085.2190, 8093.6240])
ThAr_peak_pixels      = np.array([      426,       684,       782,      1313,      1375,      1495,      1595,      2084,      2230])

n_order=12
ThAr_peak_wavelengths = np.array([ 7788.9340, 7798.3580, 7847.5390, 7865.9700, 7868.1946, 7886.2830, 7891.0750, 7916.4420, 7937.7340, 7954.5920])
ThAr_peak_pixels      = np.array([       115,       208,       725,       938,       959,      1184,      1246,      1591,      1911,      2197])

n_order=13
ThAr_peak_wavelengths = np.array([ 7670.0570, 7678.1270, 7685.3075, 7710.2692, 7728.9510, 7788.9340, 7798.3580, 7817.7670, 7841.7911])
ThAr_peak_pixels      = np.array([       222,       308,       382,       653,       866,      1653,      1798,      2118,      2586])

n_order=14
ThAr_peak_wavelengths = np.array([7549.3140, 7567.7420, 7585.7920, 7589.3151, 7628.8818, 7647.3790, 7670.0575, 7678.1270, 7685.3070,])
ThAr_peak_pixels      = np.array([      278,       473,       674,       714,      1199,      1451,      1789,      1921,      2043,])

n_order=15
ThAr_peak_wavelengths = np.array([ 7425.2940, 7428.9400, 7430.2530, 7435.3680, 7436.2970, 7447.8490, 7471.1640, 7481.3545, 7484.3267, 7549.3140, 7567.7420, 7569.5115, 7585.7920,])
ThAr_peak_pixels      = np.array([       253,       293,       306,       361,       373,       498,       767,       889,       925,      1838,      2159,      2191,      2518,])

n_order=16
ThAr_peak_wavelengths = np.array([ 7311.7160, 7316.0050, 7328.2850, 7353.2930, 7372.1184, 7412.3368, 7425.2942, 7428.9400, 7430.2530, 7435.3680, 7436.2970, 7444.7490, 7447.8490,])
ThAr_peak_pixels      = np.array([       301,       348,       485,       774,      1008,      1568,      1768,      1830,      1851,      1939,      1955,      2105,      2164,])

n_order=17
ThAr_peak_wavelengths = np.array([ 7200.0450, 7212.6900, 7218.0540, 7242.0919, 7244.6965, 7246.1280, 7265.1720, 7284.9033, 7311.7160, 7316.0050, 7324.8073, 7328.2850, 7341.1510,])
ThAr_peak_pixels      = np.array([       329,       471,       532,       823,       854,       872,      1117,      1393,      1809,      1881,      2038,      2101,      2353,])

n_order=18
ThAr_peak_wavelengths = np.array([7084.1690, 7107.4780, 7124.5610, 7150.2840, 7191.1328, 7200.0450, 7208.0060, 7212.6900, 7218.0540, 7219.1515,])
ThAr_peak_pixels      = np.array([      274,       538,       745,      1076,      1675,      1823,      1963,      2048,      2148,      2171,])

n_order=19
ThAr_peak_wavelengths = np.array([ 6989.6550, 7000.8040, 7018.5680, 7030.2510, 7060.6538, 7068.7360, 7084.1690, 7086.7044, 7107.4780, 7124.5610, 7125.8200])
ThAr_peak_pixels      = np.array([       420,       552,       771,       919,      1340,      1462,      1707,      1751,      2126,      2482,      2510])

n_order=20
ThAr_peak_wavelengths = np.array([ 6871.2890, 6874.7530, 6888.1740, 6911.2260, 6937.6640, 6943.6105, 6989.6550, 7000.8040, 7002.8830, ])
ThAr_peak_pixels      = np.array([       258,       299,       456,       737,      1086,      1172,      1903,      2114,      2155, ])

n_order=21
ThAr_peak_wavelengths = np.array([ 6778.3120, 6791.2351, 6824.6770, 6829.0350, 6834.9250, 6861.2690, 6871.2890, 6874.7530, 6879.5824, 6888.1740,])
ThAr_peak_pixels      = np.array([       351,       509,       937,       996,      1077,      1469,      1631,      1689,      1773,      1927,])

n_order=22
ThAr_peak_wavelengths = np.array([  6677.2820, 6711.2521, 6713.9700, 6719.1990, 6727.4580, 6752.8330, 6766.6120, 6778.3120, 6791.2350, 6798.4870,])# 6791.2351, 6824.6770, 6829.0350,])
ThAr_peak_pixels      = np.array([        320,       743,       780,       849,       960,      1330,      1548,      1745,      1982,      2124,])

n_order=23
ThAr_peak_wavelengths = np.array([ 6583.9060, 6588.5400, 6591.4840, 6593.9390, 6604.8530, 6619.9460, 6638.2207, 6639.7400, 6643.6980, 6674.6970, 6678.7067, 6711.2521, 6713.9700,])
ThAr_peak_pixels      = np.array([       344,       402,       438,       468,       605,       803,      1057,      1080,      1138,      1626,      1696,      2335,      2400,])

n_order=24
ThAr_peak_wavelengths = np.array([ 6509.0503, 6512.3640, 6522.0432, 6531.3420, 6554.1600, 6558.8756, 6577.2150, 6583.9060, 6588.5400, 6591.4840, 6593.9390, 6604.8530,])
ThAr_peak_pixels      = np.array([       569,       613,       741,       866,      1197,      1268,      1567,      1681,      1768,      1820,      1865,      2079,])

n_order=25
ThAr_peak_wavelengths = np.array([ 6400.6960, 6406.4460, 6411.8990, 6416.3070, 6431.5550, 6457.2820, 6483.0825, 6490.7370, 6512.3640, 6522.0432])
ThAr_peak_pixels      = np.array([       342,       413,       482,       538,       744,      1108,      1520,      1652,      2065,      2270])

n_order=26
ThAr_peak_wavelengths = np.array([ 6307.6570, 6327.2780, 6337.6201, 6342.8590, 6376.9310, 6387.3960, 6400.6960, 6406.4460, 6411.8990, 6416.3070, 6431.5550,])
ThAr_peak_pixels      = np.array([       277,       526,       664,       734,      1234,      1403,      1633,      1736,      1838,      1925,      2246,])

n_order=27
ThAr_peak_wavelengths = np.array([ 6234.8550, 6257.4240, 6261.4180, 6274.1170, 6300.9170, 6307.6570, 6327.2780, 6342.8590,])
ThAr_peak_pixels      = np.array([         435,       738,       794,       979,      1401,      1516,      1878,      2206,])

n_order=28
ThAr_peak_wavelengths = np.array([ 6145.4410, 6151.9930, 6164.4800, 6172.2780, 6198.2230, 6215.9380, 6224.5270, 6226.3700, 6234.8550, 6257.4240, 6261.4180,])
ThAr_peak_pixels      = np.array([       354,       439,       606,       714,      1094,      1384,      1532,      1566,      1722,      2185,      2275,])

n_order=29
ThAr_peak_wavelengths = np.array([ 6059.3730, 6069.0210, 6073.1040, 6085.3750, 6088.0300, 6114.9230, 6138.6460, 6145.4410, 6151.9930, 6172.2780,])
ThAr_peak_pixels      = np.array([       285,       413,       468,       635,       673,      1070,      1462,      1583,      1707,      2122,])

n_order=30
ThAr_peak_wavelengths = np.array([ 5989.0450, 5994.1290, 6010.1610, 6021.0360, 6032.1270, 6049.0510, 6059.3730, 6069.0210, 6073.1040, 6085.3750, 6088.0300,])
ThAr_peak_pixels      = np.array([       398,       466,       690,       849,      1018,      1292,      1471,      1648,      1725,      1974,      2032,])

n_order=31
ThAr_peak_wavelengths = np.array([ 5914.6710, 5918.9440, 5944.6470, 5960.7810, 5989.0450, 5994.1290, 6010.1610, 6021.0360,])
ThAr_peak_pixels      = np.array([       430,       489,       861,      1114,      1605,      1704,      2034,      2286,])

n_order=32
ThAr_peak_wavelengths = np.array([ 5834.2630, 5852.6810, 5860.3100, 5863.7180, 5885.7020, 5905.5710, 5912.0850, 5914.6710, 5918.9440, 5944.6470,])
ThAr_peak_pixels      = np.array([       355,       613,       724,       774,      1119,      1462,      1582,      1634,      1717,      2283,])

n_order=33
ThAr_peak_wavelengths = np.array([ 5760.5510, 5763.5290, 5773.9460, 5789.6450, 5800.8300, 5804.1410, 5815.4220, 5832.3700, 5852.6810, 5853.4750, 5854.1210, 5860.3100, 5863.7180,])
ThAr_peak_pixels      = np.array([       346,       388,       533,       764,       936,       989,      1175,      1475,      1876,      1892,      1907,      2043,      2121,])

n_order=34
ThAr_peak_wavelengths = np.array([ 5700.9180, 5707.1030, 5720.1830, 5725.3890, 5739.5200, 5753.0260, 5763.5290, 5773.9460, 5782.3140, 5789.6450, 5800.8300])
ThAr_peak_pixels      = np.array([       507,       599,       793,       876,      1104,      1341,      1535,      1740,      1917,      2081,      2355])

n_order=35
ThAr_peak_wavelengths = np.array([ 5639.7460, 5650.7040, 5665.1800, 5677.0530, 5685.1920, 5700.9180, 5707.1030, 5725.3890,])
ThAr_peak_pixels      = np.array([       633,       800,      1034,      1235,      1382,      1685,      1815,      2235,])

n_order=36
ThAr_peak_wavelengths = np.array([ 5573.3540, 5587.0260, 5601.6030, 5615.3190, 5633.2950, 5639.7490, 5650.7040,])
ThAr_peak_pixels      = np.array([       656,       870,      1114,      1357,      1706,      1845,      2094,])

n_order=37
ThAr_peak_wavelengths = np.array([ 5499.2550, 5504.3020, 5506.1130, 5509.9940, 5514.8730, 5521.7530, 5539.2620, 5548.1760, 5557.0450, 5573.3540, 5587.0263])
ThAr_peak_pixels      = np.array([       539,       615,       644,       703,       780,      891,      1188,      1351,      1520,       1862,      2181])

n_order=38
ThAr_peak_wavelengths = np.array([ 5447.1540, 5464.2060, 5470.7590, 5479.0751, 5492.6440, 5499.2550, 5504.3020, 5506.1130, 5509.9940, 5521.7530,])
ThAr_peak_pixels      = np.array([       736,      1016,      1128,      1280,      1540,      1675,      1784,      1823,      1911,      2193,])

n_order=39
ThAr_peak_wavelengths = np.array([ 5382.9280, 5394.7610, 5402.6050, 5407.6540, 5415.5230, 5431.1120, 5447.1540, 5452.2187])
ThAr_peak_pixels      = np.array([       721,       914,      1049,      1138,      1283,      1588,      1940,      2061,])

n_order=40
ThAr_peak_wavelengths = np.array([ 5317.4940, 5320.7700, 5326.9756, 5333.3850, 5343.5810, 5361.1560, 5372.7030, 5382.9280, 5386.6107,])
ThAr_peak_pixels      = np.array([       664,       716,       818,       927,      1105,      1435,      1673,      1901,      1987,])

n_order=41
ThAr_peak_wavelengths = np.array([ 5258.3600, 5266.7100, 5281.0690, 5300.5230, 5317.4940, 5320.7700, 5343.5810,])
ThAr_peak_pixels      = np.array([       685,       823,      1072,      1440,      1801,      1878,      2468,])

n_order=42
ThAr_peak_wavelengths = np.array([ 5195.8140, 5199.1640, 5211.2300, 5219.1100, 5231.1600, 5247.6550, 5258.3600, 5266.7100,])
ThAr_peak_pixels      = np.array([       631,       685,       887,      1025,      1246,      1580,      1814,      2016,])

n_order=43
ThAr_peak_wavelengths = np.array([ 5154.2430, 5158.6040, 5176.9610, 5195.8140, 5199.1640, 5211.2300, 5219.1100,])
ThAr_peak_pixels      = np.array([       903,       980,      1324,      1721,      1797,      2090,      2302,])

n_order = 44 # last order of previous configuration
ThAr_peak_wavelengths = np.array([5090.546, 5096.485, 5111.2781, 5115.045, 5134.746, 5154.243, 5158.604, 5163.4584])
ThAr_peak_pixels      = np.array([     780,      885,      1154,     1225,     1634,     2100,     2215,      2351])

n_order = 45
ThAr_peak_wavelengths = np.array([5028.6556, 5039.2303, 5044.7195, 5059.8611, 5062.0371, 5064.9454, 5067.9737, 5090.546, 5096.485, 5100.6211])
ThAr_peak_pixels      = np.array([      670,       851,       948,     1236,      1276,      1335,      1397,      1901,     2052,      2163])

n_order = 46
ThAr_peak_wavelengths = np.array([4985.3725, 5009.3344, 5017.1628, 5019.8062, 5028.6556, 5029.8916, 5039.2303, 5044.7195, 5047.0434, 5049.7960, 5050.7842])
ThAr_peak_pixels      = np.array([     856,       1310,      1475,      1531,      1732,      1758,      1989,      2132,      2197,      2273,      2302])

n_order = 47
ThAr_peak_wavelengths = np.array([4933.2091, 4939.6422, 4943.0642, 4945.4587, 4965.0795, 4972.1597, 4980.1859, 4982.4875, 4985.3725, 5002.0972])
ThAr_peak_pixels      = np.array([      872,       991,      1055,      1100,      1502,      1662,      1852,      1908,      1982,      2452])

n_order = 48
ThAr_peak_wavelengths = np.array([4878.7330, 4879.8635, 4889.0422, 4894.9551, 4919.8157, 4927.7803, 4933.2091, 4939.6422, 4943.0642, 4945.4587])
ThAr_peak_pixels      = np.array([      829,       848,      1018,      1131,      1661,      1851,      1989,      2163,      2257,      2329])
print("saved")

In [ ]:

# Orders red channel
n_order=1
ThAr_peak_wavelengths = np.array([9629.5693, 9632.6439, 9664.6983, 9700.5631, 9716.1420, 9739.7704, 9743.5614, 9746.4629])
ThAr_peak_pixels      = np.array([      570,       600,       894,      1254,      1422,      1700,      1743,      1781])

n_order=2
ThAr_peak_wavelengths = np.array([9399.089, 9431.5996, 9461.0279, 9467.1954, 9470.6819, 9474.8793, 9497.1891, 9505.3930, 9561.2452, 9582.8133])
ThAr_peak_pixels      = np.array([     277,       558,       832,       891,       927,       968,      1198,      1285,      1963,      2274])

n_order=3
ThAr_peak_wavelengths = np.array([ 9224.499, 9266.2070, 9276.2732, 9289.562, 9291.531, 9340.7053, 9354.220, 9383.2722])
ThAr_peak_pixels      = np.array([      395,      776,        876,     1011,     1031,      1582,     1751,      2153])

n_order=4
ThAr_peak_wavelengths = np.array([9048.250, 9075.3945, 9094.829, 9140.5559, 9165.8950, 9194.638, 9203.962,])
ThAr_peak_pixels      = np.array([     437,      693,       883,      1376,      1686,     2083,     2224,])

n_order=5
ThAr_peak_wavelengths = np.array([8849.98, 8868.833, 8892.987, 8905.658, 8910.8566, 8949.123, 8955.847, 8967.64, 8997.8763, 9008.4636, 9016.590, 9017.591,])
ThAr_peak_pixels      = np.array([    217,      387,      615,      738,       792,     1197,     1273,    1411,     1795,      1945,      2063,     2079,])

n_order=6
ThAr_peak_wavelengths = np.array([ 8678.4083, 8709.2341, 8732.4240, 8748.0309, 8758.2434, 8761.6862, 8775.5733, 8784.5621, 8799.0875, 8820.4210, 8841.1833,])
ThAr_peak_pixels      = np.array([       178,       460,       686,       845,       953,       991,      1142,      1245,      1420,      1690,      1983,])

n_order=9
ThAr_peak_wavelengths = np.array([ 8252.3940, 8275.6270, 8320.8550, 8330.4490, 8358.7260,  8366.0741, 8367.3936, 8369.3400, 8379.2260, 8384.7240,])
ThAr_peak_pixels      = np.array([       562,       806,      1332,      1453,      1849,       1955,      1981,      2010,      2172,      2266,])

n_order=17
ThAr_peak_wavelengths = np.array([ 7212.6900, 7218.0540, 7233.5365, 7242.0919, 7244.6965, 7246.1280, 7265.1720, 7284.9033, 7311.7160, 7316.0050, 7324.8073, 7328.2850,])
ThAr_peak_pixels      = np.array([       471,       532,       716,       823,       854,       872,      1117,      1393,      1809,      1881,      2038,      2101, ])

n_order=19
ThAr_peak_wavelengths = np.array([ 6989.6550, 7000.8040, 7018.5680, 7030.2510, 7036.2831, 7045.7970, 7060.6538, 7068.7360, 7084.1690, 7086.7044, 7107.4780,])
ThAr_peak_pixels      = np.array([       420,       552,       771,       919,       999,      1128,      1340,      1462,      1707,      1751,      2126,])

n_order=20
ThAr_peak_wavelengths = np.array([ 6871.2890, 6874.7530, 6888.1740, 6911.2260, 6937.6640, 6943.6105, 6951.4776, 6989.6550, 7000.8040, 7002.8830, ])
ThAr_peak_pixels      = np.array([       258,       299,       456,       737,      1086,      1172,      1285,      1903,      2114,      2155, ])

n_order=27
ThAr_peak_wavelengths = np.array([ 6234.8550, 6257.4240, 6261.4180, 6274.1170, 6291.1915, 6300.9170, 6307.6570, 6315.7750, 6327.2780, 6342.8590,])
ThAr_peak_pixels      = np.array([       435,       738,       794,       979,      1241,      1401,      1516,      1659,      1878,      2206,])

n_order=31
ThAr_peak_wavelengths = np.array([ 5914.6710, 5918.9440, 5928.8130, 5944.6470, 5960.7810, 5973.6649, 5975.0648, 5989.0450, 5994.1290, 6010.1610, 6021.0360,])
ThAr_peak_pixels      = np.array([       430,       489,       628,       861,      1114,      1327,      1352,      1605,      1704,      2034,      2286,])

n_order=34
ThAr_peak_wavelengths = np.array([ 5700.9180, 5707.1030, 5720.1830, 5725.3890, 5739.5200, 5753.0260, 5763.5290, 5773.9460, 5782.3140, 5789.6450, 5792.4304])
ThAr_peak_pixels      = np.array([       507,       599,       793,       876,      1104,      1341,      1535,      1740,      1917,      2081,      2148])

n_order=35
ThAr_peak_wavelengths = np.array([ 5639.7460, 5650.7040, 5657.9255, 5665.1800, 5677.0530, 5679.0050, 5685.1920, 5700.9180, 5707.1030, 5708.6794, 5725.3890,])
ThAr_peak_pixels      = np.array([       633,       800,       913,      1034,      1235,      1269,      1382,      1685,      1815,      1848,      2235,])

n_order=36
ThAr_peak_wavelengths = np.array([ 5573.3540, 5579.3583, 5587.0260, 5595.0635, 5601.6030, 5615.3190, 5620.9213, 5633.2950, 5639.7490, 5650.7040,])
ThAr_peak_pixels      = np.array([       656,       749,       870,      1000,      1114,      1357,      1460,      1706,      1845,      2094,])

n_order=39
ThAr_peak_wavelengths = np.array([ 5382.9280, 5386.6107, 5394.7610, 5402.6050, 5407.6540, 5415.5230, 5424.0079, 5431.1120, 5447.1540, 5452.2187])
ThAr_peak_pixels      = np.array([       721,       779,       914,      1049,      1138,      1283,      1443,      1587,      1940,      2058,])

n_order=40
ThAr_peak_wavelengths = np.array([ 5317.4940, 5320.7700, 5322.8988, 5326.9756, 5333.3850, 5340.4982, 5343.5810, 5354.6016, 5361.1560, 5370.7096, 5372.7030, 5382.9280, 5386.6107,])
ThAr_peak_pixels      = np.array([       664,       716,       751,       818,       927,      1048,      1105,      1307,      1435,      1629,      1673,      1901,      1987,])

n_order=41
ThAr_peak_wavelengths = np.array([ 5258.3600, 5266.7100, 5274.1188, 5281.0690, 5291.8163, 5294.3971, 5300.5230, 5317.4940, 5320.7700, 5326.9756, ])
ThAr_peak_pixels      = np.array([       685,       823,       949,      1072,      1271,      1317,      1440,      1801,      1878,      2024,])

n_order=42
ThAr_peak_wavelengths = np.array([ 5195.8140, 5199.1640, 5211.2300, 5219.1100, 5231.1600, 5238.8137, 5247.6550, 5258.3600, 5266.7100,])
ThAr_peak_pixels      = np.array([       631,       685,       887,      1025,      1246,      1396,      1580,      1814,      2016,])

n_order=43
ThAr_peak_wavelengths = np.array([ 5141.7827, 5145.3083, 5154.2430, 5158.6040, 5162.2846, 5176.9610, 5195.8140, 5199.1640, 5211.2300, 5219.1100,])
ThAr_peak_pixels      = np.array([       692,       751,       903,       980,      1048,      1324,      1721,      1797,      2090,      2302,])

n_order = 44 # last order of previous configuration
ThAr_peak_wavelengths = np.array([5090.5455,  5096.4848, 5100.6211, 5111.2781, 5115.0448, 5128.4897, 5134.7460, 5141.7827, 5154.2433, 5158.6041, ])
ThAr_peak_pixels      = np.array([      780,        885,       958,      1154,      1225,      1497,      1634,      1794,      2100,      2215, ])

n_order = 45
ThAr_peak_wavelengths = np.array([5028.6556, 5039.2303, 5044.7195, 5049.7960, 5059.8611, 5062.0371, 5064.9454, 5067.9737, 5081.4462, 5090.5455, 5096.4848, 5100.6211])
ThAr_peak_pixels      = np.array([      670,       851,       948,      1041,      1236,      1276,      1335,      1397,      1688,      1901,      2052,      2163])
#########################

pipe = STELES_Pipeline_PartII(config_red)
order=n_order
pipe.run_wavesol_debug(order, ThAr_peak_wavelengths, ThAr_peak_pixels, print_orders=False)

In [ ]:
# BLUE ORDERS

n_order = 1 # First order (corresponds to order n=3 on 2025-08-06)
ThAr_peak_wavelengths = np.array([5509.9938,5514.873,5521.753,5528.2272,5537.5563,5539.2618,5548.1748,5551.3719,5552.6229,5558.702,5559.8912])
ThAr_peak_pixels      = np.array([       52,     167,     330,      490,      727,      773,     1012,     1101,     1136,    1312,     1347])

n_order=2
ThAr_peak_wavelengths = np.array([ 5451.6520, 5452.2187, 5464.2055, 5467.1704, 5479.0751, 5490.1194, 5492.6435, 5495.8738, 5499.2554, 5504.3018, 5506.1128, 5509.9938,])
ThAr_peak_pixels      = np.array([       204,       217,       513,       588,       905,      1215,      1289,      1385,      1488,      1646,      1705,      1831,])

n_order=3
ThAr_peak_wavelengths = np.array([ 5386.6107, 5394.7608, 5402.6048, 5407.6535, 5415.5226, 5431.1120, 5439.9891, 5440.6015,])
ThAr_peak_pixels      = np.array([       161,       361,       561,       693,       905,      1357,      1635,      1654,])

n_order=4
ThAr_peak_wavelengths = np.array([ 5326.9756, 5340.4982, 5343.5812, 5355.6365, 5360.1501, 5361.1556, 5370.7096, 5372.7027, 5382.9276,])
ThAr_peak_pixels      = np.array([       215,       558,       640,      970,      1100,      1129,      1417,      1480,      1814,])

n_order=5
ThAr_peak_wavelengths = np.array([ 5258.3602, 5266.7103, 5274.1188, 5294.3971, 5297.7431, 5298.2824, 5312.0018, 5317.4944,])
ThAr_peak_pixels      = np.array([        16,       220,       408,      957,      1055,      1071,      1490,      1669,])

n_order=6
ThAr_peak_wavelengths = np.array([ 5209.7246, 5211.2305, 5213.3492, 5219.1099, 5233.2254, 5238.8137, 5247.6547, 5258.3602,])
ThAr_peak_pixels      = np.array([       273,       312,       367,       517,       908,      1071,      1339,      1687,])

n_order=7
ThAr_peak_wavelengths = np.array([ 5145.3083, 5154.2430, 5158.6041, 5165.7728, 5176.9610, 5187.7462, 5195.8136, 5199.1637,])
ThAr_peak_pixels      = np.array([       106,       334,       449,       642,      960,      1287,       1546,      1658,])

n_order=8
ThAr_peak_wavelengths = np.array([ 5090.5455, 5100.6211, 5101.1299, 5111.2781, 5115.0448, 5128.4897, 5134.7460, 5141.7827, 5145.3083,])
ThAr_peak_pixels      = np.array([       149,       412,       426,       705,       812,      1215,      1412,      1648,      1769,])

n_order=9
ThAr_peak_wavelengths = np.array([ 5039.2303, 5044.7195, 5049.7960, 5050.7842, 5051.8887, 5055.3473, 5062.0371, 5067.9737, 5081.4462, 5090.5455,])
ThAr_peak_pixels      = np.array([       253,       398,       537,       565,       596,       693,       885,      1064,      1495,      1808,])

n_order=10
ThAr_peak_wavelengths = np.array([ 4980.1859, 4982.4875, 4985.3725, 4989.3086, 5002.0972, 5003.5981, 5004.1279, 5009.3344, 5017.1628, 5019.8062, 5029.8916, 5039.2303,])
ThAr_peak_pixels      = np.array([       125,       185,       260,       365,       722,       766,       782,       936,      1179,      1261,      1599,      1933,])

n_order=11
ThAr_peak_wavelengths = np.array([ 4933.2091, 4933.8521, 4939.6422, 4943.0642, 4945.4587, 4950.6264, 4972.1597, 4980.1859, 4982.4875, 4985.3725,])
ThAr_peak_pixels      = np.array([       282,       299,       457,       552,       622,       772,      1449,      1732,      1816,      1923,])

n_order=12
ThAr_peak_wavelengths = np.array([ 4874.3645, 4878.0094, 4878.7330, 4879.8635, 4889.0422, 4894.9551, 4904.7516, 4919.8157, 4927.7803, 4933.2091,])
ThAr_peak_pixels      = np.array([        96,       192,       212,       244,       496,       665,       958,      1449,      1729,      1934,])

n_order=13
ThAr_peak_wavelengths = np.array([ 4822.8548, 4826.7004, 4831.1213, 4832.8025, 4840.8492, 4845.1626, 4847.8095, 4852.8685, 4865.4775, 4872.9169, 4874.3645, 4878.7330, 4879.8635,])
ThAr_peak_pixels      = np.array([        74,       177,       298,       345,       573,       700,       780,       933,      1341,      1599,      1652,      1813,      1856,])

n_order=14
ThAr_peak_wavelengths = np.array([ 4778.2940, 4789.3868, 4795.9131, 4800.1724, 4806.0205, 4808.1337, 4809.6140, 4822.8548, 4826.7004, 4831.1213,])
ThAr_peak_pixels      = np.array([       213,       528,       720,       848,      1033,      1100,      1149,      1604,      1745,      1915,])

n_order=15
ThAr_peak_wavelengths = np.array([ 4729.1282, 4732.0532, 4735.9058, 4739.6764, 4749.2002, 4752.4141, 4757.2196, 4764.8646, 4766.6006, 4778.2940,])
ThAr_peak_pixels      = np.array([       201,       283,       392,       501,       786,       884,      1040,      1294,      1353,      1775,])

n_order=16
ThAr_peak_wavelengths = np.array([ 4683.3517, 4686.1946, 4695.0381, 4702.3161, 4703.9898, 4708.2940, 4712.4814, 4723.4382, 4726.8683, 4729.1282, 4732.0532,])
ThAr_peak_pixels      = np.array([       256,       336,       595,       818,      870,       1009,      1148,      1529,      1656,      1740,      1851,])

n_order=17
ThAr_peak_wavelengths = np.array([ 4637.2328, 4638.6849, 4647.2509, 4655.2126, 4657.9012, 4663.2026, 4668.1716, 4669.9842, 4673.6609, 4676.0555, 4683.3517, 4686.1946,])
ThAr_peak_pixels      = np.array([       275,       317,       569,       814,       899,      1072,      1241,      1305,      1433,      1521,      1798,      1909,])

n_order=18
ThAr_peak_wavelengths = np.array([ 4588.4261, 4589.8978, 4592.6660, 4595.4206, 4596.0967, 4598.7627, 4603.1446, 4613.6044, 4621.1629, 4628.4409, 4631.7617, 4637.2328, 4638.6849,])
ThAr_peak_pixels      = np.array([       190,       232,       313,       393,       413,       493,       626,       956,      1212,      1471,      1594,      1805,      1862,])

n_order=19
ThAr_peak_wavelengths = np.array([ 4545.0519, 4545.9156, 4555.8127, 4567.2403, 4570.9722, 4579.3495, 4588.4261, 4589.8978, 4592.6660, 4595.4206,])
ThAr_peak_pixels      = np.array([       236,       261,       557,       918,      1042,      1335,      1672,      1730,      1840,      1951,])

n_order=20
ThAr_peak_wavelengths = np.array([  4499.9832, 4510.7332, 4515.1182, 4519.2592, 4521.1939, 4522.3230, 4530.5523, 4535.2546, 4540.9990, 4545.0519, 4545.9156,])
ThAr_peak_pixels      = np.array([        207,       528,       665,       798,       861,       898,      1179,      1345,      1561,      1717,      1752,])

n_order=21
ThAr_peak_wavelengths = np.array([ 4458.0015, 4469.5253, 4474.7594, 4481.8107, 4483.3468, 4486.8973, 4490.9816, 4493.3337, 4498.5384, 4498.9401, 4499.9832, 4502.9268, 4505.2167,])
ThAr_peak_pixels      = np.array([       245,       596,       763,       998,      1051,      1173,      1320,      1408,      1605,      1620,      1660,      1779,      1873,])

n_order=22
ThAr_peak_wavelengths = np.array([ 4408.8828, 4414.4863, 4416.8447, 4426.0011, 4430.1890, 4430.9963, 4433.8380, 4439.4614, 4448.8792, 4452.5655, 4458.0015,])
ThAr_peak_pixels      = np.array([        46,       212,       282,       565,       700,       723,       818,      1008,      1344,      1482,      1695,])

n_order=23
ThAr_peak_wavelengths = np.array([ 4375.9542, 4378.1768, 4379.6668, 4381.8601, 4385.0566, 4391.1105, 4393.7590, 4400.0968, 4400.9863, 4402.9271, 4408.8828, 4409.8991, 4414.4863, 4416.8447,])
ThAr_peak_pixels      = np.array([       304,       372,       418,       488,       589,       786,       876,      1095,      1126,      1196,      1416,      1456,      1634,      1729,])

n_order=24
ThAr_peak_wavelengths = np.array([ 4333.5612, 4335.3379, 4340.8954, 4342.4442, 4345.1680, 4348.0640, 4352.2049, 4353.4487, 4362.0662, 4363.7945, 4365.9301, 4367.8316, 4370.7532, 4371.3290, 4375.9542, 4378.1768, 4379.6668,])
ThAr_peak_pixels      = np.array([       257,       312,       486,       534,       622,       717,       857,       901,      1206,      1270,      1349,      1422,      1535,      1559,      1744,      1837,      1900,])

n_order=25
ThAr_peak_wavelengths = np.array([ 4300.1008, 4300.6495, 4307.1762, 4308.1220, 4315.2543, 4318.4157, 4320.1264, 4328.9154, 4331.1995, 4332.0297, 4333.5612, 4335.3379,])
ThAr_peak_pixels      = np.array([       467,       483,       696,       728,       972,      1085,      1148,      1480,      1571,      1602,      1665,      1739,])

n_order=26
ThAr_peak_wavelengths = np.array([ 4250.3146, 4257.4963, 4259.3619, 4266.2864, 4277.5282, 4282.0413, 4291.8098, 4300.1008,])
ThAr_peak_pixels      = np.array([       140,       362,       422,       649,      1038,      1204,      1583,      1932,])

n_order=27
ThAr_peak_wavelengths = np.array([ 4217.4308, 4218.6649, 4222.6373, 4228.1580, 4235.4636, 4237.2198, 4250.3146, 4257.4963, 4259.3619,])
ThAr_peak_pixels      = np.array([       320,       358,       487,       670,       923,       985,      1481,      1778,      1860,])

n_order=28
ThAr_peak_wavelengths = np.array([ 4181.8836, 4190.7129, 4191.0294, 4200.6745, 4208.8907, 4210.9232, 4217.4308, 4218.6649, 4222.6373,])
ThAr_peak_pixels      = np.array([       397,       690,       702,      1044,      1356,      1437,      1704,      1757,      1931,])

n_order=29
ThAr_peak_wavelengths = np.array([ 4142.7010, 4143.6491, 4154.7205, 4158.5905, 4164.1795, 4165.7661, 4170.5333, 4181.8836,])
ThAr_peak_pixels      = np.array([       336,       367,       740,       877,      1080,      1140,      1324,      1793,])

n_order=30
ThAr_peak_wavelengths = np.array([ 4100.3414, 4103.9121, 4108.4198, 4112.7545, 4115.7589, 4116.7137, 4127.4120, 4128.6400, 4131.0021, 4131.7235, 4134.0681, 4135.4800, 4142.7010,])
ThAr_peak_pixels      = np.array([       151,       265,       414,       558,       662,       696,      1083,      1130,      1220,      1248,      1340,      1397,      1698,])

n_order=31
ThAr_peak_wavelengths = np.array([ 4075.5030, 4076.6284, 4076.9432, 4086.5205, 4088.7264, 4089.1379, 4094.7470, 4100.3414, 4103.9121, 4108.4198,])
ThAr_peak_pixels      = np.array([       519,       556,       566,       906,       988,      1004,      1217,      1443,      1590,      1788,])

n_order=32
ThAr_peak_wavelengths = np.array([ 4030.8424, 4035.4600, 4036.0479, 4036.5652, 4042.8937, 4043.3948, 4044.4179, 4052.9208, 4053.5277, 4059.2529, 4063.4071, 4064.3315, 4067.4507, 4072.3849, 4075.5030,])
ThAr_peak_pixels      = np.array([       209,       361,       380,       398,       615,       632,       669,       977,      1000,      1222,      1389,      1427,      1557,      1772,      1913,])

n_order=33
ThAr_peak_wavelengths = np.array([ 4001.0581, 4001.8935, 4005.0928, 4008.2102, 4012.4952, 4013.8566, 4019.1289, 4024.8025, 4027.0091, 4030.8424, 4035.4600, 4036.0479, 4036.5652,])
ThAr_peak_pixels      = np.array([       374,       401,       511,       620,       773,       822,      1021,      1243,      1331,      1491,      1690,      1716,      1739,])

n_order=34
ThAr_peak_wavelengths = np.array([ 3967.3921, 3972.1545, 3974.4766, 3979.3559, 3990.4922, 3991.7309, 3992.0535, 3993.2989, 4001.0581, 4001.8935, 4005.0928,])
ThAr_peak_pixels      = np.array([       389,       556,       637,       814,      1245,      1297,      1309,      1360,      1693,      1731,      1877,])

n_order=35
ThAr_peak_wavelengths = np.array([ 3929.6693, 3932.9113, 3947.5046, 3948.0305, 3948.9789, 3950.3951, 3955.1699, 3955.8903, 3956.6908, 3959.3000, 3962.4196, 3967.3921, 3972.1545,])
ThAr_peak_pixels      = np.array([       249,       361,       884,       905,       940,       994,      1183,      1212,      1245,      1353,      1484,      1702,      1923,])

n_order=36
ThAr_peak_wavelengths = np.array([ 3895.4192, 3898.4374, 3900.8785, 3903.1024, 3911.9091, 3916.4176, 3919.0234, 3923.7995, 3925.7188, 3928.6233, 3929.6693, 3932.9113,])
ThAr_peak_pixels      = np.array([       208,       309,       393,       472,       792,       963,      1065,      1259,      1338,      1461,      1507,      1650,])

n_order=37
ThAr_peak_wavelengths = np.array([ 3868.5284, 3869.6633, 3873.8224, 3879.6441, 3891.9792, 3895.4192, 3898.4374, 3903.1024,])
ThAr_peak_pixels      = np.array([       398,       439,       588,       804,      1295,      1440,      1573,      1785,])

n_order=38
ThAr_peak_wavelengths = np.array([ 3834.6787, 3836.5851, 3837.8752, 3839.6953, 3845.4055, 3850.5813, 3852.1353, 3863.4059, 3868.5284, 3869.6633, 3873.8224,])
ThAr_peak_pixels      = np.array([       332,       399,       446,       510,       720,       918,       980,      1448,      1676,      1729,      1928,])

n_order=39
ThAr_peak_wavelengths = np.array([ 3809.4561, 3813.0680, 3818.6855, 3820.7926, 3825.1331, 3828.3846, 3830.7736, 3834.6787, 3837.8752, 3839.6953,])
ThAr_peak_pixels      = np.array([       554,       689,       904,       988,      1165,      1301,      1404,      1577,      1724,      1811,])

n_order=40
ThAr_peak_wavelengths = np.array([ 3771.3708, 3776.2711, 3783.2964, 3786.3824, 3789.1680, 3790.7950, 3795.3800, 3799.3820, 3807.8740, 3809.4561,])
ThAr_peak_pixels      = np.array([       297,       473,       734,       854,       964,      1031,      1220,      1393,      1780,      1857,])

n_order=41
ThAr_peak_wavelengths = np.array([ 3741.1830, 3742.9234, 3750.4939, 3751.0219, 3752.5689, 3753.5177, 3755.2121, 3757.6941, 3758.4671, 3763.5053, 3765.2700, 3766.1186, 3767.9007, 3770.5200, 3771.3708, 3772.6498, 3776.2711,])
ThAr_peak_pixels      = np.array([       305,       366,       646,       667,       725,       762,       828,       927,       958,      1165,      1241,      1277,      1355,      1471,      1509,      1568,      1738,])

n_order=42
ThAr_peak_wavelengths = np.array([ 3717.1713, 3718.2065, 3719.4347, 3721.8254, 3727.9027, 3729.3087, 3730.3683, 3732.9854, 3737.5125, 3737.8890, 3741.1830, 3742.9234,])
ThAr_peak_pixels      = np.array([       521,       558,       605,       698,       937,      995,      1039,      1148,      1343,      1359,      1507,      1588,])

n_order=43
ThAr_peak_wavelengths = np.array([ 3680.0609, 3687.9841, 3690.6238, 3691.4117, 3692.5664, 3695.2889, 3698.1061, 3699.8808, 3703.2299, 3706.7672, 3717.1713, 3718.2065, 3719.4347,])
ThAr_peak_pixels      = np.array([       240,       532,       632,       663,       707,       814,       928,      1000,      1141,      1295,      1776,      1827,      1888,])

n_order=44
ThAr_peak_wavelengths = np.array([ 3656.6938, 3658.8087, 3659.6294, 3660.4370, 3661.6214, 3663.2025, 3666.9811, 3668.1398, 3671.5398, 3675.5675, 3678.2701, 3680.0609, 3687.9841, 3690.6238,])
ThAr_peak_pixels      = np.array([       446,       525,       557,       588,       633,       695,       845,       892,      1032,      1205,      1324,      1404,      1780,      1912,])

n_order=45
ThAr_peak_wavelengths = np.array([ 3629.8503, 3635.9433, 3639.8329, 3642.2490, 3643.5123, 3656.6938, 3658.8087, 3659.6294, 3660.4370,])
ThAr_peak_pixels      = np.array([       510,       748,       904,      1006,      1061,      1655,      1761,      1801,      1843,])

n_order=46
ThAr_peak_wavelengths = np.array([ 3598.1199, 3608.3770, 3609.4452, 3612.8666, 3615.1327, 3615.8503, 3618.3633, 3619.2119, 3622.7954, 3629.8503,])
ThAr_peak_pixels      = np.array([       374,       772,       816,       957,      1053,      1084,      1193,      1232,      1392,      1731,])

n_order=47
ThAr_peak_wavelengths = np.array([ 3576.6156, 3581.6084, 3582.3546, 3584.1756, 3588.4407, 3591.4524, 3592.7794, 3598.1199, 3605.8792,])
ThAr_peak_pixels      = np.array([       611,       812,       841,       918,      1102,      1235,      1295,      1543,      1932,])

n_order=48
ThAr_peak_wavelengths = np.array([ 3548.5144, 3549.5959, 3551.4019, 3554.3058, 3556.9041, 3559.5081, 3561.0304, 3565.0298, 3576.6156,])
ThAr_peak_pixels      = np.array([       578,       621,       694,       813,       922,      1033,      1100,      1279,      1840,])

n_order=49
ThAr_peak_wavelengths = np.array([ 3518.4040, 3523.5061, 3531.4505, 3535.3196, 3539.5872, 3544.0179, 3545.5956, 3545.8450, 3548.5144, 3549.5959, 3551.4019,])
ThAr_peak_pixels      = np.array([       450,       653,       985,      1155,      1350,      1561,      1640,      1651,      1787,      1843,      1938,])

n_order=50
ThAr_peak_wavelengths = np.array([ 3486.5512, 3491.5360, 3496.8107, 3498.6210, 3501.8666, 3509.7785, 3511.1574, 3514.3877, 3518.4040, 3519.9936,])
ThAr_peak_pixels      = np.array([       241,       432,       643,       717,       853,      1199,      1264,      1416,      1611,      1692,])

n_order=51
ThAr_peak_wavelengths = np.array([ 3468.2198, 3469.9208, 3471.9593, 3476.7474, 3480.5055, 3486.5512, 3489.5076, 3491.5360, 3496.8107,])
ThAr_peak_pixels      = np.array([       541,       610,       694,       896,      1060,      1337,      1480,      1579,      1852,])

n_order=52
ThAr_peak_wavelengths = np.array([ 3437.3071, 3442.5790, 3446.5474, 3451.7023, 3457.0691, 3464.1272, 3468.2198, 3469.3454, 3469.9208,])
ThAr_peak_pixels      = np.array([       333,       542,       705,       926,      1168,      1503,      1709,      1766,      1799,])

n_order=53
ThAr_peak_wavelengths = np.array([ 3417.4978, 3421.2100, 3422.6561, 3423.9897, 3428.9992, 3433.9988, 3437.3071, 3439.3987, 3442.5790,])
ThAr_peak_pixels      = np.array([       551,       705,       766,       824,      1044,      1275,      1434,      1537,      1700,])

n_order=54
ThAr_peak_wavelengths = np.array([ 3392.0349, 3396.7278, 3397.5161, 3398.5448, 3402.6952, 3405.5584, 3408.7499, 3413.0130, 3415.8846, 3417.4978, 3421.2100,])
ThAr_peak_pixels      = np.array([       529,       725,       759,       803,       986,      1117,      1267,      1475,      1619,      1704,      1903,])

n_order=55
ThAr_peak_wavelengths = np.array([ 3372.8230, 3373.4925, 3374.9749, 3380.8595, 3385.5316, 3388.5309, 3389.4632, 3392.0349, 3396.7278,])
ThAr_peak_pixels      = np.array([       755,       784,       849,      1115,      1337,      1485,      1532,      1664,      1917,])

n_order=56
ThAr_peak_wavelengths = np.array([ 3340.7254, 3348.7684, 3351.2286, 3358.6020, 3360.9982, 3365.3383, 3367.5822, 3371.7967, 3372.8230,])
ThAr_peak_pixels      = np.array([       425,       764,       871,      1211,      1328,      1546,      1663,      1891,      1948,])

n_order=57
ThAr_peak_wavelengths = np.array([ 3322.0933, 3325.1207, 3327.1931, 3330.4770, 3333.1290, 3336.1618, 3337.8703, 3340.7254, 3345.8804,])
ThAr_peak_pixels      = np.array([       644,       776,       867,      1017,      1140,      1288,      1371,      1516,      1786,])

n_order=58
ThAr_peak_wavelengths = np.array([ 3298.0498, 3299.6687, 3304.2383, 3305.3036, 3307.2283, 3309.3654, 3313.6784, 3314.8268, 3322.0933, 3324.7527,])
ThAr_peak_pixels      = np.array([       623,       693,       896,       944,      1033,      1136,      1344,      1401,      1784,      1933,])

n_order=59
ThAr_peak_wavelengths = np.array([ 3275.0676, 3281.7016, 3286.5829, 3287.7893, 3291.7394, 3292.5209, 3298.0498,])
ThAr_peak_pixels      = np.array([       633,       929,      1158,      1217,      1415,      1455,      1748,])

n_order=60
ThAr_peak_wavelengths = np.array([ 3244.4488, 3245.7603, 3249.8003, 3251.9159, 3256.2738, 3257.3667, 3262.6684, 3272.0268, 3273.9157, 3275.0676, 3276.1712,])
ThAr_peak_pixels      = np.array([       306,       360,       530,       622,       815,       864,      1112,      1585,      1688,      1751,      1813,])

n_order=61
ThAr_peak_wavelengths = np.array([ 3229.0096, 3238.1157, 3243.6887, 3244.4488, 3245.7603, 3249.8003, 3251.9159, 3253.8659,])
ThAr_peak_pixels      = np.array([       607,      1022,      1293,      1333,      1400,      1613,      1728,      1837,])

n_order=62
ThAr_peak_wavelengths = np.array([ 3204.3210, 3210.3088, 3211.1943, 3212.0220, 3220.3508, 3229.0096, 3230.8679,])
ThAr_peak_pixels      = np.array([       501,       766,       806,       844,      1243,      1700,      1805,])

n_order=63
ThAr_peak_wavelengths = np.array([ 3178.2433, 3179.0482, 3180.1937, 3181.0376, 3181.6722, 3204.3210, 3210.3088, 3211.1943,])
ThAr_peak_pixels      = np.array([       324,       358,       407,       443,       470,      1559,      1895,      1947,])

n_order=64
ThAr_peak_wavelengths = np.array([ 3161.3726, 3169.6685, 3171.2755, 3173.4266, 3174.2043, 3178.2433, 3179.0482, 3180.1937, 3181.0376, 3181.6722,])
ThAr_peak_pixels      = np.array([       531,       906,       983,      1088,      1127,      1330,      1373,      1433,      1479,      1512,])

n_order=65
ThAr_peak_wavelengths = np.array([ 3140.2720, 3145.6370, 3154.3009, 3156.4006, 3161.3726,])
ThAr_peak_pixels      = np.array([       539,       783,      1204,      1312,      1578,])

n_order = 66 # last order with ThAr lines (corresponds to order n=68 on 2025-08-06)
ThAr_peak_wavelengths = np.array([ 3119.5262, 3136.2161, 3136.8293, 3139.02, 3140.272, 3145.637])
ThAr_peak_pixels      = np.array([       555,      1364,      1396,    1514,     1583,     1891])

n_order=67
ThAr_peak_wavelengths = np.array([ 3108.2968, 3110.0209, 3116.2630, 3119.5262,])
ThAr_peak_pixels      = np.array([      1004,      1089,      1414,      1592,])

n_order=68
ThAr_peak_wavelengths = np.array([ 3067.7294, 3080.2170, 3088.4800, 3093.4019, 3102.6639,])
ThAr_peak_pixels      = np.array([       580,       910,      1154,      1295,      1622,])


print("saved")

In [ ]:
# UVES ThAr catalog: https://astronomy.swin.edu.au/~mmurphy/thar/pipeline_thar.dat
n_order=9
ThAr_peak_wavelengths = np.array([ 5039.2303, 5044.7195, 5047.0434, 5049.7960, 5050.7842, 5051.8887, 5055.3473, 5062.0371, 5067.9737, 5081.4462,])# 5090.5455,])
ThAr_peak_pixels      = np.array([       253,       398,       462,       537,       565,       596,       693,       885,      1064,      1495,])#      1808,])

n_order=18
ThAr_peak_wavelengths = np.array([ 4588.4261, 4589.8978, 4592.6660, 4595.4206, 4596.0967, 4598.7627, 4603.1446, 4607.9347, 4613.6044, 4615.3340, 4621.1629, 4624.3140, 4628.4409, ])# 4638.6849,])
ThAr_peak_pixels      = np.array([       190,       232,       313,       393,       413,       493,       626,       775,       956,      1015,      1212,      1323,      1471, ])#      1862,])

n_order=22
ThAr_peak_wavelengths = np.array([4414.4863, 4416.8447, 4426.0011, 4430.1890, 4430.9963, 4433.8380, 4439.4614, 4443.6657, 4445.9012, 4448.8792, 4452.5655, 4458.0015,])
ThAr_peak_pixels      = np.array([      212,       282,       565,       700,       723,       818,      1008,      1155,      1235,      1344,      1482,      1695,])

n_order=26
ThAr_peak_wavelengths = np.array([ 4250.3146, 4253.5385, 4257.4963, 4259.3619, 4269.9426, 4275.1596, 4281.0678, 4282.0413, 4286.2285, 4291.8098])
ThAr_peak_pixels      = np.array([       140,       241,       362,       422,       774,       954,      1169,      1204,      1363,      1583])

n_order=29
ThAr_peak_wavelengths = np.array([ 4142.7010, 4143.6491, 4154.7205, 4158.5905, 4161.7389, 4164.1795, 4165.7661, 4170.5333, 4171.3410, 4178.8481, 4181.8836,])
ThAr_peak_pixels      = np.array([       336,       367,       740,       877,       990,      1080,      1140,      1324,      1356,      1663,      1793,])

n_order=34
ThAr_peak_wavelengths = np.array([ 3967.3921, 3969.0026, 3972.1545, 3974.4766, 3979.3559, 3987.2061, 3990.4922, 3991.7309, 3992.0535, 3993.2989, 4001.0581, 4001.8935,])
ThAr_peak_pixels      = np.array([       389,       446,       556,       637,       814,      1115,      1245,      1297,      1309,      1360,      1693,      1731,])

n_order=35
ThAr_peak_wavelengths = np.array([ 3929.6693, 3932.5466, 3932.9113, 3942.0732, 3948.0305, 3948.9789, 3950.3951, 3955.1699, 3955.8903, 3959.3000, 3962.4196,])
ThAr_peak_pixels      = np.array([       249,       348,       361,       682,       905,       940,       994,      1183,      1212,      1353,      1484,])

n_order=36
ThAr_peak_wavelengths = np.array([ 3895.4192, 3898.4374, 3901.6621, 3903.1024, 3911.9091, 3916.4176, 3923.7995, 3925.7188, 3928.6233, 3929.6693, 3932.9113,])
ThAr_peak_pixels      = np.array([       208,       309,       422,       472,       792,       963,      1259,      1338,      1461,      1507,      1650,])

n_order=57
ThAr_peak_wavelengths = np.array([ 3314.8268, 3319.9101, 3322.0933, 3325.1207, 3327.1931, 3330.4770, 3333.1290, 3334.6041, 3337.8703, 3340.7254,])
ThAr_peak_pixels      = np.array([       343,       552,       644,       776,       867,      1017,      1140,      1212,      1371,      1516,])

n_order=58
ThAr_peak_wavelengths = np.array([ 3291.7394,  3292.5209, 3298.0498, 3299.6687, 3304.2383, 3305.3036, 3307.2283, 3309.3654, 3313.6784, 3314.8268, ])
ThAr_peak_pixels      = np.array([       359,       391,       623,       693,       896,       944,      1033,      1136,       1344,      1401, ])

n_order=59
ThAr_peak_wavelengths = np.array([ 3270.7459, 3272.0250, 3275.0676, 3276.1712, 3281.7016, 3282.6166, 3282.9795, 3286.5829, 3287.7893, 3291.7394, 3292.5209, 3298.0498])
ThAr_peak_pixels      = np.array([       456,       504,       633,       683,      929,        973,       991,      1158,      1217,      1415,      1455,      1748])

n_order=60
ThAr_peak_wavelengths = np.array([ 3243.6887, 3244.4488, 3245.7603, 3249.8003, 3251.9159, 3256.2738, 3257.3667, 3258.1054, 3259.0608, 3262.6684, 3272.0268, 3273.9157, 3275.0676,])
ThAr_peak_pixels      = np.array([       276,       306,       360,       530,       622,       815,       864,       901,       944,      1112,      1585,      1688,      1751,])

n_order=61
ThAr_peak_wavelengths = np.array([ 3229.0096, 3230.8679, 3235.8400, 3238.1157, 3243.6887, 3244.4488, 3245.7603, 3249.8003, 3251.9159, 3253.8659,])
ThAr_peak_pixels      = np.array([       607,       690,       917,      1022,      1293,      1333,      1400,      1613,      1728,      1837,])

n_order=62
ThAr_peak_wavelengths = np.array([ 3204.3210, 3205.0010, 3210.3088, 3211.1943, 3212.0220, 3213.5741, 3216.7380, 3220.3508, 3222.4040, 3229.0096, 3230.8679,])
ThAr_peak_pixels      = np.array([       501,       531,       766,       806,       844,       918,      1069,      1243,      1350,      1700,      1805,])

n_order=63
ThAr_peak_wavelengths = np.array([ 3178.2433, 3179.0482, 3180.1937, 3181.0376, 3181.6722, 3184.9490, 3192.5860,  3195.6890, 3202.5210, 3204.3210, 3210.3088, 3211.1943,])
ThAr_peak_pixels      = np.array([       324,       358,       407,       443,       470,       616,       965,      1116,       1464,      1559,      1895,      1947,])

n_order=64
ThAr_peak_wavelengths = np.array([ 3161.3726, 3165.2960,  3166.0980, 3169.6685, 3171.2755, 3173.4266, 3174.2043, 3178.2433, 3179.0482, 3180.1937, 3181.0376, 3181.6722,])
ThAr_peak_pixels      = np.array([       531,       708,        743,       906,       983,      1088,      1127,      1330,      1373,      1433,      1479,      1512,])

n_order=65
ThAr_peak_wavelengths = np.array([ 3136.2160, 3136.8290, 3139.0170, 3140.2720, 3142.8360, 3145.6370, 3150.4550, 3154.3009, 3156.4006, 3161.3726,])
ThAr_peak_pixels      = np.array([       366,       393,       488,       539,       657,       783,      1015,      1204,      1312,      1578,])

# from here
n_order = 66 # last order with ThAr lines (corresponds to order n=68 on 2025-08-06)
ThAr_peak_wavelengths = np.array([ 3119.5262, 3122.9630, 3124.3870, 3131.0700, 3136.2161, 3136.8293, 3139.02, 3140.272, 3142.8360, ])
ThAr_peak_pixels      = np.array([       555,       712,       778,      1100,      1364,      1396,    1514,     1583,      1729, ])

pipe = STELES_Pipeline_PartII(config_blue)
order=n_order
pipe.run_wavesol_debug(order, ThAr_peak_wavelengths, ThAr_peak_pixels)

In [ ]:
pipe = STELES_Pipeline_PartII(config_blue)
#pipe.run()
#pipe.run_wavesol()
pipe.run_science_only()

In [ ]:
#2026-01-29 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-29_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_20260128_0033","flat_isb_20260128_0034", "flat_isb_20260128_0035"],
    "sci_list":  ["sci_0055"],
    "arc_list":  ["thar_0075", "thar_0076"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HR1996_0055",
    "fileout_arc":  "master_thar_0075",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"aperture_extract": [-6,+18],  # blue 2026-01-28
    #"aperture_extract": [-8,+14],  # blue 2025-09-09
    #"aperture_extract": [-13,+11], # blue 2025-09-10
    #"aperture_extract": [-11,+13], # blue 2025-09-11
    #"aperture_extract": [-10,+15], # blue 2026-01-28
    "aperture_extract": [-10,+13], # blue 2026-01-29
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-29_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_20260128_0031","flat_isb_3000mas_20260128_0032", "flat_isb_3000mas_20260128_0033","flat_isb_3000mas_20260128_0034", "flat_isb_3000mas_20260128_0035",
                  "flat_isb_3000mas_20260128_0036","flat_isb_3000mas_20260128_0037", "flat_isb_3000mas_20260128_0038","flat_isb_3000mas_20260128_0039", "flat_isb_3000mas_20260128_0040"],
    "sci_list":  ["sci_0055"],
    "arc_list":  ["thar_0075", "thar_0076"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HR1996_0055",
    "fileout_arc":  "master_thar_0075",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"aperture_extract": [-12,+4],  # blue 2026-01-28
    #"aperture_extract": [-8,+11],  # 2026-09-09
    #"aperture_extract": [-7,+10],  # 2026-09-10
    #"aperture_extract": [-10,+8],  # 2026-09-11
    #"aperture_extract": [-10,+6], # blue 2026-01-28
    "aperture_extract": [-10,+4], # blue 2026-01-29
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:
pipe = STELES_Pipeline_PartII(config_blue)
pipe.run()
#pipe.run_wavesol()
#pipe.run_science_only()

In [ ]:
#2026-01-29 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-29_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_20260128_0033","flat_isb_20260128_0034", "flat_isb_20260128_0035"],
    "fileout_flat": "master_flat",
    "arc_list":  ["thar_0075", "thar_0076"],
    "fileout_arc":  "master_thar_0075",
    # name of master frames (stored in the <RED> directory)
    #
    #"sci_list":  ["sci_0053"],
    #"fileout_sci":  "master_HR1996_0053",
    #
    #"sci_list":  ["sci_0054"],
    #"fileout_sci":  "master_HR1996_0054",
    #
    #"sci_list":  ["sci_0056"],
    #"fileout_sci":  "master_HR1996_3000mas_0056",
    #
    #"sci_list":  ["sci_0057", "sci_0058", "sci_0059"],
    #"fileout_sci":  "master_HR1996_3000mas_0057_0059",
    #
    #"sci_list":  ["sci_0060"],
    #"fileout_sci":  "master_HD49798_0060",
    #
    #"sci_list":  ["sci_0061"],
    #"fileout_sci":  "master_HD49798_0061",
    #
    #"sci_list":  ["sci_0062"],
    #"fileout_sci":  "master_HD49798_0062",
    #
    #"sci_list":  ["sci_0063"],
    #"fileout_sci":  "master_HD49798_0063",
    #
    #"sci_list":  ["sci_0064"],
    #"fileout_sci":  "master_gammaVel_0064",
    #
    #"sci_list":  ["sci_0065"],
    #"fileout_sci":  "master_gammaVel_0065",
    #
    #"sci_list":  ["sci_0066","sci_0067","sci_0068","sci_0069","sci_0070"],
    #"fileout_sci":  "master_gammaVel_0066_0070",
    #
    #"sci_list":  ["sci_0071"],
    #"fileout_sci":  "master_gammaVelB_0071",
    #
    #"sci_list":  ["sci_0072"],
    #"fileout_sci":  "master_gammaVelB_0072",
    #
    #"sci_list":  ["sci_0073"],
    #"fileout_sci":  "master_V662Car_0073",
    #
    "sci_list":  ["sci_0074"],
    "fileout_sci":  "master_V662Car_0074",
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    #"x_extract": [-8,+14],  # blue 2025-09-09
    #"x_extract": [-13,+11], # blue 2025-09-10
    #"x_extract": [-11,+13], # blue 2025-09-11
    #"x_extract": [-10,+15], # blue 2026-01-28
    "x_extract": [-10,+13], # blue 2026-01-29
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-29_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_20260128_0031","flat_isb_3000mas_20260128_0032", "flat_isb_3000mas_20260128_0033","flat_isb_3000mas_20260128_0034", "flat_isb_3000mas_20260128_0035",
                  "flat_isb_3000mas_20260128_0036","flat_isb_3000mas_20260128_0037", "flat_isb_3000mas_20260128_0038","flat_isb_3000mas_20260128_0039", "flat_isb_3000mas_20260128_0040"],
    "fileout_flat": "master_flat",
    "arc_list":  ["thar_0075", "thar_0076"],
    "fileout_arc":  "master_thar_0075",
    # name of master frames (stored in the <RED> directory)
    #
    #"sci_list":  ["sci_0053"],
    #"fileout_sci":  "master_HR1996_0053",
    #
    #"sci_list":  ["sci_0054"],
    #"fileout_sci":  "master_HR1996_0054",
    #
    #"sci_list":  ["sci_0056"],
    #"fileout_sci":  "master_HR1996_3000mas_0056",
    #
    #"sci_list":  ["sci_0057", "sci_0058", "sci_0059"],
    #"fileout_sci":  "master_HR1996_3000mas_0057_0059",
    #
    #"sci_list":  ["sci_0060"],
    #"fileout_sci":  "master_HD49798_0060",
    #
    #"sci_list":  ["sci_0061"],
    #"fileout_sci":  "master_HD49798_0061",
    #
    #"sci_list":  ["sci_0062"],
    #"fileout_sci":  "master_HD49798_0062",
    #
    #"sci_list":  ["sci_0063"],
    #"fileout_sci":  "master_HD49798_0063",
    #
    #"sci_list":  ["sci_0064"],
    #"fileout_sci":  "master_gammaVel_0064",
    #
    #"sci_list":  ["sci_0065"],
    #"fileout_sci":  "master_gammaVel_0065",
    #
    #"sci_list":  ["sci_0066","sci_0067","sci_0068","sci_0069","sci_0070"],
    #"fileout_sci":  "master_gammaVel_0066_0070",
    #
    #"sci_list":  ["sci_0071"],
    #"fileout_sci":  "master_gammaVelB_0071",
    #
    #"sci_list":  ["sci_0072"],
    #"fileout_sci":  "master_gammaVelB_0072",
    #
    #"sci_list":  ["sci_0073"],
    #"fileout_sci":  "master_V662Car_0073",
    #
    "sci_list":  ["sci_0074"],
    "fileout_sci":  "master_V662Car_0074",
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    #"x_extract": [-8,+11],  # 2026-09-09
    #"x_extract": [-7,+10],  # 2026-09-10
    #"x_extract": [-10,+8],  # 2026-09-11
    #"x_extract": [-10,+6], # blue 2026-01-28
    "x_extract": [-10,+4], # blue 2026-01-29
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:

pipe = STELES_Pipeline_PartII(config_blue)
#pipe.run()
#pipe.run_wavesol()
pipe.run_science_only()

#pipe.load_rectified_products()
#pipe.load_wavelength_solution()
#pipe.fit_spatial_distortion()


pipe_r = STELES_Pipeline_PartII(config_red)
#pipe_r.run()
#pipe_r.run_wavesol()
pipe_r.run_science_only()


#pipe_r.load_rectified_products()
#pipe_r.load_wavelength_solution()
#pipe_r.fit_spatial_distortion()
#pipe_r.apply_calibration()
#pipe_r.extract_1d_spectra()
#pipe_r.compute_blaze_function()
#pipe_r.extract_1d_spectra()
#pipe_r.qc_extracted_products()
#pipe_r.qc_blaze()
#pipe_r.save_report()


In [ ]:
#2026-01-28 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-28_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_20260128_0033","flat_isb_20260128_0034", "flat_isb_20260128_0035"],
    "sci_list":  ["sci_0103"],
    "arc_list":  ["thar_0116", "thar_0117"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HIP20363_0103",
    "fileout_arc":  "master_thar_0116",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    #"x_extract": [-8,+14],  # blue 2025-09-09
    #"x_extract": [-13,+11], # blue 2025-09-10
    #"x_extract": [-11,+13], # blue 2025-09-11
    "x_extract": [-10,+15], # blue 2026-01-28
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-28_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_20260128_0031","flat_isb_3000mas_20260128_0032", "flat_isb_3000mas_20260128_0033","flat_isb_3000mas_20260128_0034", "flat_isb_3000mas_20260128_0035",
                  "flat_isb_3000mas_20260128_0036","flat_isb_3000mas_20260128_0037", "flat_isb_3000mas_20260128_0038","flat_isb_3000mas_20260128_0039", "flat_isb_3000mas_20260128_0040"],
    "sci_list":  ["HIP105140_0055"],
    "sci_list":  ["sci_0103"],
    "arc_list":  ["thar_0116", "thar_0117"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HIP20363_0103",
    "fileout_arc":  "master_thar_0116",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    #"x_extract": [-8,+11],  # 2026-09-09
    #"x_extract": [-7,+10],  # 2026-09-10
    #"x_extract": [-10,+8],  # 2026-09-11
    "x_extract": [-10,+6], # blue 2026-01-28
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": +2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:
#2026-01-28 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-28_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_20260128_0033","flat_isb_20260128_0034", "flat_isb_20260128_0035"],
    "fileout_flat": "master_flat",
    "arc_list":  ["thar_0116", "thar_0117"],
    "fileout_arc":  "master_thar_0116",
    # name of master frames (stored in the <RED> directory)
    #"sci_list":  ["sci_0100"],
    #fileout_sci":  "master_HIP20363_0100",
    #"sci_list":  ["sci_0101"],
    #"fileout_sci":  "master_HIP20363_0101",
    #"sci_list":  ["sci_0102"],
    #"fileout_sci":  "master_HIP20363_0102",
    #"sci_list":  ["sci_0104"],
    #"fileout_sci":  "master_LTT3218_0104",
    #"sci_list":  ["sci_0105"],
    #"fileout_sci":  "master_LTT3218_0105",
    #"sci_list":  ["sci_0106"],
    #"fileout_sci":  "master_HD64568_0106",
    #"sci_list":  ["sci_0107"],
    #"fileout_sci":  "master_HD64568_0107",
    #"sci_list":  ["sci_0108"],
    #"fileout_sci":  "master_HD49798_0108",
    #"sci_list":  ["sci_0109"],
    #"fileout_sci":  "master_HD49798_0109",
    #"sci_list":  ["sci_0110"],
    #"fileout_sci":  "master_etaCar_0110",
    #"sci_list":  ["sci_0111"],
    #"fileout_sci":  "master_etaCar_0111",
    #"sci_list":  ["sci_0112"],
    #"fileout_sci":  "master_etaCar_0112",
    #"sci_list":  ["sci_0113"],
    #"fileout_sci":  "master_etaCar_0113",
    #"sci_list":  ["sci_0114"],
    #"fileout_sci":  "master_etaCar_0114",
    "sci_list":  ["sci_0115"],
    "fileout_sci":  "master_etaCar_0115",
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    #"x_extract": [-8,+14],  # blue 2025-09-09
    #"x_extract": [-13,+11], # blue 2025-09-10
    #"x_extract": [-11,+13], # blue 2025-09-11
    "x_extract": [-10,+15], # blue 2026-01-28
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-28_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_20260128_0031","flat_isb_3000mas_20260128_0032", "flat_isb_3000mas_20260128_0033","flat_isb_3000mas_20260128_0034", "flat_isb_3000mas_20260128_0035",
                  "flat_isb_3000mas_20260128_0036","flat_isb_3000mas_20260128_0037", "flat_isb_3000mas_20260128_0038","flat_isb_3000mas_20260128_0039", "flat_isb_3000mas_20260128_0040"],
    "fileout_flat": "master_flat",
    "arc_list":  ["thar_0116", "thar_0117"],
    "fileout_arc":  "master_thar_0116",
    # name of master frames (stored in the <RED> directory)
    #"sci_list":  ["sci_0100"],
    #"fileout_sci":  "master_HIP20363_0100",
    #"sci_list":  ["sci_0101"],
    #"fileout_sci":  "master_HIP20363_0101",
    #"sci_list":  ["sci_0102"],
    #"fileout_sci":  "master_HIP20363_0102",
    #"sci_list":  ["sci_0104"],
    #"fileout_sci":  "master_LTT3218_0104",
    #"sci_list":  ["sci_0105"],
    #"fileout_sci":  "master_LTT3218_0105",
    #"sci_list":  ["sci_0106"],
    #"fileout_sci":  "master_HD64568_0106",
    #"sci_list":  ["sci_0107"],
    #"fileout_sci":  "master_HD64568_0107",
    #"sci_list":  ["sci_0108"],
    #"fileout_sci":  "master_HD49798_0108",
    #"sci_list":  ["sci_0109"],
    #"fileout_sci":  "master_HD49798_0109",
    #"sci_list":  ["sci_0110"],
    #"fileout_sci":  "master_etaCar_0110",
    #"sci_list":  ["sci_0111"],
    #"fileout_sci":  "master_etaCar_0111",
    #"sci_list":  ["sci_0112"],
    #"fileout_sci":  "master_etaCar_0112",
    #"sci_list":  ["sci_0113"],
    #"fileout_sci":  "master_etaCar_0113",
    #"sci_list":  ["sci_0114"],
    #"fileout_sci":  "master_etaCar_0114",
    "sci_list":  ["sci_0115"],
    "fileout_sci":  "master_etaCar_0115",
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    #"x_extract": [-8,+11],  # 2026-09-09
    #"x_extract": [-7,+10],  # 2026-09-10
    #"x_extract": [-10,+8],  # 2026-09-11
    "x_extract": [-10,+6], # blue 2026-01-28
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": +2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:

pipe = STELES_Pipeline_PartII(config_blue)
#pipe.run()
#pipe.run_wavesol()
pipe.run_science_only()

#pipe.load_rectified_products()
#pipe.load_wavelength_solution()
#pipe.fit_spatial_distortion()


pipe_r = STELES_Pipeline_PartII(config_red)
#pipe_r.run()
#pipe_r.run_wavesol()
pipe_r.run_science_only()


#pipe_r.load_rectified_products()
#pipe_r.load_wavelength_solution()
#pipe_r.fit_spatial_distortion()
#pipe_r.apply_calibration()
#pipe_r.extract_1d_spectra()
#pipe_r.compute_blaze_function()
#pipe_r.extract_1d_spectra()
#pipe_r.qc_extracted_products()
#pipe_r.qc_blaze()
#pipe_r.save_report()


In [ ]:
#2025-09-11 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-11_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_0012","quartz20w_0013", "quartz20w_0014","quartz20w_0015", "quartz20w_0016"],
    "sci_list":  ["HIP105140_0055"],
    "arc_list":  ["thar_0057", "thar_0058"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HIP105140_0055",
    "fileout_arc":  "master_thar_0057",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    #"x_extract": [-8,+14],  # blue 2025-09-09
    #"x_extract": [-13,+11], # blue 2025-09-10
    "x_extract": [-11,+13], # blue 2025-09-11
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-11_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_0018","quartz20w_0019", "quartz20w_0020","quartz20w_0021", "quartz20w_0022"],
    "sci_list":  ["HIP105140_0055"],
    "arc_list":  ["thar_0057", "thar_0058"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HIP105140_0055",
    "fileout_arc":  "master_thar_0057",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    #"x_extract": [-8,+11],  # 2026-09-09
    #"x_extract": [-7,+10],  # 2026-09-10
    "x_extract": [-10,+8],  # 2026-09-11
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:
#2025-09-11 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-11_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_0012","quartz20w_0013", "quartz20w_0014","quartz20w_0015", "quartz20w_0016"],
    "fileout_flat": "master_flat",
    # name of master frames (stored in the <RED> directory)
    #"sci_list":  ["thetaMus_0044"],
    #"fileout_sci":  "master_thetaMus_0044",
    #"arc_list":  ["thar_0047", "thar_0048"],
    #"fileout_arc":  "master_thar_0047",
    #
    #"sci_list":  ["thetaMusB_0049"],
    #"fileout_sci":  "master_thetaMusB_0049",
    #"arc_list":  ["thar_0050", "thar_0051"],
    #"fileout_arc":  "master_thar_0050",
    #
    #"sci_list":  ["rhoOphA_0052"],
    #"fileout_sci":  "master_rhoOphA_0052",
    #"arc_list":  ["thar_0053", "thar_0054"],
    #"fileout_arc":  "master_thar_0053",
    #
    #"sci_list":  ["HD135160_0059"],
    #"fileout_sci":  "master_HD135160_0059",
    #"arc_list":  ["thar_0060", "thar_0061"],
    #"fileout_arc":  "master_thar_0060",
    #
    #"sci_list":  ["HD136352_0062"],
    #"fileout_sci":  "master_HD136352_0062",
    #"arc_list":  ["thar_0063", "thar_0064"],
    #"fileout_arc":  "master_thar_0063",
    #
    #"sci_list":  ["HD168431_0065"],
    #"fileout_sci":  "master_HD168431_0065",
    #"arc_list":  ["thar_0066", "thar_0067"],
    #"fileout_arc":  "master_thar_0066",
    #
    #"sci_list":  ["MWC882_0069"],
    #"fileout_sci":  "master_MWC882_0069",
    #"arc_list":  ["thar_0070", "thar_0071"],
    #"fileout_arc":  "master_thar_0070",
    #
    #"sci_list":  ["HIP105140_0072"],
    #"fileout_sci":  "master_HIP105140_0072",
    #"arc_list":  ["thar_0073", "thar_0074"],
    #"fileout_arc":  "master_thar_0073",
    #
    #"sci_list":  ["piAqr_0111"],
    #"fileout_sci":  "master_piAqr_0111",
    #"arc_list":  ["thar_0112", "thar_0113"],
    #"fileout_arc":  "master_thar_0112",
    #
    #"sci_list":  ["HIP10700_0075","HIP10700_0076","HIP10700_0077","HIP10700_0078","HIP10700_0079","HIP10700_0080","HIP10700_0081","HIP10700_0082","HIP10700_0083","HIP10700_0084"],
    #"fileout_sci":  "master_HIP10700_0075_0084",
    #"arc_list":  ["thar_0085", "thar_0086"],
    #"fileout_arc":  "master_thar_0085",
    #
    #"sci_list":  ["HIP10700_0087","HIP10700_0088","HIP10700_0089","HIP10700_0090","HIP10700_0091","HIP10700_0092","HIP10700_0093","HIP10700_0094","HIP10700_0095","HIP10700_0096"],
    #"fileout_sci":  "master_HIP10700_0087_0096",
    #"arc_list":  ["thar_0097", "thar_0098"],
    #"fileout_arc":  "master_thar_0097",
    #
    #"sci_list":  ["HIP10700_0100","HIP10700_0101","HIP10700_0102","HIP10700_0103","HIP10700_0104","HIP10700_0105","HIP10700_0106","HIP10700_0107","HIP10700_0108"],
    #"fileout_sci":  "master_HIP10700_0100_0108",
    #"arc_list":  ["thar_0109", "thar_0110"],
    #"fileout_arc":  "master_thar_0109",
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    #"x_extract": [-8,+14],  # blue 2025-09-09
    #"x_extract": [-13,+11], # blue 2025-09-10
    "x_extract": [-11,+13], # blue 2025-09-11
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-11_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_0018","quartz20w_0019", "quartz20w_0020","quartz20w_0021", "quartz20w_0022"],
    "fileout_flat": "master_flat",
    # name of master frames (stored in the <RED> directory)
    #"sci_list":  ["thetaMus_0044"],
    #"fileout_sci":  "master_thetaMus_0044",
    #"arc_list":  ["thar_0047", "thar_0048"],
    #"fileout_arc":  "master_thar_0047",
    #
    #"sci_list":  ["thetaMusB_0049"],
    #"fileout_sci":  "master_thetaMusB_0049",
    #"arc_list":  ["thar_0050", "thar_0051"],
    #"fileout_arc":  "master_thar_0050",
    #
    #"sci_list":  ["rhoOphA_0052"],
    #"fileout_sci":  "master_rhoOphA_0052",
    #"arc_list":  ["thar_0053", "thar_0054"],
    #"fileout_arc":  "master_thar_0053",
    #
    #"sci_list":  ["HD135160_0059"],
    #"fileout_sci":  "master_HD135160_0059",
    #"arc_list":  ["thar_0060", "thar_0061"],
    #"fileout_arc":  "master_thar_0060",
    #
    #"sci_list":  ["HD136352_0062"],
    #"fileout_sci":  "master_HD136352_0062",
    #"arc_list":  ["thar_0063", "thar_0064"],
    #"fileout_arc":  "master_thar_0063",
    #
    #"sci_list":  ["HD168431_0065"],
    #"fileout_sci":  "master_HD168431_0065",
    #"arc_list":  ["thar_0066", "thar_0067"],
    #"fileout_arc":  "master_thar_0066",
    #
    #"sci_list":  ["MWC882_0069"],
    #"fileout_sci":  "master_MWC882_0069",
    #"arc_list":  ["thar_0070", "thar_0071"],
    #"fileout_arc":  "master_thar_0070",
    #
    #"sci_list":  ["HIP105140_0072"],
    #"fileout_sci":  "master_HIP105140_0072",
    #"arc_list":  ["thar_0073", "thar_0074"],
    #"fileout_arc":  "master_thar_0073",
    #
    # FULL RUN BELOW:
    #
    #"sci_list":  ["piAqr_0111"],
    #"fileout_sci":  "master_piAqr_0111",
    #"arc_list":  ["thar_0112", "thar_0113"],
    #"fileout_arc":  "master_thar_0112",
    #
    #"sci_list":  ["HIP10700_0075","HIP10700_0076","HIP10700_0077","HIP10700_0078","HIP10700_0079","HIP10700_0080","HIP10700_0081","HIP10700_0082","HIP10700_0083","HIP10700_0084"],
    #"fileout_sci":  "master_HIP10700_0075_0084",
    #"arc_list":  ["thar_0085", "thar_0086"],
    #"fileout_arc":  "master_thar_0085",
    #
    #"sci_list":  ["HIP10700_0087","HIP10700_0088","HIP10700_0089","HIP10700_0090","HIP10700_0091","HIP10700_0092","HIP10700_0093","HIP10700_0094","HIP10700_0095","HIP10700_0096"],
    #"fileout_sci":  "master_HIP10700_0087_0096",
    #"arc_list":  ["thar_0097", "thar_0098"],
    #"fileout_arc":  "master_thar_0097",
    #
    #"sci_list":  ["HIP10700_0100","HIP10700_0101","HIP10700_0102","HIP10700_0103","HIP10700_0104","HIP10700_0105","HIP10700_0106","HIP10700_0107","HIP10700_0108"],
    #"fileout_sci":  "master_HIP10700_0100_0108",
    #"arc_list":  ["thar_0109", "thar_0110"],
    #"fileout_arc":  "master_thar_0109",
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    #"x_extract": [-8,+11],  # 2026-09-09
    #"x_extract": [-7,+10],  # 2026-09-10
    "x_extract": [-10,+8],  # 2026-09-11
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")                  

In [ ]:
#2025-09-11 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-11_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_0012","quartz20w_0013", "quartz20w_0014","quartz20w_0015", "quartz20w_0016"],
    "fileout_flat": "master_flat",
    # name of master frames (stored in the <RED> directory)
    #"sci_list":  ["thetaMus_0045"],
    #"fileout_sci":  "master_thetaMus_0045",
    #"sci_list":  ["thetaMus_ADC_0046"],
    #"fileout_sci":  "master_thetaMus_ADC_0046",
    #"arc_list":  ["thar_0047", "thar_0048"],
    #"fileout_arc":  "master_thar_0047",
    #
    #"sci_list":  ["HIP105140_ADC_0056"],
    #"fileout_sci":  "master_HIP105140_ADC_0056",
    #"arc_list":  ["thar_0057", "thar_0058"],
    #"fileout_arc":  "master_thar_0057",
    #
    "sci_list":  ["MWC882_0068"],
    "fileout_sci":  "master_MWC882_0068",
    "arc_list":  ["thar_0070", "thar_0071"],
    "fileout_arc":  "master_thar_0070",
    #
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    #"x_extract": [-8,+14],  # blue 2025-09-09
    #"x_extract": [-13,+11], # blue 2025-09-10
    "x_extract": [-11,+13], # blue 2025-09-11
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-11_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_0018","quartz20w_0019", "quartz20w_0020","quartz20w_0021", "quartz20w_0022"],
    "fileout_flat": "master_flat",
    # name of master frames (stored in the <RED> directory)
    #"sci_list":  ["thetaMus_0045"],
    #"fileout_sci":  "master_thetaMus_0045",
    #"sci_list":  ["thetaMus_ADC_0046"],
    #"fileout_sci":  "master_thetaMus_ADC_0046",
    #"arc_list":  ["thar_0047", "thar_0048"],
    #"fileout_arc":  "master_thar_0047",
    #
    #"sci_list":  ["HIP105140_ADC_0056"],
    #"fileout_sci":  "master_HIP105140_ADC_0056",
    #"arc_list":  ["thar_0057", "thar_0058"],
    #"fileout_arc":  "master_thar_0057",
    #
    "sci_list":  ["MWC882_0068"],
    "fileout_sci":  "master_MWC882_0068",
    "arc_list":  ["thar_0070", "thar_0071"],
    "fileout_arc":  "master_thar_0070",
    #
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    #"x_extract": [-8,+11],  # 2026-09-09
    #"x_extract": [-7,+10],  # 2026-09-10
    "x_extract": [-10,+8],  # 2026-09-11
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")                  

In [ ]:

pipe = STELES_Pipeline_PartII(config_blue)
#pipe.run()
#pipe.run_wavesol()
pipe.run_science_only()

#pipe.load_rectified_products()
#pipe.load_wavelength_solution()
#pipe.fit_spatial_distortion()


pipe_r = STELES_Pipeline_PartII(config_red)
#pipe_r.run()
#pipe_r.run_wavesol()
pipe_r.run_science_only()


#pipe_r.load_rectified_products()
#pipe_r.load_wavelength_solution()
#pipe_r.fit_spatial_distortion()
#pipe_r.apply_calibration()
#pipe_r.extract_1d_spectra()
#pipe.compute_blaze_function()
#pipe.extract_1d_spectra()
#pipe.qc_extracted_products()
#pipe.qc_blaze()
#pipe.save_report()


In [ ]:
#2025-09-10 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-10_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_100s_0028","quartz20w_100s_0029", "quartz20w_100s_0030"],
    "sci_list":  ["HIP105140_0039"],
    "arc_list":  ["thar_0040", "thar_0041"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HIP105140_0039",
    "fileout_arc":  "master_thar_0040",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    #"x_extract": [-8,+14],  # blue 2025-09-09
    "x_extract": [-13,+11], # blue 2025-09-10
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("DONE")

In [ ]:
#2025-09-10 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-10_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_100s_0028","quartz20w_100s_0029", "quartz20w_100s_0030"],
    "fileout_flat": "master_flat",
    # name of master frames (stored in the <RED> directory)
    #
    #"sci_list":  ["SAO240645_0035"],
    #"fileout_sci":  "master_SAO240645_0035",
    #"arc_list":  ["thar_0036", "thar_0037"],
    #"fileout_arc":  "master_thar_0036",
    #
    #"sci_list":  ["rhoOphB_0043"],
    #"fileout_sci":  "master_rhoOphB_0043",
    #"arc_list":  ["thar_0044", "thar_0045"],
    #"fileout_arc":  "master_thar_0044",
    #
    #"sci_list":  ["rhoOphC_0046"],
    #"fileout_sci":  "master_rhoOphC_0046",
    #"arc_list":  ["thar_0047", "thar_0048"],
    #"fileout_arc":  "master_thar_0047",
    #
    #"sci_list":  ["rhoOphDE_0049"],
    #"fileout_sci":  "master_rhoOphDE_0049",
    #"arc_list":  ["thar_0050", "thar_0051"],
    #"fileout_arc":  "master_thar_0050",
    #
    #"sci_list":  ["HIP105779_0052"],
    #"fileout_sci":  "master_HIP105779_0052",
    #"arc_list":  ["thar_0053", "thar_0054"],
    #"fileout_arc":  "master_thar_0053",
    #
    #"sci_list":  ["HD154873B_0056"],
    #"fileout_sci":  "master_HD154873B_0056",
    #"arc_list":  ["thar_0057", "thar_0058"],
    #"fileout_arc":  "master_thar_0057",
    #
    #"sci_list":  ["HIP105140_0059"],
    #"fileout_sci":  "master_HIP105140_0059",
    #"arc_list":  ["thar_0060", "thar_0061"],
    #"fileout_arc":  "master_thar_0060",
    #
    #"sci_list":  ["HD10700_0063","HD10700_0064","HD10700_0065","HD10700_0066","HD10700_0067"],
    #"fileout_sci":  "master_HD10700_0063_0067",
    #"arc_list":  ["thar_0068", "thar_0069"],
    #"fileout_arc":  "master_thar_0068",
    #
    #"sci_list":  ["HD10700_0070","HD10700_0071","HD10700_0072","HD10700_0073","HD10700_0074"],
    #"fileout_sci":  "master_HD10700_0070_0074",
    #"arc_list":  ["thar_0075", "thar_0076"],
    #"fileout_arc":  "master_thar_0075",
    #
    #"sci_list":  ["TOI4643_0116", "TOI4643_0117"],
    #"fileout_sci":  "master_TOI4643_0116",
    #"arc_list":  ["thar_0118", "thar_0119"],
    #"fileout_arc":  "master_thar_0118",
    #
    #"sci_list":  ["HD16424_0120"],
    #"fileout_sci":  "master_HD16424_0120",
    #"arc_list":  ["thar_0121", "thar_0122"],
    #"fileout_arc":  "master_thar_0121",
    #
    #"sci_list":  ["HIP105140_0124"],
    #"fileout_sci":  "master_HIP105140_0124",
    #"arc_list":  ["thar_0125", "thar_0126"],
    #"fileout_arc":  "master_thar_0125",
    #
    #"sci_list":  ["MoonBackground_0123"],
    #"fileout_sci":  "master_MoonBackground_0123",
    #"arc_list":  ["thar_0125", "thar_0126"],
    #"fileout_arc":  "master_thar_0125",
    #
    #"sci_list":  ["HD18636_0127"],
    #"fileout_sci":  "master_HD18636_0127",
    #"arc_list":  ["thar_0128", "thar_0129"],
    #"fileout_arc":  "master_thar_0128",
    #
    #"sci_list":  ["HD36665_0130"],
    #"fileout_sci":  "master_HD36665_0130",
    #"arc_list":  ["thar_0131", "thar_0132"],
    #"fileout_arc":  "master_thar_0131",
    #
    #"sci_list":  ["HD217522_0133"],
    #"fileout_sci":  "master_HD217522_0133",
    #"arc_list":  ["thar_0134", "thar_0135"],
    #"fileout_arc":  "master_thar_0134",
    #
    #"sci_list":  ["HD10700_0077","HD10700_0078","HD10700_0079","HD10700_0080","HD10700_0081"],
    #"fileout_sci":  "master_HD10700_0077_0081",
    #"arc_list":  ["thar_0082", "thar_0083"],
    #"fileout_arc":  "master_thar_0082",
    #
    #"sci_list":  ["HD10700_0084","HD10700_0085","HD10700_0086","HD10700_0087","HD10700_0088","HD10700_0089","HD10700_0090","HD10700_0091","HD10700_0092","HD10700_0093"],
    #"fileout_sci":  "master_HD10700_0084_0093",
    #"arc_list":  ["thar_0094", "thar_0095"],
    #"fileout_arc":  "master_thar_0094",
    #
    #"sci_list":  ["HD10700_0096","HD10700_0097","HD10700_0098","HD10700_0099","HD10700_0100","HD10700_0101","HD10700_0102","HD10700_0103","HD10700_0104","HD10700_0105"],
    #"fileout_sci":  "master_HD10700_0096_0105",
    #"arc_list":  ["thar_0106"],
    #"fileout_arc":  "master_thar_0106",
    #
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    #"x_extract": [-8,+14],  # blue 2025-09-09
    "x_extract": [-13,+11], # blue 2025-09-10
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("LOAD")

In [ ]:
pipe = STELES_Pipeline_PartII(config_blue)
#pipe.run()
pipe.run_wavesol()
#pipe.run_science_only()

#pipe_r = STELES_Pipeline_PartII(config_red)
#pipe_r.run()
#pipe_r.run_science_only()


#pipe.load_rectified_products()
#pipe.load_wavelength_solution()
#pipe.fit_spatial_distortion()
#pipe.apply_calibration()
#pipe.compute_blaze_function()
#pipe.extract_1d_spectra()
#pipe.qc_extracted_products()
#pipe.qc_blaze()
#pipe.save_report()


In [ ]:
config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-10_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_0015"],
    "sci_list":  ["HIP105140_0039"],
    "arc_list":  ["thar_0040", "thar_0041"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HIP105140_0039",
    "fileout_arc":  "master_thar_0040",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    #"x_extract": [-8,+11],  # 2026-09-09
    "x_extract": [-7,+10],  # 2026-09-10
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-10_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     True,                             # important for batch mode
    # name of master frames (stored in the <RED> directory)
    "flat_list": ["quartz20w_0015"],
    "fileout_flat": "master_flat",
    #
    #"sci_list":  ["SAO240645_0035"],
    #"fileout_sci":  "master_SAO240645_0035",
    #"arc_list":  ["thar_0036", "thar_0037"],
    #"fileout_arc":  "master_thar_0036",
    #
    #"sci_list":  ["rhoOphB_0043"],
    #"fileout_sci":  "master_rhoOphB_0043",
    #"arc_list":  ["thar_0044", "thar_0045"],
    #"fileout_arc":  "master_thar_0044",
    #
    #"sci_list":  ["rhoOphC_0046"],
    #"fileout_sci":  "master_rhoOphC_0046",
    #"arc_list":  ["thar_0047", "thar_0048"],
    #"fileout_arc":  "master_thar_0047",
    #
    #"sci_list":  ["rhoOphDE_0049"],
    #"fileout_sci":  "master_rhoOphDE_0049",
    #"arc_list":  ["thar_0050", "thar_0051"],
    #"fileout_arc":  "master_thar_0050",
    #
    #"sci_list":  ["HIP105779_0052"],
    #"fileout_sci":  "master_HIP105779_0052",
    #"arc_list":  ["thar_0053", "thar_0054"],
    #"fileout_arc":  "master_thar_0053",
    #
    #"sci_list":  ["HD154873B_0056"],
    #"fileout_sci":  "master_HD154873B_0056",
    #"arc_list":  ["thar_0057", "thar_0058"],
    #"fileout_arc":  "master_thar_0057",
    #
    #"sci_list":  ["HIP105140_0059"],
    #"fileout_sci":  "master_HIP105140_0059",
    #"arc_list":  ["thar_0060", "thar_0061"],
    #"fileout_arc":  "master_thar_0060",
    #
    #"sci_list":  ["TOI4643_0116", "TOI4643_0117"],
    #"fileout_sci":  "master_TOI4643_0116",
    #"arc_list":  ["thar_0118", "thar_0119"],
    #"fileout_arc":  "master_thar_0118",
    #
    #"sci_list":  ["HD16424_0120"],
    #"fileout_sci":  "master_HD16424_0120",
    #"arc_list":  ["thar_0121", "thar_0122"],
    #"fileout_arc":  "master_thar_0121",
    #
    #"sci_list":  ["HIP105140_0124"],
    #"fileout_sci":  "master_HIP105140_0124",
    #"arc_list":  ["thar_0125", "thar_0126"],
    #"fileout_arc":  "master_thar_0125",
    #
    #"sci_list":  ["MoonBackground_0123"],
    #"fileout_sci":  "master_MoonBackground_0123",
    #"arc_list":  ["thar_0125", "thar_0126"],
    #"fileout_arc":  "master_thar_0125",
    #
    #"sci_list":  ["HD18636_0127"],
    #"fileout_sci":  "master_HD18636_0127",
    #"arc_list":  ["thar_0128", "thar_0129"],
    #"fileout_arc":  "master_thar_0128",
    #
    #"sci_list":  ["HD36665_0130"],
    #"fileout_sci":  "master_HD36665_0130",
    #"arc_list":  ["thar_0131", "thar_0132"],
    #"fileout_arc":  "master_thar_0131",
    #
    #"sci_list":  ["HD217522_0133"],
    #"fileout_sci":  "master_HD217522_0133",
    #"arc_list":  ["thar_0134", "thar_0135"],
    #"fileout_arc":  "master_thar_0134",
    #
    #"sci_list":  ["HD10700_0063","HD10700_0064","HD10700_0065","HD10700_0066","HD10700_0067"],
    #"fileout_sci":  "master_HD10700_0063_0067",
    #"arc_list":  ["thar_0068", "thar_0069"],
    #"fileout_arc":  "master_thar_0068",
    #
    #"sci_list":  ["HD10700_0070","HD10700_0071","HD10700_0072","HD10700_0073","HD10700_0074"],
    #"fileout_sci":  "master_HD10700_0070_0074",
    #"arc_list":  ["thar_0075", "thar_0076"],
    #"fileout_arc":  "master_thar_0075",
    #
    #"sci_list":  ["HD10700_0077","HD10700_0078","HD10700_0079","HD10700_0080","HD10700_0081"],
    #"fileout_sci":  "master_HD10700_0077_0081",
    #"arc_list":  ["thar_0082", "thar_0083"],
    #"fileout_arc":  "master_thar_0082",
    #
    #"sci_list":  ["HD10700_0084","HD10700_0085","HD10700_0086","HD10700_0087","HD10700_0088","HD10700_0089","HD10700_0090","HD10700_0091","HD10700_0092","HD10700_0093"],
    #"fileout_sci":  "master_HD10700_0084_0093",
    #"arc_list":  ["thar_0094", "thar_0095"],
    #"fileout_arc":  "master_thar_0094",
    #
    "sci_list":  ["HD10700_0096","HD10700_0097","HD10700_0098","HD10700_0099","HD10700_0100","HD10700_0101","HD10700_0102","HD10700_0103","HD10700_0104","HD10700_0105"],
    "fileout_sci":  "master_HD10700_0096_0105",
    "arc_list":  ["thar_0106"],
    "fileout_arc":  "master_thar_0106",
    #
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    #"x_extract": [-8,+11],  # 2026-09-09
    "x_extract": [-7,+10],  # 2026-09-10
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 4,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

pipe = STELES_Pipeline_PartII(config_red)
pipe.run_wavesol()
#pipe.run_science_only()

In [ ]:

#pipe.run_science_only()
#pipe.run()

#pipe_r = STELES_Pipeline_PartII(config_red)
#pipe_r.run()
#pipe_r.run_science_only()


#pipe.load_rectified_products()
#pipe.load_wavelength_solution()
#pipe.fit_spatial_distortion()
#pipe.apply_calibration()
#pipe.compute_blaze_function()
#pipe.extract_1d_spectra()
#pipe.qc_extracted_products()
#pipe.qc_blaze()
#pipe.save_report()


In [ ]:
#2025-09-09 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-09_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_100s_0028","quartz20w_100s_0029", "quartz20w_100s_0030"],
    "sci_list":  ["HIP105140_0034"],
    "arc_list":  ["thar_0031", "thar_0032"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HIP105140_0034",
    "fileout_arc":  "master_thar_0031",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    "x_extract": [-8,+14],  # master_ThAr_20260623_800mas
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": +1,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-09_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_0015"],
    "sci_list":  ["HIP105140_0034"],
    "arc_list":  ["thar_0031", "thar_0032"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HIP105140_0034",
    "fileout_arc":  "master_thar_0031",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    "x_extract": [-8,+11],  # 2026-06-24
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:
pipe = STELES_Pipeline_PartII(config_blue)
#pipe.run()
#pipe.run_science_only()

#pipe_r = STELES_Pipeline_PartII(config_red)
#pipe_r.run()
#pipe_r.run_science_only()


#pipe.load_rectified_products()
#pipe.load_wavelength_solution()
#pipe.fit_spatial_distortion()
#pipe.apply_calibration()
#pipe.compute_blaze_function()
#pipe.extract_1d_spectra()
pipe.qc_extracted_products()
#pipe.qc_blaze()
#pipe.save_report()


In [ ]:
pipe = STELES_Pipeline_PartII(config_red)
#pipe.run()
pipe.qc_extracted_products()
#pipe.run_science_only()


#pipe.load_rectified_products()
#pipe.load_wavelength_solution()
#pipe.fit_spatial_distortion()
#pipe.apply_calibration()
#pipe.compute_blaze_function()
#pipe.extract_1d_spectra()
#pipe.qc_extracted_products()
#pipe.qc_blaze()
#pipe.save_report()


In [ ]:
#2025-09-09 (First light) - use trace and redo science and ThAr lamp
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-09_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_100s_0028","quartz20w_100s_0029", "quartz20w_100s_0030"],
    "fileout_flat": "master_flat",
    # name of master frames (stored in the <RED> directory)
    #
    #"sci_list":  ["etaCar_0123"],
    #"fileout_sci":  "master_etaCar_0123",
    #"arc_list":  ["thar_0126", "thar_0127"],
    #"fileout_arc":  "master_thar_0126",
    #
    #"sci_list":  ["alphaCenA_0025"],
    #"fileout_sci":  "master_alphaCenA_0025",
    #"arc_list":  ["thar_0031", "thar_0032"],
    #"fileout_arc":  "master_thar_0031",
    #
    #"sci_list":  ["WR70_0037"],
    #"fileout_sci":  "master_WR70_0037",
    #"arc_list":  ["thar_0038"],
    #"fileout_arc":  "master_thar_0038",
    #
    #"sci_list":  ["18Sco_0040"],
    #"fileout_sci":  "master_18Sco_0040",
    #"arc_list":  ["thar_0041", "thar_0042"],
    #"fileout_arc":  "master_thar_0041",
    #
    #"sci_list":  ["HD146850_0043"],            ###############FAILED - SPAT DIST
    #"fileout_sci":  "master_HD146850_0043",    ###############FAILED - SPAT DIST
    #"arc_list":  ["thar_0044"],                ###############FAILED - SPAT DIST
    #"fileout_arc":  "master_thar_0044",        ###############FAILED - SPAT DIST
    #
    #"sci_list":  ["rhoOphA_0047"],
    #"fileout_sci":  "master_rhoOphA_0047",
    #"arc_list":  ["thar_0048", "thar_0049"],
    #"fileout_arc":  "master_thar_0048",
    #
    #"sci_list":  ["HD175329_0050", "HD175329_0051"],
    #"fileout_sci":  "master_HD175329_0050",
    #"arc_list":  ["thar_0053", "thar_0054"],
    #"fileout_arc":  "master_thar_0053",
    #
    #"sci_list":  ["V1290Sco_0056"],
    #"fileout_sci":  "master_V1290Sco_0056",
    #"arc_list":  ["thar_0057", "thar_0058"],
    #"fileout_arc":  "master_thar_0057",
    #
    #"sci_list":  ["HD168431_0062"],
    #"fileout_sci":  "master_HD168431_0062",
    #"arc_list":  ["thar_0063"],
    #"fileout_arc":  "master_thar_0063",
    #
    #"sci_list":  ["AUMic_0064"],
    #"fileout_sci":  "master_AUMic_0064",
    #"arc_list":  ["thar_0065"],
    #"fileout_arc":  "master_thar_0065",
    #
    #"sci_list":  ["HD10700_0069"],
    #"fileout_sci":  "master_HD10700_0069",
    #"arc_list":  ["thar_0070", "thar_0071"],
    #"fileout_arc":  "master_thar_0070",
    #
    #"sci_list":  ["Achernar_0074"],
    #"fileout_sci":  "master_Achernar_0074",
    #"arc_list":  ["thar_0075"],
    #"fileout_arc":  "master_thar_0075",
    #
    #"sci_list":  ["HIP105779_0077"],
    #"fileout_sci":  "master_HIP105779_0077",
    #"arc_list":  ["thar_0078"],
    #"fileout_arc":  "master_thar_0078",
    #
    #"sci_list":  ["HD8676_0080"],
    #"fileout_sci":  "master_HD8676_0080",
    #"arc_list":  ["thar_0081", "thar_0082"],
    #"fileout_arc":  "master_thar_0081",
    #
    #"sci_list":  ["piAqr_0084","piAqr_0085","piAqr_0086"],
    #"fileout_sci":  "master_piAqr_0084",
    #"arc_list":  ["piAqr_thar0087", "piAqr_thar0088"],
    #"fileout_arc":  "master_thar_0088",
    #
    #"sci_list":  ["WASP95_0092"],
    #"fileout_sci":  "master_WASP95_0092",
    #"arc_list":  ["thar_0093"],
    #"fileout_arc":  "master_thar_0093",
    #
    #"sci_list":  ["HD10700_0097"],
    #"fileout_sci":  "master_HD10700_0097",
    #"arc_list":  ["thar_0098"],
    #"fileout_arc":  "master_thar_0098",
    #
    ##################################################################################
    #
    #"sci_list":  ["BAT99_0099"],
    #"fileout_sci":  "master_BAT99_0099",
    #"arc_list":  ["thar_0100"],
    #"fileout_arc":  "master_thar_0100",
    #
    #"sci_list":  ["HD31764_0105"],
    #"fileout_sci":  "master_HD31764_0105",
    #"arc_list":  ["thar_0106"],
    #"fileout_arc":  "master_thar_0106",
    #
    #"sci_list":  ["BD14797_0108"],
    #"fileout_sci":  "master_BD14797_0108",
    #"arc_list":  ["thar_0109"],
    #"fileout_arc":  "master_thar_0109",
    #
    #"sci_list":  ["TOI4643_0110"],
    #"fileout_sci":  "master_TOI4643_0110",
    #"arc_list":  ["thar_0111", "thar_0112"],
    #"fileout_arc":  "master_thar_0111",
    #
    #"sci_list":  ["gammaVel_0115"],
    #"fileout_sci":  "master_gammaVel_0115",
    #"arc_list":  ["thar_0116"],
    #"fileout_arc":  "master_thar_0116",
    #
    #"sci_list":  ["CD27_0119"],
    #"fileout_sci":  "master_CD27_0119",
    #"arc_list":  ["thar_0120"],
    #"fileout_arc":  "master_thar_0120",
    #
    #"sci_list":  ["LRGem_0121"],
    #"fileout_sci":  "master_LRGem_0121",
    #"arc_list":  ["thar_0122"],
    #"fileout_arc":  "master_thar_0122",
    #
    #
    #
    ##################################################################################
    #
    # RETURN FROM SOURCE BELOW:
    #
    #"sci_list":  ["HIP105140_0045"],
    #"fileout_sci":  "master_HIP105140_0045",
    #"arc_list":  ["thar_0046"],
    #"fileout_arc":  "master_thar_0046",
    #
    #"sci_list":  ["HIP105140_0059"],
    #"fileout_sci":  "master_HIP105140_0059",
    #"arc_list":  ["thar_0060"],
    #"fileout_arc":  "master_thar_0060",
    #
    #"sci_list":  ["HIP105140_0066"],
    #"fileout_sci":  "master_HIP105140_0066",
    #"arc_list":  ["thar_0067"],
    #"fileout_arc":  "master_thar_0067",
    #
    #"sci_list":  ["HIP105140_0089"],
    #"fileout_sci":  "master_HIP105140_0089",
    #"arc_list":  ["thar_0090", "thar_0091"],
    #"fileout_arc":  "master_thar_0090",
    #
    #"sci_list":  ["HIP105140_0101"],
    #"fileout_sci":  "master_HIP105140_0101",
    #"arc_list":  ["thar_0102", "thar_0103"],
    #"fileout_arc":  "master_thar_0102",
    #
    #"sci_list":  ["HIP105140_0117"],
    #"fileout_sci":  "master_HIP105140_0117",
    #"arc_list":  ["thar_0118"],
    #"fileout_arc":  "master_thar_0118",
    #
    #
    #
    #
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    "x_extract": [-10,+14],  # master_ThAr_20260623_800mas
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": +1,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}
print("Loaded")

In [ ]:
pipe = STELES_Pipeline_PartII(config_blue)
#pipe.run()
pipe.run_wavesol()

#pipe.run_science_only()

#pipe_r = STELES_Pipeline_PartII(config_red)
#pipe_r.run()
#pipe_r.run_science_only()


#pipe.load_rectified_products()
#pipe.load_wavelength_solution()
#pipe.fit_spatial_distortion()
#pipe.apply_calibration()
#pipe.compute_blaze_function()
#
#pipe.extract_1d_spectra()
#pipe.qc_extracted_products()
#pipe.qc_blaze()
#
#pipe.save_report()


In [ ]:
#2025-09-09 (First light) - use trace and redo science and ThAr lamp
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-09_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_100s_0028","quartz20w_100s_0029", "quartz20w_100s_0030"],
    "fileout_flat": "master_flat",
    # name of master frames (stored in the <RED> directory)
    #
    "sci_list":  ["etaCar_0123"],
    "fileout_sci":  "master_etaCar_0123",
    "arc_list":  ["thar_0126", "thar_0127"],
    "fileout_arc":  "master_thar_0126",
    #
    #"sci_list":  ["alphaCenA_0025"],
    #"fileout_sci":  "master_alphaCenA_0025",
    #"arc_list":  ["thar_0031", "thar_0032"],
    #"fileout_arc":  "master_thar_0031",
    #
    #"sci_list":  ["WR70_0037"],
    #"fileout_sci":  "master_WR70_0037",
    #"arc_list":  ["thar_0038"],
    #"fileout_arc":  "master_thar_0038",
    #
    #"sci_list":  ["18Sco_0040"],
    #"fileout_sci":  "master_18Sco_0040",
    #"arc_list":  ["thar_0041", "thar_0042"],
    #"fileout_arc":  "master_thar_0041",
    #
    #"sci_list":  ["HD146850_0043"],            ###############FAILED - SPAT DIST
    #"fileout_sci":  "master_HD146850_0043",    ###############FAILED - SPAT DIST
    #"arc_list":  ["thar_0044"],                ###############FAILED - SPAT DIST
    #"fileout_arc":  "master_thar_0044",        ###############FAILED - SPAT DIST
    #
    #"sci_list":  ["rhoOphA_0047"],
    #"fileout_sci":  "master_rhoOphA_0047",
    #"arc_list":  ["thar_0048", "thar_0049"],
    #"fileout_arc":  "master_thar_0048",
    #
    #"sci_list":  ["HD175329_0050", "HD175329_0051"],
    #"fileout_sci":  "master_HD175329_0050",
    #"arc_list":  ["thar_0053", "thar_0054"],
    #"fileout_arc":  "master_thar_0053",
    #
    #"sci_list":  ["V1290Sco_0056"],
    #"fileout_sci":  "master_V1290Sco_0056",
    #"arc_list":  ["thar_0057", "thar_0058"],
    #"fileout_arc":  "master_thar_0057",
    #
    #"sci_list":  ["HD168431_0062"],
    #"fileout_sci":  "master_HD168431_0062",
    #"arc_list":  ["thar_0063"],
    #"fileout_arc":  "master_thar_0063",
    #
    #"sci_list":  ["AUMic_0064"],
    #"fileout_sci":  "master_AUMic_0064",
    #"arc_list":  ["thar_0065"],
    #"fileout_arc":  "master_thar_0065",
    #
    #"sci_list":  ["HD10700_0069"],
    #"fileout_sci":  "master_HD10700_0069",
    #"arc_list":  ["thar_0070", "thar_0071"],
    #"fileout_arc":  "master_thar_0070",
    #
    #"sci_list":  ["Achernar_0074"],
    #"fileout_sci":  "master_Achernar_0074",
    #"arc_list":  ["thar_0075"],
    #"fileout_arc":  "master_thar_0075",
    #
    #"sci_list":  ["HIP105779_0077"],
    #"fileout_sci":  "master_HIP105779_0077",
    #"arc_list":  ["thar_0078"],
    #"fileout_arc":  "master_thar_0078",
    #
    #"sci_list":  ["HD8676_0080"],
    #"fileout_sci":  "master_HD8676_0080",
    #"arc_list":  ["thar_0081", "thar_0082"],
    #"fileout_arc":  "master_thar_0081",
    #
    #"sci_list":  ["piAqr_0084","piAqr_0085","piAqr_0086"],
    #"fileout_sci":  "master_piAqr_0084",
    #"arc_list":  ["piAqr_thar0087", "piAqr_thar0088"],
    #"fileout_arc":  "master_thar_0088",
    #
    #"sci_list":  ["WASP95_0092"],
    #"fileout_sci":  "master_WASP95_0092",
    #"arc_list":  ["thar_0093"],
    #"fileout_arc":  "master_thar_0093",
    #
    #"sci_list":  ["HD10700_0097"],
    #"fileout_sci":  "master_HD10700_0097",
    #"arc_list":  ["thar_0098"],
    #"fileout_arc":  "master_thar_0098",
    #
    ##################################################################################
    #
    #"sci_list":  ["BAT99_0099"],
    #"fileout_sci":  "master_BAT99_0099",
    #"arc_list":  ["thar_0100"],
    #"fileout_arc":  "master_thar_0100",
    #
    #"sci_list":  ["HD31764_0105"],
    #"fileout_sci":  "master_HD31764_0105",
    #"arc_list":  ["thar_0106"],
    #"fileout_arc":  "master_thar_0106",
    #
    #"sci_list":  ["BD14797_0108"],
    #"fileout_sci":  "master_BD14797_0108",
    #"arc_list":  ["thar_0109"],
    #"fileout_arc":  "master_thar_0109",
    #
    #"sci_list":  ["TOI4643_0110"],
    #"fileout_sci":  "master_TOI4643_0110",
    #"arc_list":  ["thar_0111", "thar_0112"],
    #"fileout_arc":  "master_thar_0111",
    #
    #"sci_list":  ["gammaVel_0115"],
    #"fileout_sci":  "master_gammaVel_0115",
    #"arc_list":  ["thar_0116"],
    #"fileout_arc":  "master_thar_0116",
    #
    #"sci_list":  ["CD27_0119"],
    #"fileout_sci":  "master_CD27_0119",
    #"arc_list":  ["thar_0120"],
    #"fileout_arc":  "master_thar_0120",
    #
    #"sci_list":  ["LRGem_0121"],
    #"fileout_sci":  "master_LRGem_0121",
    #"arc_list":  ["thar_0122"],
    #"fileout_arc":  "master_thar_0122",
    #
    #
    #
    ##################################################################################
    #
    # RETURN FROM SOURCE BELOW:
    #
    #"sci_list":  ["HIP105140_0045"],
    #"fileout_sci":  "master_HIP105140_0045",
    #"arc_list":  ["thar_0046"],
    #"fileout_arc":  "master_thar_0046",
    #
    #"sci_list":  ["HIP105140_0059"],
    #"fileout_sci":  "master_HIP105140_0059",
    #"arc_list":  ["thar_0060"],
    #"fileout_arc":  "master_thar_0060",
    #
    #"sci_list":  ["HIP105140_0066"],
    #"fileout_sci":  "master_HIP105140_0066",
    #"arc_list":  ["thar_0067"],
    #"fileout_arc":  "master_thar_0067",
    #
    #"sci_list":  ["HIP105140_0089"],
    #"fileout_sci":  "master_HIP105140_0089",
    #"arc_list":  ["thar_0090", "thar_0091"],
    #"fileout_arc":  "master_thar_0090",
    #
    #"sci_list":  ["HIP105140_0101"],
    #"fileout_sci":  "master_HIP105140_0101",
    #"arc_list":  ["thar_0102", "thar_0103"],
    #"fileout_arc":  "master_thar_0102",
    #
    #"sci_list":  ["HIP105140_0117"],
    #"fileout_sci":  "master_HIP105140_0117",
    #"arc_list":  ["thar_0118"],
    #"fileout_arc":  "master_thar_0118",
    #
    #
    #
    #
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    "x_extract": [-10,+14],  # master_ThAr_20260623_800mas
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": +1,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}
print("Loaded")

In [ ]:
#2025-09-09 (First light) - use trace and redo science and ThAr lamp
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-09_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_100s_0028","quartz20w_100s_0029", "quartz20w_100s_0030"],
    "fileout_flat": "master_flat",
    # name of master frames (stored in the <RED> directory)
    #
    # name of master frames (stored in the <RED> directory)
    #
    #"sci_list":  ["alphaCenA_0026"],
    #"fileout_sci":  "master_alphaCenA_0026",
    #"sci_list":  ["alphaCenA_0027"],
    #"fileout_sci":  "master_alphaCenA_0027",
    #"sci_list":  ["alphaCenB_0028"],
    #"fileout_sci":  "master_alphaCenB_0028",
    #"sci_list":  ["alphaCenB_0029"],
    #"fileout_sci":  "master_alphaCenB_0029",
    #"sci_list":  ["alphaCenB_0030"],
    #"fileout_sci":  "master_alphaCenB_0030",
    #
    #"sci_list":  ["HIP105140_0033"],
    #"fileout_sci":  "master_HIP105140_0033",
    #"arc_list":  ["thar_0031", "thar_0032"],
    #"fileout_arc":  "master_thar_0031",
    #
    # # FAZER A PARTIR DAQUI.....
    #
    #"sci_list":  ["WR70_0036"],
    #"fileout_sci":  "master_WR70_0036",
    #"arc_list":  ["thar_0038"],
    #"fileout_arc":  "master_thar_0038",
    #
    #"sci_list":  ["18Sco_0039"],
    #"fileout_sci":  "master_18Sco_0039",
    #"arc_list":  ["thar_0041", "thar_0042"],
    #"fileout_arc":  "master_thar_0041",
    #
    #"sci_list":  ["HD146850_0043"],            
    #"fileout_sci":  "master_HD146850_0043",    
    #"arc_list":  ["thar_0044"],                
    #"fileout_arc":  "master_thar_0044",    
    #
    #"sci_list":  ["V1290Sco_0055"],
    #"fileout_sci":  "master_V1290Sco_0055",
    #"arc_list":  ["thar_0057", "thar_0058"],
    #"fileout_arc":  "master_thar_0057",
    #
    #"sci_list":  ["HD168431_0061"],
    #"fileout_sci":  "master_HD168431_0061",
    #"arc_list":  ["thar_0063"],
    #"fileout_arc":  "master_thar_0063",
    #
    #"sci_list":  ["HD10700_0068"],
    #"fileout_sci":  "master_HD10700_0068",
    #"arc_list":  ["thar_0070", "thar_0071"],
    #"fileout_arc":  "master_thar_0070",
    #
    #"sci_list":  ["Achernar_0072"],
    #"fileout_sci":  "master_Achernar_0072",
    #"sci_list":  ["Achernar_0073"],
    #"fileout_sci":  "master_Achernar_0073",
    #"arc_list":  ["thar_0075"],
    #"fileout_arc":  "master_thar_0075",
    #
    #"sci_list":  ["HIP105779_0076"],
    #"fileout_sci":  "master_HIP105779_0076",
    #"arc_list":  ["thar_0078"],
    #"fileout_arc":  "master_thar_0078",
    #
    #"sci_list":  ["HD8676_0079"],
    #"fileout_sci":  "master_HD8676_0079",
    #"arc_list":  ["thar_0081", "thar_0082"],
    #"fileout_arc":  "master_thar_0081",
    #
    #"sci_list":  ["piAqr_0083"],
    #"fileout_sci":  "master_piAqr_0083",
    #"arc_list":  ["piAqr_thar0087", "piAqr_thar0088"],
    #"fileout_arc":  "master_thar_0088",
    #
    #"sci_list":  ["HD10700_0094"],
    #"fileout_sci":  "master_HD10700_0094",
    #"sci_list":  ["HD10700_0095"],
    #"fileout_sci":  "master_HD10700_0095",
    #"sci_list":  ["HD10700_0096"],
    #"fileout_sci":  "master_HD10700_0096",
    #"arc_list":  ["thar_0098"],
    #"fileout_arc":  "master_thar_0098",
    #
    #"sci_list":  ["HD31764_0104"],
    #"fileout_sci":  "master_HD31764_0104",
    #"arc_list":  ["thar_0106"],
    #"fileout_arc":  "master_thar_0106",
    #
    #"sci_list":  ["HD31747_0107"],
    #"fileout_sci":  "master_HD31747_0107",
    #"arc_list":  ["thar_0109"],
    #"fileout_arc":  "master_thar_0109",
    #
    #"sci_list":  ["gammaVel_0113"],
    #"fileout_sci":  "master_gammaVel_0113",
    #"sci_list":  ["gammaVel_0114"],
    #"fileout_sci":  "master_gammaVel_0114",
    #"arc_list":  ["thar_0116"],
    #"fileout_arc":  "master_thar_0116",
    #
    #"sci_list":  ["etaCar_0124"],
    #"fileout_sci":  "master_etaCar_0124",
    "sci_list":  ["etaCar_0125"],
    "fileout_sci":  "master_etaCar_0125",
    "arc_list":  ["thar_0126", "thar_0127"],
    "fileout_arc":  "master_thar_0126",
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    "x_extract": [-10,+14],  # master_ThAr_20260623_800mas
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": +1,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 7,
    "qc_final1d_n_panels": 7,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}
print("Loaded")

In [ ]:
pipe = STELES_Pipeline_PartII(config_blue)
#pipe.run()
#pipe.run_wavesol()
pipe.run_science_only()

#pipe_r = STELES_Pipeline_PartII(config_red)
#pipe_r.run()
#pipe_r.run_science_only()


#pipe.load_rectified_products()
#pipe.load_wavelength_solution()
#pipe.fit_spatial_distortion()
#pipe.apply_calibration()
#pipe.compute_blaze_function()
#
#pipe.extract_1d_spectra()
#pipe.qc_extracted_products()
#pipe.qc_blaze()
#
#pipe.save_report()


In [ ]:
config_red = { # science only - using spat dist and wave sol from previous sources
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-09-09_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["quartz20w_0015"],
    "fileout_flat": "master_flat",
    # name of master frames (stored in the <RED> directory)
    #
    #"sci_list":  ["alphaCenA_0026"],
    #"fileout_sci":  "master_alphaCenA_0026",
    #"sci_list":  ["alphaCenA_0027"],
    #"fileout_sci":  "master_alphaCenA_0027",
    #"sci_list":  ["alphaCenB_0028"],
    #"fileout_sci":  "master_alphaCenB_0028",
    #"sci_list":  ["alphaCenB_0029"],
    #"fileout_sci":  "master_alphaCenB_0029",
    #"sci_list":  ["alphaCenB_0030"],
    #"fileout_sci":  "master_alphaCenB_0030",
    #"sci_list":  ["HIP105140_0033"],
    #"fileout_sci":  "master_HIP105140_0033",
    #"arc_list":  ["thar_0031", "thar_0032"],
    #"fileout_arc":  "master_thar_0031",
    #
    #"sci_list":  ["WR70_0036"],
    #"fileout_sci":  "master_WR70_0036",
    #"arc_list":  ["thar_0038"],
    #"fileout_arc":  "master_thar_0038",
    #
    #"sci_list":  ["18Sco_0039"],
    #"fileout_sci":  "master_18Sco_0039",
    #"arc_list":  ["thar_0041", "thar_0042"],
    #"fileout_arc":  "master_thar_0041",
    #
    #"sci_list":  ["HD146850_0043"],            
    #"fileout_sci":  "master_HD146850_0043",    
    #"arc_list":  ["thar_0044"],                
    #"fileout_arc":  "master_thar_0044",    
    #
    #"sci_list":  ["V1290Sco_0055"],
    #"fileout_sci":  "master_V1290Sco_0055",
    #"arc_list":  ["thar_0057", "thar_0058"],
    #"fileout_arc":  "master_thar_0057",
    #
    #"sci_list":  ["HD168431_0061"],
    #"fileout_sci":  "master_HD168431_0061",
    #"arc_list":  ["thar_0063"],
    #"fileout_arc":  "master_thar_0063",
    #
    #"sci_list":  ["HD10700_0068"],
    #"fileout_sci":  "master_HD10700_0068",
    #"arc_list":  ["thar_0070", "thar_0071"],
    #"fileout_arc":  "master_thar_0070",
    #
    #"sci_list":  ["Achernar_0072"],
    #"fileout_sci":  "master_Achernar_0072",
    #"sci_list":  ["Achernar_0073"],
    #"fileout_sci":  "master_Achernar_0073",
    #"arc_list":  ["thar_0075"],
    #"fileout_arc":  "master_thar_0075",
    #
    #"sci_list":  ["HIP105779_0076"],
    #"fileout_sci":  "master_HIP105779_0076",
    #"arc_list":  ["thar_0078"],
    #"fileout_arc":  "master_thar_0078",
    #
    #"sci_list":  ["HD8676_0079"],
    #"fileout_sci":  "master_HD8676_0079",
    #"arc_list":  ["thar_0081", "thar_0082"],
    #"fileout_arc":  "master_thar_0081",
    #
    #"sci_list":  ["piAqr_0083"],
    #"fileout_sci":  "master_piAqr_0083",
    #"arc_list":  ["piAqr_thar0087", "piAqr_thar0088"],
    #"fileout_arc":  "master_thar_0088",
    #
    #"sci_list":  ["HD10700_0094"],
    #"fileout_sci":  "master_HD10700_0094",
    #"sci_list":  ["HD10700_0095"],
    #"fileout_sci":  "master_HD10700_0095",
    #"sci_list":  ["HD10700_0096"],
    #"fileout_sci":  "master_HD10700_0096",
    #"arc_list":  ["thar_0098"],
    #"fileout_arc":  "master_thar_0098",
    #
    #"sci_list":  ["HD31764_0104"],
    #"fileout_sci":  "master_HD31764_0104",
    #"arc_list":  ["thar_0106"],
    #"fileout_arc":  "master_thar_0106",
    #
    #"sci_list":  ["HD31747_0107"],
    #"fileout_sci":  "master_HD31747_0107",
    #"arc_list":  ["thar_0109"],
    #"fileout_arc":  "master_thar_0109",
    #
    #"sci_list":  ["gammaVel_0113"],
    #"fileout_sci":  "master_gammaVel_0113",
    #"sci_list":  ["gammaVel_0114"],
    #"fileout_sci":  "master_gammaVel_0114",
    #"arc_list":  ["thar_0116"],
    #"fileout_arc":  "master_thar_0116",
    #
    #"sci_list":  ["etaCar_0124"],
    #"fileout_sci":  "master_etaCar_0124",
    #"sci_list":  ["etaCar_0125"],
    #"fileout_sci":  "master_etaCar_0125",
    #"arc_list":  ["thar_0126", "thar_0127"],
    #"fileout_arc":  "master_thar_0126",
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    "x_extract": [-10,+14],  # master_ThAr_20260623_800mas
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products    
    
}

print("Loaded")

In [ ]:
pipe = STELES_Pipeline_PartII(config_red)
#pipe.run()
#pipe.run_wavesol()
pipe.run_science_only()

#pipe_r = STELES_Pipeline_PartII(config_red)
#pipe_r.run()
#pipe_r.run_science_only()


#pipe.load_rectified_products()
#pipe.load_wavelength_solution()
#pipe.fit_spatial_distortion()
#pipe.apply_calibration()
#pipe.compute_blaze_function()
#
#pipe.extract_1d_spectra()
#pipe.qc_extracted_products()
#pipe.qc_blaze()
#
#pipe.save_report()


In [ ]:
#### 2025-08-06

In [ ]:
#2025-08-06 (First light)
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-08-06_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_0039"],
    "sci_list":  ["HIP73049_0074"],
    "arc_list":  ["thar_0082", "thar_0083"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HIP73049_0074",
    "fileout_arc":  "master_ThAr_0082",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    "x_extract": [-6,+14],  # master_ThAr_20260623_800mas
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-08-06_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_0043"],
    "sci_list":  ["HIP73049_0072"],
    "arc_list":  ["thar_0082", "thar_0083"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HIP73049_0072",
    "fileout_arc":  "master_ThAr_0082",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    "x_extract": [-10,+10],  # 2026-06-24
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:

#2025-08-06 (First light) - use trace and redo science and ThAr lamp
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-08-06_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_0039"],
    "fileout_flat": "master_flat",
    # name of master frames (stored in the <RED> directory)
    "sci_list":  ["18Sco_0093"],
    "arc_list":  ["thar_0094", "thar_0095"],
    #
    #"fileout_sci":  "master_HR4819_0067",
    #"fileout_sci":  "master_etaCar_0080",
    #"fileout_arc":  "master_ThAr_0082",
    #"fileout_sci":  "master_etaCar_0081",
    #"fileout_sci":  "master_HD146850_0084",
    #"fileout_sci":  "master_PDS456_0088",
    #
    #"fileout_sci":  "master_HD163296_0089",
    #"fileout_arc":  "master_ThAr_0090",
    #"fileout_sci":  "master_HD130694_0085",
    #
    #"fileout_sci":  "master_18Sco_0093",
    #"fileout_arc":  "master_ThAr_0094",
    #"fileout_sci":  "master_HD135160_0092",
    #
    #"fileout_sci":  "master_AUMic_0096",
    #"fileout_arc":  "master_thar_0098",
    #"fileout_sci":  "master_PNeTc1_0097",
    #
    #"fileout_sci":  "master_HD201601_0100",
    "fileout_arc":  "master_thar_0102",
    "fileout_sci":  "master_ESO13734_0101",
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    "x_extract": [-6,+14],  # master_ThAr_20260623_800mas
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    #
    "qc_blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2025-08-06_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_0043"],
    "fileout_flat": "master_flat",
    # name of master frames (stored in the <RED> directory)
    "sci_list":  ["18Sco_0093"],
    "arc_list":  ["thar_0094", "thar_0095"],
    #
    #"fileout_sci":  "master_etaCar_0080",
    #"fileout_arc":  "master_ThAr_0082",
    #"fileout_sci":  "master_etaCar_0081",
    #"fileout_sci":  "master_HD146850_0084",
    #"fileout_sci":  "master_PDS456_0088",
    #
    #"fileout_sci":  "master_HD163296_0089",
    #"fileout_arc":  "master_ThAr_0090",
    #"fileout_sci":  "master_HD130694_0085",
    #
    #"fileout_sci":  "master_18Sco_0093",
    #"fileout_arc":  "master_ThAr_0094",
    #"fileout_sci":  "master_HD135160_0092",
    #
    #"fileout_sci":  "master_AUMic_0096",
    #"fileout_arc":  "master_thar_0098",
    #"fileout_sci":  "master_PNeTc1_0097",
    #
    #"fileout_sci":  "master_HD201601_0100",
    "fileout_arc":  "master_thar_0102",
    "fileout_sci":  "master_ESO13734_0101",
    #
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    "x_extract": [-10,+10],  # 2026-06-24
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    #
    "qc_blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    "qc_snr_n_panels": 4,
    "qc_final1d_n_panels": 4,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:
pipe = STELES_Pipeline_PartII(config_blue)
#pipe.run()
pipe.run_science_only()

pipe_r = STELES_Pipeline_PartII(config_red)
#pipe_r.run()
pipe_r.run_science_only()


#pipe_r = STELES_Pipeline_PartII(config_red)
#pipe_r.run()


#pipe = STELES_Pipeline_PartII(config_red)
#pipe._identify_channel()
#pipe._get_steles_version()
#pipe._get_channel_parameters()
#print("now check if the parameters loaded with load_instrument_config are fine!")

#pipe.load_master_frames()
#pipe.load_trace()
#pipe.quality_control_apertures()
#pipe.extract_orders()

#pipe.load_rectified_products()
#pipe.load_wavelength_solution()
#pipe.fit_spatial_distortion()
#pipe.apply_calibration()
#pipe.compute_blaze_function()
#pipe.extract_1d_spectra()
#pipe.qc_extracted_products()
#pipe.qc_blaze()
#pipe.save_report()

In [ ]:
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-29_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_20260128_0033", "flat_isb_3000mas_20260128_0034", "flat_isb_3000mas_20260128_0035"],
    "sci_list":  ["sci_0054"],
    "arc_list":  ["thar_600mas_0013", "thar_600mas_0014"], # 1-sec and 60-sec ThAr exposure
    # name of master frames (stored in the <RED> directory)
    #"fileout_flat": "master_flat",
    #"fileout_sci":  "master_HR1996_600mas",
    #"fileout_arc":  "master_ThAr_600mas",
    #
    "fileout_flat": "master_flat_20260623_800mas",
    "fileout_sci":  "master_HR1996_20260623_800mas",
    "fileout_arc":  "master_ThAr_20260623_800mas",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    "x_extract": [-6,+10],  # master_ThAr_20260623_800mas
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -8,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-29_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_0034", "flat_isb_3000mas_0035", "flat_isb_3000mas_0036"],
    "sci_list":  ["sci_0053"],
    "arc_list":  ["thar_600mas_0013", "thar_600mas_0014"], # 1-sec and 60-sec ThAr exposure
    # name of master frames (stored in the <RED> directory)
    #"fileout_flat": "master_flat",
    #"fileout_sci":  "master_HR1996_600mas",
    #"fileout_arc":  "master_ThAr_600mas",
    #
    "fileout_flat": "master_flat_20260624_800mas",
    "fileout_sci":  "master_HR1996_20260624_800mas",
    #"fileout_arc":  "master_ThAr_20260624_800mas",
    "fileout_arc":  "master_ThAr_20260624_800mas_90s",
    # arc scaling parameters
    "saturation_threshold_for_arc": 40000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 39],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    "x_extract": [-12,+2],  # 2026-06-24
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:
#2026-06-30
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-06-30_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_20260128_0033", "flat_isb_3000mas_20260128_0034", "flat_isb_3000mas_20260128_0035"],
    "sci_list":  ["sci_0054"],
    "arc_list":  ["thar_1s_0004", "thar_120s_0007"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HR1996_600mas",
    "fileout_arc":  "master_ThAr_800mas",
    # arc scaling parameters
    "saturation_threshold_for_arc": 10000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    "x_extract": [-6,+10],  # master_ThAr_20260623_800mas
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-06-30_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_20260128_0034", "flat_isb_3000mas_20260128_0035", "flat_isb_3000mas_20260128_0036"],
    "sci_list":  ["sci_0053"],
    "arc_list":  ["thar_1s_0004", "thar_120s_0007"],
    # name of master frames (stored in the <RED> directory)
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HR1996_600mas",
    "fileout_arc":  "master_ThAr_800mas",
    # arc scaling parameters
    "saturation_threshold_for_arc": 10000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 38],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    "x_extract": [-12,+2],  # 2026-06-24
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:
#2026-07-06
config_blue = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    #"raw_directory":  "D:/SOAR/Steles/2026-01-29_blue/", # set full path contaninig the raw data
    "raw_directory":  "D:/SOAR/Steles/2026-07-06_blue/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_20260128_0033", "flat_isb_3000mas_20260128_0034", "flat_isb_3000mas_20260128_0035"],
    "sci_list":  ["sci_0054"],
    #"arc_list":  ["thar_600mas_0013", "thar_600mas_0014"], # 1-sec and 60-sec ThAr exposure
    #"arc_list":  ["thar_bestfocus_1s_0007", "thar_bestfocus_60s_0008"],
    "arc_list":  ["ThAr_1sec_B_0123", "ThAr_60sec_B_0124"],
    # name of master frames (stored in the <RED> directory)
    #"fileout_flat": "master_flat",
    #"fileout_sci":  "master_HR1996_600mas",
    #"fileout_arc":  "master_ThAr_600mas",
    #"fileout_flat": "master_flat_20260623_800mas",
    #"fileout_sci":  "master_HR1996_20260623_800mas",
    #"fileout_arc":  "master_ThAr_20260623_800mas",
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HR1996",
    "fileout_arc":  "master_ThAr_20260706_800mas",
    # arc scaling parameters
    "saturation_threshold_for_arc": 30000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": True, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 14, 29, 44, 59],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-6,+18],  # blue 2026-01-28
    "x_extract": [-6,+10],  # master_ThAr_20260623_800mas
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": -2,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

config_red = {
    # =========================================================
    # General execution control
    # =========================================================
    "dry_run": False,
    # =========================================================
    # PART I
    # =========================================================
    # Input / Output
    "raw_directory":  "D:/SOAR/Steles/2026-01-29_red/", # set full path contaninig the raw data
    "raw_directory":  "D:/SOAR/Steles/2026-07-06_red/", # set full path contaninig the raw data
    "save_plots":     True,
    "show_plots":     False,                             # important for batch mode
    # name of input files for flats, science and arcs
    "flat_list": ["flat_isb_3000mas_20260128_0034", "flat_isb_3000mas_20260128_0035", "flat_isb_3000mas_20260128_0036"],
    "sci_list":  ["sci_0053"],
    #"arc_list":  ["thar_600mas_0013", "thar_600mas_0014"], # 1-sec and 60-sec ThAr exposure
    #"arc_list":  ["thar_20260624_800mas_1s_0010", "thar_20260624_800mas_30s_0011"],
    #"arc_list":  ["thar_20260624_800mas_1s_0014", "thar_20260624_800mas_90s_0016"],
    "arc_list":  ["ThAr_1sec_R_0127", "ThAr_60sec_R_0128"],
    # name of master frames (stored in the <RED> directory)
    #"fileout_flat": "master_flat",
    #"fileout_sci":  "master_HR1996_600mas",
    #"fileout_arc":  "master_ThAr_600mas",
    #"fileout_flat": "master_flat_20260624_800mas",
    #"fileout_sci":  "master_HR1996_20260624_800mas",
    #"fileout_arc":  "master_ThAr_20260624_800mas_90s",
    "fileout_flat": "master_flat",
    "fileout_sci":  "master_HR1996",
    "fileout_arc":  "master_ThAr_20260706_800mas",
    # arc scaling parameters
    "saturation_threshold_for_arc": 30000,
    # Tracing parameters
    "trace_source": "sci",    # "sci" or flat", default='sci'
    "trace_from_observations": False, # True if refiting trace from observations instead of using the trace from the database
    "refit_trace_skip_orders": [None],
    "sigma_clip": 5,
    "fail_on_trace_intersection": False,  # or True for strict mode
    # QC visualization parameters
    "aperture_orders": [0, 9, 19, 29, 38],
    "aperture_vmax": {"flat": 4.0, "arc": 3.5, "sci": 4.0},
    # Extracting parameters
    #"x_extract": [-12,+4],  # blue 2026-01-28
    "x_extract": [-12,+2],  # 2026-06-24
    # =========================================================
    # PART II
    # =========================================================
    "window_fit_peak": 11,           # pixel window for peak refinement
    "max_fwhm_threshold": 10.0,      # exclude peaks broader than this threshold (in pixels) 
    "wavelength_pixel_offset": 0,   # offset the reference wavelength peak lines by a number of pixels (helps improving the wavelength solution)  
    "wavesolution_ndeg": 3,          # polynomial degree
    "wavesolution_niter": 3,         # number of iterations to refine wavelength solution
    "wavesolution_rms_threshold": 2.0, # sigma-threshold for removing lines from the wavelength solution
    # =========================================================
    # Spatial distortion parameters
    # =========================================================
    "ndeg_distortion": 2,         # polynomial degree for distortion
    "n_pix_step": 1,              # sampling along spatial axis
    # line filtering
    "saturation_threshold_for_spatial_distortion": 1000000,
    "sigma_threshold": 3.0,
    # fitting robustness
    "fitting_sigma_threshold": 3.0,
    "fitting_niter": 3,
    "distortion_sigma_init": 3.0,
    "distortion_sigma_min": 1.0,
    "distortion_sigma_step": 0.5,
    # QC visualization parameters
    "wave_pixel_reference": [2, 1024, 2045],  # typical for 2048 pix detector
    "spectral_pixel_limits": [1500, 2000],
    # =========================================================
    # Blaze function parameters
    # =========================================================
    "blaze_mode": "simple", # "simple" or "continuum"
    "blaze_min_deg": 15,
    "blaze_max_deg": 55,
    "blaze_sigma": 5,
    "blaze_ncols": 7,
    "qc_blaze_n_panels": 10,
    # =========================================================
    # Optional behavior
    # =========================================================
    "overwrite": True,   # overwrite existing products
}

print("Loaded")

In [ ]:
#pipeline_partII_blue = STELES_Pipeline_PartII(config_blue)
pipe = STELES_Pipeline_PartII(config_red)
#pipeline_partII_blue.run()
#pipeline_partII_blue.load_rectified_products()
#pipeline_partII_blue.wavelength_solution()
#pipeline_partII_blue.load_wavelength_solution()
#pipeline_partII_blue.fit_spatial_distortion()
#pipeline_partII_blue.apply_calibration()

pipe.load_rectified_products()
pipe.load_wavelength_solution()
pipe.fit_spatial_distortion()
pipe.apply_calibration()

In [ ]:
pipeline_partII_blue = STELES_Pipeline_PartII(config_blue)
#pipeline_partII_blue.run()
pipeline_partII_blue.load_rectified_products()
pipeline_partII_blue.load_wavelength_solution()
pipeline_partII_blue.fit_spatial_distortion()
pipeline_partII_blue.apply_calibration()

In [ ]:
pipeline_partII_red = STELES_Pipeline_PartII(config_red)
pipeline_partII_red.run()